# chip_microrefrigeration_stable

Updated hardware-aware chip-model documentation so the hardware-aware tab uses R_ext,hw from the cold-plate hardware model and no longer presents the reduced-order R_ext(u), R0/Rinf cooling surrogate inside the hardware-aware chip equations.


In [1]:

# Self-contained full-hybrid-optimum hot-spot MR model.
# Units:
#   Area A: mm^2
#   Heat flux q: W/mm^2
#   Total chip power Q = q*A: W
#   Thermal resistance-area products R: K*mm^2/W
#   Pump/MR power densities: W/mm^2
#   Frequency: GHz
#   Voltage: V

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, replace
import ipywidgets as widgets
from IPython.display import display, clear_output

@dataclass(frozen=True)
class PumpParams:
    p_static: float = 0.02   # W/mm^2
    p_scale: float = 0.02    # W/mm^2
    n: float = 3.0           # dimensionless
    r: float = 1.0           # dimensionless

@dataclass(frozen=True)
class ThermalParams:
    DeltaT: float = 60.0     # K
    A_mm2: float = 100.0     # mm^2
    beta: float = 0.01       # effective total hot-spot area / die area (beta_eff)
    beta_spot: float = 0.01  # single representative hot-spot area / die area
    alpha_tot: float = 0.05  # hot-spot power / total chip power
    R_die: float = 5.0       # K*mm^2/W
    R0: float = 60.0         # K*mm^2/W
    R_inf: float = 15.0      # K*mm^2/W
    m: float = 1.0           # dimensionless
    chi: float = 1.0         # K*mm^2/W
    p_sp: float = 0.5        # dimensionless
    rho_c: float = 0.5       # dimensionless
    xi: float = 0.6          # fraction of spreading above MR plane
    COP_micro: float = 0.1   # dimensionless
    pump: PumpParams = PumpParams()

@dataclass(frozen=True)
class BUParams:
    V0: float = 0.8          # V
    f0: float = 2.0          # GHz
    Q_dyn0: float = 200.0    # W
    Q_leak0: float = 50.0    # W
    alpha_dyn0: float = 0.25
    alpha_leak0: float = 0.45
    beta: float = 0.01
    Vth: float = 0.30        # V
    nu: float = 1.5
    eta_D: float = 0.10
    n: float = 1.5
    V_t: float = 0.02585     # V
    a: float = 0.75
    b: float = 0.75

def pump_shape(u, pump):
    u = np.asarray(u, dtype=float)
    out = np.zeros_like(u)
    mask = u > 0
    if np.any(mask):
        uu = u[mask]
        out[mask] = uu**pump.n * ((1.0 + uu**2) / (2.0 * uu))**pump.r
    return out

def p_pump(u, pump):
    u = np.asarray(u, dtype=float)
    return np.where(u <= 0, 0.0, pump.p_static + pump.p_scale * pump_shape(u, pump))

def R_ext(u, tp):
    u = np.asarray(u, dtype=float)
    return tp.R_inf + (tp.R0 - tp.R_inf) / (1.0 + u**tp.m)

def spreading_terms(tp):
    # Local spreading is controlled by the area of one representative hotspot,
    # not by the aggregate multi-hotspot area. The aggregate beta_eff is still
    # used in gammas() for the chip-level power/area partition.
    beta = getattr(tp, "beta_spot", tp.beta)
    RspH = tp.chi * (beta**(-tp.p_sp) - 1.0)
    RspB = tp.chi * ((1.0 - beta)**(-tp.p_sp) - 1.0)
    Rc = tp.rho_c * np.sqrt(max(RspH, 0.0) * max(RspB, 0.0))
    return RspH, RspB, Rc

def matrix_elements(tp):
    RspH, RspB, Rc = spreading_terms(tp)
    return dict(
        RHHd=tp.R_die + (1.0 - tp.xi) * RspH,
        RHBd=(1.0 - tp.xi) * Rc,
        RBHd=(1.0 - tp.xi) * Rc,
        RBBd=tp.R_die + (1.0 - tp.xi) * RspB,
        RHHu_const=tp.xi * RspH,
        RHBu=tp.xi * Rc,
        RBHu=tp.xi * Rc,
        RBBu_const=tp.xi * RspB,
    )

def gammas(tp):
    # tp.alpha_tot and tp.beta are the effective aggregate hotspot fractions:
    # alpha_eff = N_H*alpha_spot and beta_eff = N_H*beta_spot.
    return tp.alpha_tot / tp.beta, (1.0 - tp.alpha_tot) / (1.0 - tp.beta)

def RHH_up(u, tp):
    e = matrix_elements(tp)
    return e["RHHu_const"] + R_ext(u, tp)

def RBB_up(u, tp):
    e = matrix_elements(tp)
    return e["RBBu_const"] + R_ext(u, tp)

def Phi(q, tp):
    e = matrix_elements(tp)
    gH, gB = gammas(tp)
    return tp.DeltaT / q - gH * e["RHHd"] - gB * e["RHBd"]

def s_required(u, q, tp):
    e = matrix_elements(tp)
    gH, gB = gammas(tp)
    numerator = Phi(q, tp) - gB * e["RHBu"]
    denom = gH * RHH_up(u, tp)
    return np.maximum(0.0, 1.0 - numerator / denom)

def q_walls(tp):
    e = matrix_elements(tp)
    gH, gB = gammas(tp)
    q0 = tp.DeltaT / (
        gH * (e["RHHd"] + e["RHHu_const"] + tp.R0)
        + gB * (e["RHBd"] + e["RHBu"])
    )
    qinf = tp.DeltaT / (
        gH * (e["RHHd"] + e["RHHu_const"] + tp.R_inf)
        + gB * (e["RHBd"] + e["RHBu"])
    )
    qdown = tp.DeltaT / (
        gH * e["RHHd"] + gB * (e["RHBd"] + e["RHBu"])
    )
    return dict(q0=q0, qinf=qinf, qdown=qdown)


def q0_bulk_passive(tp):
    """Bulk passive wall q_{0,B} for u=0 and s=0.

    Units:
        q0B: W/mm^2
    """
    e = matrix_elements(tp)
    gH, gB = gammas(tp)
    R_eff_B0 = (
        gH * (e["RBHd"] + e["RBHu"])
        + gB * (e["RBBd"] + e["RBBu_const"] + tp.R0)
    )
    return tp.DeltaT / R_eff_B0


def qinf_bulk_passive(tp):
    """Bulk high-flow convection wall q_{inf,B}, defined by DeltaT_B(s=0,u->infinity)=DeltaT.

    Units:
        W/mm^2
    """
    e = matrix_elements(tp)
    gH, gB = gammas(tp)
    R_eff_Binf = (
        gH * (e["RBHd"] + e["RBHu"])
        + gB * (e["RBBd"] + e["RBBu_const"] + tp.R_inf)
    )
    return tp.DeltaT / R_eff_Binf


def convection_u_for_q(q, tp):
    e = matrix_elements(tp)
    gH, gB = gammas(tp)
    target = (Phi(q, tp) - gB * e["RHBu"]) / gH - e["RHHu_const"]
    if not (tp.R_inf < target < tp.R0):
        return None
    val = (tp.R0 - tp.R_inf) / (target - tp.R_inf) - 1.0
    if val <= 0:
        return None
    return val**(1.0 / tp.m)

def convection_candidate(q, tp):
    # Convection-only candidate must satisfy both hot-spot and bulk constraints.
    uc = convection_u_for_q(q, tp)
    if uc is None:
        return None
    TH, TB = temperatures(q, float(uc), 0.0, tp)
    if TB > tp.DeltaT:
        return None
    return dict(p=float(p_pump(np.array([uc]), tp.pump)[0]), u=float(uc), s=0.0, branch="convection")

def hybrid_candidate(q, tp, u_max=1e4, n_u=1600):
    # Full hybrid optimum over all u, including u=0.
    # Feasible candidates must satisfy BOTH DeltaT_H <= DeltaT and DeltaT_B <= DeltaT.
    # s_required enforces the hot-spot constraint; we explicitly filter on the bulk constraint.
    ug = np.r_[0.0, np.logspace(-5, np.log10(u_max), n_u)]
    sg = s_required(ug, q, tp)
    pg = p_pump(ug, tp.pump) + tp.alpha_tot * sg * q / tp.COP_micro

    # Bulk constraint after applying the hotspot-required MR fraction.
    TB = np.array([temperatures(q, float(u), float(s), tp)[1] for u, s in zip(ug, sg)])
    ok = np.isfinite(pg) & (pg >= 0.0) & np.isfinite(TB) & (TB <= tp.DeltaT)

    if not np.any(ok):
        return None
    j = np.nanargmin(np.where(ok, pg, np.nan))
    return dict(p=float(pg[j]), u=float(ug[j]), s=float(sg[j]), branch="hybrid")

def temperatures(q, u, s, tp):
    e = matrix_elements(tp)
    gH, gB = gammas(tp)
    TH = q * (
        gH * e["RHHd"]
        + gB * e["RHBd"]
        + gH * (1.0 - s) * RHH_up(u, tp)
        + gB * e["RHBu"]
    )
    TB = q * (
        gH * e["RBHd"]
        + gB * e["RBBd"]
        + gH * (1.0 - s) * e["RBHu"]
        + gB * RBB_up(u, tp)
    )
    return TH, TB

def optimize_at_q(q, tp):
    # Full hybrid optimum: min over convection candidate and full hybrid candidate.
    # All returned active candidates satisfy BOTH hotspot and bulk constraints.
    w = q_walls(tp)
    TH0, TB0 = temperatures(q, 0.0, 0.0, tp)
    if q <= w["q0"] and TB0 <= tp.DeltaT:
        return dict(q=q, u=0.0, s=0.0, p=0.0, branch="passive", TH=TH0, TB=TB0)

    candidates = []
    conv = convection_candidate(q, tp)
    if conv is not None:
        candidates.append(conv)
    hyb = hybrid_candidate(q, tp)
    if hyb is not None:
        candidates.append(hyb)
    if not candidates:
        return dict(q=q, u=np.nan, s=np.nan, p=np.nan, branch="infeasible", TH=np.nan, TB=np.nan)

    best = min(candidates, key=lambda z: z["p"])
    TH, TB = temperatures(q, best["u"], best["s"], tp)
    best.update(dict(q=q, TH=TH, TB=TB))
    return best

def solve_curve(qvals, tp):
    rows = [optimize_at_q(float(q), tp) for q in qvals]
    out = {k: np.array([r[k] for r in rows]) for k in ["q", "u", "s", "p", "TH", "TB"]}
    out["branch"] = np.array([r["branch"] for r in rows], dtype=object)
    q0 = q_walls(tp)["q0"]
    with np.errstate(divide="ignore", invalid="ignore"):
        out["COP_total"] = np.where(out["p"] > 0, out["q"] / out["p"], np.nan)
        out["COP_active"] = np.where((out["p"] > 0) & (out["q"] > q0), (out["q"] - q0) / out["p"], np.nan)
        out["p_pump"] = p_pump(out["u"], tp.pump)
        out["p_micro"] = tp.alpha_tot * out["s"] * out["q"] / tp.COP_micro
        out["pump_burden"] = np.where(out["q"] > 0, out["p_pump"] / out["q"], np.nan)
        out["micro_burden"] = np.where(out["q"] > 0, out["p_micro"] / out["q"], np.nan)
        out["total_burden"] = np.where(out["q"] > 0, out["p"] / out["q"], np.nan)
    return out

def convection_curve(qvals, tp):
    q0 = q_walls(tp)["q0"]
    p = np.full_like(qvals, np.nan, dtype=float)
    u = np.full_like(qvals, np.nan, dtype=float)
    for i, q in enumerate(qvals):
        if q <= q0:
            p[i] = 0.0
            u[i] = 0.0
        else:
            conv = convection_candidate(float(q), tp)
            if conv is not None:
                p[i] = conv["p"]
                u[i] = conv["u"]
    with np.errstate(divide="ignore", invalid="ignore"):
        return dict(
            q=qvals,
            u=u,
            p=p,
            COP_total=np.where(p > 0, qvals / p, np.nan),
            COP_active=np.where((p > 0) & (qvals > q0), (qvals - q0) / p, np.nan),
            burden=np.where(p > 0, p / qvals, np.nan),
        )

def normalize_pump(tp, target_peak=1.0):
    w = q_walls(tp)
    q0, qinf = w["q0"], w["qinf"]
    if qinf <= q0:
        return tp
    qscan = np.linspace(q0 * 1.001, qinf * 0.999, 8000)
    conv = convection_curve(qscan, tp)
    peak = np.nanmax(conv["COP_active"])
    if not np.isfinite(peak) or peak <= 0:
        return tp
    factor = peak / target_peak
    return replace(tp, pump=replace(tp.pump, p_static=tp.pump.p_static * factor, p_scale=tp.pump.p_scale * factor))

def solve_voltage(lam, bu):
    target = lam**(1.0 - bu.a)
    if np.isclose(target, 1.0):
        return bu.V0
    V0, Vth, nu = bu.V0, bu.Vth, bu.nu
    if V0 <= Vth:
        raise ValueError("Need V0 > Vth")
    def g(V):
        return (V0 / V) * ((V - Vth) / (V0 - Vth))**nu - target
    lo = Vth * (1.0 + 1e-9)
    hi = max(V0 * 1.2, Vth + 0.1)
    while g(hi) < 0:
        hi *= 1.5
        if hi > 5:
            raise RuntimeError("Could not bracket voltage")
    for _ in range(150):
        mid = 0.5 * (lo + hi)
        if g(mid) < 0:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

def bottom_up_state(lam, bu):
    V = solve_voltage(lam, bu)
    f = bu.f0 * lam**(1.0 - bu.a)
    Qdyn = bu.Q_dyn0 * lam * (V / bu.V0)**2
    Qleak = bu.Q_leak0 * lam**bu.b * (V / bu.V0) * np.exp(bu.eta_D * (V - bu.V0) / (bu.n * bu.V_t))
    Q = Qdyn + Qleak
    fleak = Qleak / Q
    alpha = (1.0 - fleak) * bu.alpha_dyn0 + fleak * bu.alpha_leak0
    gamma = alpha / bu.beta
    return dict(lam=lam, V=V, f=f, Q_dyn=Qdyn, Q_leak=Qleak, Q_total=Q, f_leak=fleak, alpha_tot=alpha, gamma_H=gamma)

print("Model loaded. Run the UI cell next.")


Model loaded. Run the UI cell next.


In [2]:

# Clean UI for chip-microrefrigeration.
# Full hybrid optimum only.
from IPython.display import display, Math, Markdown

style = {'description_width': '175px'}
slider_layout = widgets.Layout(width='360px')
text_layout = widgets.Layout(width='145px')
var_box_layout = widgets.Layout(width='575px', flex='0 0 575px')

def linked_float(label, value, minv, maxv, step, units="", fmt=".4g"):
    slider = widgets.FloatSlider(
        value=value, min=minv, max=maxv, step=step, description=label,
        style=style, layout=slider_layout, continuous_update=False, readout_format=fmt
    )
    text = widgets.FloatText(
        value=value, description="value", style={'description_width': '45px'},
        layout=text_layout
    )
    def slider_to_text(change):
        text.value = change["new"]
    def text_to_slider(change):
        v = change["new"]
        if slider.min <= v <= slider.max:
            slider.value = v
    slider.observe(slider_to_text, names="value")
    text.observe(text_to_slider, names="value")
    return {"slider": slider, "text": text, "box": widgets.HBox([slider, text, widgets.HTML(f"<small>{units}</small>")], layout=var_box_layout)}

def getv(group, name):
    return group[name]["text"].value

def setv(group, name, value):
    if name in group:
        group[name]["text"].value = value
        sl = group[name]["slider"]
        if sl.min <= value <= sl.max:
            sl.value = value

def two_col(boxes):
    return widgets.VBox([
        widgets.HBox([boxes[i], boxes[i+1] if i+1 < len(boxes) else widgets.HTML("")],
                     layout=widgets.Layout(width='1180px', overflow_x='auto'))
        for i in range(0, len(boxes), 2)
    ])

def eq_box(title, lines):
    out = widgets.Output(layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px",
        margin="8px 0"
    ))
    with out:
        display(Markdown("### " + title))
        for line in lines:
            display(Math(line))
    return out

def html_note(title, body):
    return widgets.HTML(
        "<div style='background:#fff8e1;border:1px solid #e0c36a;padding:10px;margin:8px 0;'>"
        f"<h3 style='margin-top:0'>{title}</h3>{body}</div>"
    )


def collapsed_section(title, child, open_by_default=False):
    """One-line accordion wrapper for notes/equations/tables.

    The child can be an Output, HTML, VBox, or any other widget. Closed by
    default to keep the workflow controls uncluttered while keeping the full
    documentation nearby.
    """
    acc = widgets.Accordion(children=[child], layout=widgets.Layout(width="100%"))
    acc.set_title(0, title)
    acc.selected_index = 0 if open_by_default else None
    return acc

# ---------------------------
# Controls
# ---------------------------
thermal = {}
arch = {}
plot_thermal = {}
plot_arch = {}

def add_base_controls(group, include_alpha):
    group["A_mm2"] = linked_float("A", 100.0, 1.0, 1000.0, 1.0, "[mm²]")
    group["DeltaT"] = linked_float("DeltaT", 60.0, 10.0, 120.0, 1.0, "[K]")
    group["beta"] = linked_float("beta_spot", 0.01, 0.001, 0.5, 0.001, "[-]")
    group["N_H"] = linked_float("N_H", 1.0, 1.0, 128.0, 1.0, "[-]", ".0f")
    if include_alpha:
        group["alpha_tot"] = linked_float("alpha_spot", 0.05, 0.001, 0.9, 0.001, "[-]")
    group["COP_micro"] = linked_float("COP_micro", 0.10, 0.01, 2.0, 0.01, "[-]")
    group["R_die"] = linked_float("R_die", 5.0, 0.1, 100.0, 0.1, "[K·mm²/W]")
    group["R0"] = linked_float("R0", 60.0, 1.0, 10000.0, 1.0, "[K·mm²/W]")
    group["R_inf"] = linked_float("R_inf", 15.0, 0.1, 2000.0, 0.1, "[K·mm²/W]")
    group["m"] = linked_float("m", 1.0, 0.2, 4.0, 0.1, "[-]")
    group["chi"] = linked_float("chi", 1.0, 0.0, 20.0, 0.1, "[K·mm²/W]")
    group["p_sp"] = linked_float("p_sp", 0.5, 0.0, 1.5, 0.05, "[-]")
    group["rho_c"] = linked_float("rho_c", 0.5, 0.0, 1.0, 0.05, "[-]")
    group["xi"] = linked_float("xi", 0.6, 0.0, 1.0, 0.05, "[-]")
    group["p_static"] = linked_float("p_static", 0.02, 0.0, 2.0, 0.005, "[W/mm²]")
    group["p_scale"] = linked_float("p_scale", 0.02, 0.0, 2.0, 0.005, "[W/mm²]")
    group["pump_n"] = linked_float("pump n", 3.0, 1.0, 5.0, 0.1, "[-]")
    group["pump_r"] = linked_float("pump r", 1.0, 0.0, 3.0, 0.1, "[-]")

add_base_controls(thermal, include_alpha=True)
add_base_controls(arch, include_alpha=False)

arch["lam"] = linked_float("lambda", 1.0, 0.5, 10.0, 0.05, "[-]")
arch["V0"] = linked_float("V0", 0.8, 0.5, 1.3, 0.01, "[V]")
arch["f0"] = linked_float("f0", 2.0, 0.1, 6.0, 0.1, "[GHz]")
arch["Q_dyn0"] = linked_float("Q_dyn0", 200.0, 1.0, 2000.0, 5.0, "[W]")
arch["Q_leak0"] = linked_float("Q_leak0", 50.0, 0.0, 2000.0, 5.0, "[W]")
arch["alpha_dyn0"] = linked_float("alpha_dyn0", 0.25, 0.001, 0.9, 0.001, "[-]")
arch["alpha_leak0"] = linked_float("alpha_leak0", 0.45, 0.001, 0.99, 0.001, "[-]")
arch["Vth"] = linked_float("Vth", 0.30, 0.05, 0.8, 0.01, "[V]")
arch["nu"] = linked_float("nu", 1.5, 1.0, 2.5, 0.05, "[-]")
arch["eta_D"] = linked_float("eta_D", 0.10, 0.0, 0.3, 0.01, "[-]")
arch["sub_n"] = linked_float("subthreshold n", 1.5, 1.0, 2.5, 0.05, "[-]")
arch["V_t"] = linked_float("V_t", 0.02585, 0.020, 0.040, 0.0005, "[V]", ".4f")
arch["a"] = linked_float("a", 0.75, 0.0, 1.0, 0.05, "[-]")
arch["b"] = linked_float("b", 0.75, 0.0, 1.5, 0.05, "[-]")

def add_plot_controls(group):
    group["xmax"] = linked_float("q/q0,H max", 25.0, 1.1, 100.0, 0.5, "[-]")
    group["cop_ymax"] = linked_float("COP y max", 4.0, 0.5, 100.0, 0.5, "[-]")
    group["burden_ymax"] = linked_float("burden y max", 3.0, 0.2, 100.0, 0.1, "[-]")
    group["var_ymax"] = linked_float("u,s y max", 4.0, 0.5, 100.0, 0.5, "[-]")
    group["clip_conv_burden"] = linked_float("conv burden clip", 3.0, 0.2, 100.0, 0.1, "[-]")
    group["normalize"] = widgets.Checkbox(value=True, description="Normalize peak convection active COP = 1", layout=widgets.Layout(width="520px"))

add_plot_controls(plot_thermal)
add_plot_controls(plot_arch)

mode = widgets.ToggleButtons(
    # This selector chooses the chip / hotspot abstraction.
    # Internal values are kept stable for existing model logic.
    options=[
        ("Reduced-order chip model", "Compact / reduced-order model"),
        ("Architecture-aware chip model", "Architecture-aware electrothermal model"),
    ],
    value="Compact / reduced-order model",
    description="Chip model",
    style=style,
    layout=widgets.Layout(width="1120px")
)
# ---------------------------
# Presets
# ---------------------------
selected_cooling_preset_name = None
selected_chip_preset_name = None
selected_hotspot_preset_name = None

COOLING_PRESETS = {
    "Passive spreader / vapor chamber": {
        # Passive phone/tablet-class graphite sheet, heat spreader, or vapor chamber.
        # No actively pumped cold plate is implied. R0~Rinf and the active-flow optimum is near zero.
        "R0": 600.0, "R_inf": 300.0, "R_die": 5.0, "chi": 2.5,
        "m": 0.7, "p_sp": 0.5, "rho_c": 0.5, "xi": 0.75,
        "pump_n": 2.0, "pump_r": 0.0,
        "target_peak_COP_active": 10000.0,
        "u_peak_target": 0.05,
    },
    "Forced-air heatsink": {
        # Representative fan + heatsink / vapor-chamber class cooling.
        # Uniform-reference capability target: ~50-60 W/cm^2 = 0.5-0.6 W/mm^2.
        "R0": 500.0, "R_inf": 100.0, "R_die": 5.0, "chi": 2.0,
        "m": 1.0, "p_sp": 0.5, "rho_c": 0.5, "xi": 0.7,
        "pump_n": 3.0, "pump_r": 0.0,
        "target_peak_COP_active": 15.0,
        "u_peak_target": 1.0,
    },
    "Conduction-cooled ruggedized chassis": {
        # Ruggedized conduction-cooled card/chassis or cold-wall system.
        # Sealed or low-airflow defense/avionics compute; limited active-flow leverage.
        "R0": 250.0, "R_inf": 60.0, "R_die": 5.0, "chi": 1.8,
        "m": 1.0, "p_sp": 0.5, "rho_c": 0.55, "xi": 0.7,
        "pump_n": 2.5, "pump_r": 0.5,
        "target_peak_COP_active": 30.0,
        "u_peak_target": 0.6,
    },
    "Direct-to-chip liquid cold plate": {
        # Conventional package/direct-to-chip liquid cold plate.
        # Targets ~300 W/cm^2 = 3 W/mm^2 class uniform heat-flux capability.
        "R0": 120.0, "R_inf": 15.0, "R_die": 5.0, "chi": 1.0,
        "m": 1.5, "p_sp": 0.5, "rho_c": 0.5, "xi": 0.6,
        "pump_n": 3.0, "pump_r": 1.0,
        "target_peak_COP_active": 75.0,
        "u_peak_target": 2.0,
    },
    "Advanced microstructured cold plate": {
        # Microjet / short-channel / microconvective cold plate class.
        # Targets ~500-600 W/cm^2 = 5-6 W/mm^2 class package cooling.
        "R0": 50.0, "R_inf": 7.0, "R_die": 5.0, "chi": 0.5,
        "m": 2.0, "p_sp": 0.5, "rho_c": 0.4, "xi": 0.55,
        "pump_n": 2.5, "pump_r": 0.5,
        "target_peak_COP_active": 200.0,
        "u_peak_target": 3.0,
    },
    "Embedded microfluidics": {
        # Near-junction/backside embedded microfluidics.
        # Very high heat-flux capability with optimum shifted to higher normalized flow effort.
        "R0": 10.0, "R_inf": 1.5, "R_die": 0.5, "chi": 0.05,
        "m": 2.0, "p_sp": 0.5, "rho_c": 0.25, "xi": 0.4,
        "pump_n": 2.0, "pump_r": 0.5,
        "target_peak_COP_active": 3000.0,
        "u_peak_target": 6.0,
    },
}

def uniform_R_ext(u, R0, R_inf, m):
    return R_inf + (R0 - R_inf) / (1.0 + np.asarray(u)**m)

def uniform_pump_power(u, p_static, p_scale, n, r):
    u = np.asarray(u, dtype=float)
    out = np.zeros_like(u)
    mask = u > 0
    if np.any(mask):
        uu = u[mask]
        out[mask] = p_static + p_scale * uu**n * ((1 + uu**2)/(2*uu))**r
    return out

def uniform_reference_terms(group, u):
    """Uniform-load hardware reference quantities for cooling-preset calibration.

    Reference convention:
        alpha_ref = beta_ref = 1, so the hotspot/bulk partition is bypassed.

    Units:
        q0u, q_u, N: W/mm^2
        dN_du: W/mm^2 per normalized flow effort u
        S, dS_du: dimensionless pump-shape terms
    """
    DeltaT = getv(group, "DeltaT")
    R_die = getv(group, "R_die")
    R0 = getv(group, "R0")
    R_inf = getv(group, "R_inf")
    m = getv(group, "m")
    n = getv(group, "pump_n")
    r = getv(group, "pump_r")

    u = float(max(u, 1e-9))
    R = R_inf + (R0 - R_inf)/(1.0 + u**m)
    q0u = DeltaT/(R_die + R0)
    q_u = DeltaT/(R_die + R)
    N = q_u - q0u

    dR_du = -(R0 - R_inf)*m*u**(m - 1.0)/(1.0 + u**m)**2
    dN_du = -DeltaT*dR_du/(R_die + R)**2

    S = u**n * ((1.0 + u*u)/(2.0*u))**r
    log_deriv_S = (n - r)/u + (2.0*r*u)/(1.0 + u*u)
    dS_du = S * log_deriv_S
    return q0u, q_u, N, dN_du, S, dS_du


def uniform_peak_conv_active_COP(group, p_static, p_scale, return_u=False):
    """Peak active COP of the uniform-load reference pump/convection model.

    This is used only for verification/diagnostics after calibration.
    """
    DeltaT = getv(group, "DeltaT")
    R_die = getv(group, "R_die")
    R0 = getv(group, "R0")
    R_inf = getv(group, "R_inf")
    m = getv(group, "m")
    n = getv(group, "pump_n")
    r = getv(group, "pump_r")

    q0u = DeltaT / (R_die + R0)
    u = np.logspace(-5, 4, 3000)
    q_u = DeltaT / (R_die + uniform_R_ext(u, R0, R_inf, m))
    p_u = uniform_pump_power(u, p_static, p_scale, n, r)
    with np.errstate(divide="ignore", invalid="ignore"):
        cop = np.where(p_u > 0, (q_u - q0u)/p_u, np.nan)
    idx = int(np.nanargmax(cop))
    if return_u:
        return float(np.nanmax(cop)), float(u[idx])
    return float(np.nanmax(cop))


def calibrate_uniform_reference_pump(group, target_cop, u_peak_target):
    """Calibrate p_static and p_scale using two physical conditions.

    Unknowns solved:
        p_static [W/mm^2]
        p_scale  [W/mm^2]

    Conditions imposed at the prescribed preset flow effort u_peak_target:
        1) COP_conv,active_ref(u_peak_target) = target_cop
        2) d COP_conv,active_ref / du = 0 at u_peak_target

    With:
        COP = N(u) / [p_static + p_scale S(u)]
        N(u) = q_ref(u) - q0_ref

    The two equations give:
        p_scale  = N'(u*) / [target_cop S'(u*)]
        p_static = N(u*)/target_cop - p_scale S(u*)

    If numerical roundoff makes either coefficient nonpositive, it is clipped to a
    tiny positive value so the notebook remains interactive.
    """
    u_star = float(max(u_peak_target, 1e-6))
    q0u, q_u, N, dN_du, S, dS_du = uniform_reference_terms(group, u_star)

    if target_cop <= 0 or N <= 0 or dN_du <= 0 or dS_du <= 0:
        return 0.0, 0.0

    p_scale = dN_du/(target_cop*dS_du)
    p_static = N/target_cop - p_scale*S

    eps = 1e-15
    if not np.isfinite(p_scale) or p_scale < eps:
        p_scale = eps
    if not np.isfinite(p_static) or p_static < eps:
        p_static = eps

    return float(p_static), float(p_scale)

def apply_preset(group, plot_group, preset_name):
    p = COOLING_PRESETS[preset_name]
    for k, val in p.items():
        if k in ["target_peak_COP_active", "u_peak_target"]:
            continue
        setv(group, k, val)

    # Pump/fan coefficients are calibrated using a uniform-load hardware reference:
    # alpha_ref = beta_ref = 1. The user's actual alpha,beta are then left unchanged.
    p_static, p_scale = calibrate_uniform_reference_pump(
        group,
        p["target_peak_COP_active"],
        p["u_peak_target"]
    )
    setv(group, "p_static", p_static)
    setv(group, "p_scale", p_scale)

    # The preset is already physically calibrated, so turn off artificial normalization.
    plot_group["normalize"].value = False

def preset_buttons(group, plot_group, model_name):
    buttons = []
    for name in COOLING_PRESETS:
        target = COOLING_PRESETS[name]["target_peak_COP_active"]
        ustar = COOLING_PRESETS[name]["u_peak_target"]
        b = widgets.Button(description=f"{name} (COP≈{target:g}, u*≈{ustar:g})", layout=widgets.Layout(width="310px"))
        def _on_cooling_click(_, n=name, g=group, pg=plot_group):
            global selected_cooling_preset_name
            selected_cooling_preset_name = n
            apply_preset(g, pg, n)
        b.on_click(_on_cooling_click)
        buttons.append(b)
    return widgets.VBox([
        widgets.HTML(
            f"<b>Cooling scenario presets for {model_name}</b><br>"
            "<small>Presets set thermal capability and calibrate pump/fan coefficients using a two-condition uniform-load reference: peak COP target and target flow location u*. Applying a preset turns off artificial normalization. Units are W/mm².</small>"
        ),
        widgets.HBox(buttons)
    ])


# ---------------------------
# Chip scenario presets
# ---------------------------
CHIP_PRESETS_THERMAL = {
    "NAND / low-power memory": {"A_mm2":50.0, "R_die":8.0, "N_H":4.0, "alpha_tot":0.0025, "beta":0.025},
    "DRAM / commodity memory": {"A_mm2":80.0, "R_die":7.0, "N_H":4.0, "alpha_tot":0.005, "beta":0.020},
    "HBM stack": {"A_mm2":100.0, "R_die":10.0, "N_H":8.0, "alpha_tot":0.010, "beta":0.0025},
    "Mobile SoC": {"A_mm2":100.0, "R_die":8.0, "N_H":2.0, "alpha_tot":0.025, "beta":0.005},
    "Server CPU": {"A_mm2":300.0, "R_die":5.0, "N_H":8.0, "alpha_tot":0.010, "beta":0.001},
    "GPU / AI accelerator distributed": {"A_mm2":800.0, "R_die":4.0, "N_H":10.0, "alpha_tot":0.015, "beta":0.003},
    "Chiplet / 2.5D HPC package": {"A_mm2":600.0, "R_die":4.0, "N_H":4.0, "alpha_tot":0.05, "beta":0.00125},
    "3D logic-on-logic / stacked compute": {"A_mm2":200.0, "R_die":2.0, "N_H":4.0, "alpha_tot":0.0625, "beta":0.00075},
}

CHIP_PRESETS_ARCH = {
    "NAND / low-power memory": {"A_mm2":50.0, "R_die":8.0, "N_H":4.0, "V0":0.8, "f0":0.5, "Q_dyn0":2.0, "Q_leak0":0.5, "alpha_dyn0":0.0025, "alpha_leak0":0.0125, "beta":0.025},
    "DRAM / commodity memory": {"A_mm2":80.0, "R_die":7.0, "N_H":4.0, "V0":1.0, "f0":1.5, "Q_dyn0":8.0, "Q_leak0":2.0, "alpha_dyn0":0.005, "alpha_leak0":0.020, "beta":0.020},
    "HBM stack": {"A_mm2":100.0, "R_die":10.0, "N_H":8.0, "V0":1.0, "f0":3.2, "Q_dyn0":25.0, "Q_leak0":8.0, "alpha_dyn0":0.010, "alpha_leak0":0.015, "beta":0.0025},
    "Mobile SoC": {"A_mm2":100.0, "R_die":8.0, "N_H":2.0, "V0":0.8, "f0":2.5, "Q_dyn0":12.0, "Q_leak0":3.0, "alpha_dyn0":0.025, "alpha_leak0":0.060, "beta":0.005},
    "Server CPU": {"A_mm2":300.0, "R_die":5.0, "N_H":8.0, "V0":0.9, "f0":3.5, "Q_dyn0":220.0, "Q_leak0":80.0, "alpha_dyn0":0.010, "alpha_leak0":0.0225, "beta":0.001},
    "GPU / AI accelerator distributed": {"A_mm2":800.0, "R_die":4.0, "N_H":10.0, "V0":0.95, "f0":2.0, "Q_dyn0":700.0, "Q_leak0":180.0, "alpha_dyn0":0.015, "alpha_leak0":0.025, "beta":0.003},
    "Chiplet / 2.5D HPC package": {"A_mm2":600.0, "R_die":4.0, "N_H":4.0, "V0":0.9, "f0":2.5, "Q_dyn0":500.0, "Q_leak0":120.0, "alpha_dyn0":0.05, "alpha_leak0":0.07, "beta":0.00125},
    "3D logic-on-logic / stacked compute": {"A_mm2":200.0, "R_die":2.0, "N_H":4.0, "V0":0.85, "f0":3.0, "Q_dyn0":400.0, "Q_leak0":150.0, "alpha_dyn0":0.0625, "alpha_leak0":0.0875, "beta":0.00075},
}

HOTSPOT_ARCHETYPES_THERMAL = {
    "Localized tensor-core cluster": {"N_H":1.0, "alpha_tot":0.15, "beta":0.005},
    "HBM PHY hotspot": {"N_H":4.0, "alpha_tot":0.025, "beta":0.0015},
    "Power-delivery hotspot": {"N_H":2.0, "alpha_tot":0.06, "beta":0.0015},
    "Stacked-cache hotspot": {"N_H":4.0, "alpha_tot":0.025, "beta":0.002},
    "Localized RF / power block": {"N_H":1.0, "alpha_tot":0.25, "beta":0.002},
}

HOTSPOT_ARCHETYPES_ARCH = {
    "Localized tensor-core cluster": {"N_H":1.0, "alpha_dyn0":0.15, "alpha_leak0":0.25, "beta":0.005},
    "HBM PHY hotspot": {"N_H":4.0, "alpha_dyn0":0.025, "alpha_leak0":0.035, "beta":0.0015},
    "Power-delivery hotspot": {"N_H":2.0, "alpha_dyn0":0.06, "alpha_leak0":0.08, "beta":0.0015},
    "Stacked-cache hotspot": {"N_H":4.0, "alpha_dyn0":0.025, "alpha_leak0":0.04, "beta":0.002},
    "Localized RF / power block": {"N_H":1.0, "alpha_dyn0":0.25, "alpha_leak0":0.30, "beta":0.002},
}

def apply_chip_preset_thermal(group,name):
    p=CHIP_PRESETS_THERMAL[name]
    for k,v in p.items():
        setv(group,k,v)

def apply_chip_preset_arch(group,name):
    p=CHIP_PRESETS_ARCH[name]
    for k,v in p.items():
        setv(group,k,v)

def chip_preset_buttons_thermal():
    buttons=[]
    for name in CHIP_PRESETS_THERMAL:
        b=widgets.Button(description=name,layout=widgets.Layout(width="320px"))
        def _on_chip_thermal_click(_, n=name):
            global selected_chip_preset_name
            selected_chip_preset_name = n
            apply_chip_preset_thermal(thermal,n)
        b.on_click(_on_chip_thermal_click)
        buttons.append(b)
    rows=[widgets.HBox(buttons[i:i+2]) for i in range(0,len(buttons),2)]
    return widgets.VBox([widgets.HTML("<b>Chip scenario presets (compact / reduced-order)</b><br><small>Sets A, R_die, N_H, alpha_spot, and beta_spot.</small>")]+rows)

def chip_preset_buttons_arch():
    buttons=[]
    for name in CHIP_PRESETS_ARCH:
        b=widgets.Button(description=name,layout=widgets.Layout(width="320px"))
        def _on_chip_arch_click(_, n=name):
            global selected_chip_preset_name
            selected_chip_preset_name = n
            apply_chip_preset_arch(arch,n)
        b.on_click(_on_chip_arch_click)
        buttons.append(b)
    rows=[widgets.HBox(buttons[i:i+2]) for i in range(0,len(buttons),2)]
    return widgets.VBox([widgets.HTML("<b>Chip scenario presets (architecture-aware)</b><br><small>Sets A, R_die, N_H, DVFS, power, leakage, alpha_dyn/leak per hotspot, and beta_spot.</small>")]+rows)


def hotspot_preset_buttons_thermal():
    buttons = []
    for name, vals in HOTSPOT_ARCHETYPES_THERMAL.items():
        b = widgets.Button(description=name, layout=widgets.Layout(width="260px"))
        def on_click(_, vals=vals, name=name):
            global selected_hotspot_preset_name
            selected_hotspot_preset_name = name
            for k, v in vals.items():
                setv(thermal, k, v)
            run()
        b.on_click(on_click)
        buttons.append(b)
    rows = [widgets.HBox(buttons[i:i+2]) for i in range(0, len(buttons), 2)]
    return widgets.VBox([widgets.HTML("<b>Hotspot archetype presets (compact / reduced-order)</b><br><small>Sets N_H, alpha_spot, and beta_spot only.</small>")] + rows)

def hotspot_preset_buttons_arch():
    buttons = []
    for name, vals in HOTSPOT_ARCHETYPES_ARCH.items():
        b = widgets.Button(description=name, layout=widgets.Layout(width="260px"))
        def on_click(_, vals=vals, name=name):
            global selected_hotspot_preset_name
            selected_hotspot_preset_name = name
            for k, v in vals.items():
                setv(arch, k, v)
            run()
        b.on_click(on_click)
        buttons.append(b)
    rows = [widgets.HBox(buttons[i:i+2]) for i in range(0, len(buttons), 2)]
    return widgets.VBox([widgets.HTML("<b>Hotspot archetype presets (architecture-aware)</b><br><small>Sets hotspot localization parameters only.</small>")] + rows)

# ---------------------------
# Uniform dropdown preset selectors
# ---------------------------
def make_chip_model_notes_box():
    """Return a fresh copy of the chip/hotspot preset notes for each tab/section."""
    return widgets.HTML(chip_preset_note.value)

def cooling_preset_selector(group, plot_group, model_name):
    dd = widgets.Dropdown(
        options=["— choose cooling preset —"] + list(COOLING_PRESETS.keys()),
        value="— choose cooling preset —",
        description="Cooling preset",
        style=style,
        layout=widgets.Layout(width="620px")
    )
    def on_change(change):
        if change["name"] == "value" and change["new"] != "— choose cooling preset —":
            global selected_cooling_preset_name
            selected_cooling_preset_name = change["new"]
            apply_preset(group, plot_group, change["new"])
    dd.observe(on_change, names="value")
    return widgets.VBox([
        widgets.HTML(f"<b>Cooling scenario preset for {model_name}</b><br><small>Reduced-order cooling surrogate preset; sets R0, Rinf, m, pump-shape/calibration parameters, and related spreading/cooling parameters. Selecting an item applies it immediately.</small>"),
        dd
    ])

def chip_hotspot_selector_thermal(section_name="Reduced-order chip model"):
    chip_dd = widgets.Dropdown(
        options=["— choose chip preset —"] + list(CHIP_PRESETS_THERMAL.keys()),
        value="— choose chip preset —",
        description="Chip preset",
        style=style,
        layout=widgets.Layout(width="620px")
    )
    hot_dd = widgets.Dropdown(
        options=["None / keep chip preset hotspot"] + list(HOTSPOT_ARCHETYPES_THERMAL.keys()),
        value="None / keep chip preset hotspot",
        description="Hotspot archetype",
        style=style,
        layout=widgets.Layout(width="620px")
    )
    def on_chip(change):
        if change["name"] == "value" and change["new"] != "— choose chip preset —":
            global selected_chip_preset_name
            selected_chip_preset_name = change["new"]
            apply_chip_preset_thermal(thermal, change["new"])
    def on_hot(change):
        if change["name"] == "value" and change["new"] != "None / keep chip preset hotspot":
            global selected_hotspot_preset_name
            selected_hotspot_preset_name = change["new"]
            for k, v in HOTSPOT_ARCHETYPES_THERMAL[change["new"]].items():
                setv(thermal, k, v)
    chip_dd.observe(on_chip, names="value")
    hot_dd.observe(on_hot, names="value")
    return widgets.VBox([
        widgets.HTML(f"<b>{section_name} presets</b><br><small>Chip presets set A, R_die, N_H, alpha_spot, and beta_spot. Hotspot archetypes override only N_H, alpha_spot, and beta_spot. Selecting an item applies it immediately.</small>"),
        chip_dd,
        hot_dd
    ])

def chip_hotspot_selector_arch(section_name="Architecture-aware chip model"):
    chip_dd = widgets.Dropdown(
        options=["— choose chip preset —"] + list(CHIP_PRESETS_ARCH.keys()),
        value="— choose chip preset —",
        description="Chip preset",
        style=style,
        layout=widgets.Layout(width="620px")
    )
    hot_dd = widgets.Dropdown(
        options=["None / keep chip preset hotspot"] + list(HOTSPOT_ARCHETYPES_ARCH.keys()),
        value="None / keep chip preset hotspot",
        description="Hotspot archetype",
        style=style,
        layout=widgets.Layout(width="620px")
    )
    def on_chip(change):
        if change["name"] == "value" and change["new"] != "— choose chip preset —":
            global selected_chip_preset_name
            selected_chip_preset_name = change["new"]
            apply_chip_preset_arch(arch, change["new"])
    def on_hot(change):
        if change["name"] == "value" and change["new"] != "None / keep chip preset hotspot":
            global selected_hotspot_preset_name
            selected_hotspot_preset_name = change["new"]
            for k, v in HOTSPOT_ARCHETYPES_ARCH[change["new"]].items():
                setv(arch, k, v)
    chip_dd.observe(on_chip, names="value")
    hot_dd.observe(on_hot, names="value")
    return widgets.VBox([
        widgets.HTML(f"<b>{section_name} presets</b><br><small>Chip presets set A, R_die, N_H, architecture state, dynamic/leakage power, hotspot power fractions, and beta_spot. Hotspot archetypes override only hotspot localization parameters. Selecting an item applies it immediately.</small>"),
        chip_dd,
        hot_dd
    ])

def chip_model_equations_reduced_box():
    """Detailed reduced-order chip/hotspot/MR equations.

    This is the full reduced-order chip model used in both the reduced-order
    cooling surrogate and hardware-aware cooling tabs.  The cooling tab changes
    how R_ext and cooling power are computed; the hotspot/bulk constraints and
    spreading/constriction model are the same.
    """
    out = widgets.Output(layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px",
        margin="8px 0"
    ))
    with out:
        display(Markdown("### Reduced-order chip model equations"))

        display(Markdown("#### Heat flux, areas, and hotspot partition"))
        for line in [
            r"q=\frac{Q}{A},\qquad A\ [\mathrm{mm^2}],\qquad q\ [\mathrm{W/mm^2}]",
            r"\alpha_{\rm spot}=\frac{Q_{H,\rm spot}}{Q},\qquad \beta_{\rm spot}=\frac{A_{H,\rm spot}}{A}",
            r"\alpha_{\rm eff}=N_H\alpha_{\rm spot},\qquad \beta_{\rm eff}=N_H\beta_{\rm spot}",
            r"\gamma_H=\frac{\alpha_{\rm eff}}{\beta_{\rm eff}},\qquad \gamma_B=\frac{1-\alpha_{\rm eff}}{1-\beta_{\rm eff}}",
            r"q_H=\gamma_H q,\qquad q_B=\gamma_B q",
        ]:
            display(Math(line))
        display(Markdown(
            "The model distinguishes one representative local hotspot from the aggregate hotspot population. "
            "The aggregate fractions alpha_eff and beta_eff set the chip-level heat-flux partition through gamma_H and gamma_B. "
            "The local beta_spot, not beta_eff, sets the constriction/spreading penalty of each representative hotspot."
        ))

        display(Markdown("#### Reduced-order external cooling surrogate"))
        for line in [
            r"R_{\rm ext}(u)=R_\infty+\frac{R_0-R_\infty}{1+u^m}",
            r"p_{\rm pump}(u)=p_{\rm static}+p_{\rm scale}u^{n_{\rm pump}}\left(\frac{1+u^2}{2u}\right)^{r_{\rm pump}}\qquad (u>0)",
        ]:
            display(Math(line))
        display(Markdown(
            "This external-cooling surrogate is used only in the reduced-order cooling surrogate tab. "
            "In the hardware-aware cooling tab, R_ext and transport power are replaced by the cold-plate geometry/flow model."
        ))

        display(Markdown("#### Hotspot constriction/spreading resistance"))
        for line in [
            r"R_{\rm sp,H}=\chi\left(\beta_{\rm spot}^{-p_{\rm sp}}-1\right)",
            r"R_{\rm sp,B}=\chi\left((1-\beta_{\rm spot})^{-p_{\rm sp}}-1\right)",
            r"R_c=\rho_c\sqrt{R_{\rm sp,H}R_{\rm sp,B}}",
        ]:
            display(Math(line))
        display(Markdown(
            "R_sp,H is the local hotspot constriction/spreading resistance; R_sp,B is the corresponding bulk/background spreading term. "
            "R_c is the cross-coupling term between hotspot and bulk regions. "
            "The parameters chi, p_sp, rho_c, and xi are reduced-order surrogates for package/die spreading geometry."
        ))

        display(Markdown("#### Two-region resistance matrices"))
        for line in [
            r"R^\downarrow=R_{\rm die}I+(1-\xi)\begin{bmatrix}R_{\rm sp,H}&R_c\\R_c&R_{\rm sp,B}\end{bmatrix}",
            r"R^\uparrow(u)=\xi\begin{bmatrix}R_{\rm sp,H}&R_c\\R_c&R_{\rm sp,B}\end{bmatrix}+R_{\rm ext}(u)I",
            r"R^\uparrow_{\rm hw}=\xi\begin{bmatrix}R_{\rm sp,H}&R_c\\R_c&R_{\rm sp,B}\end{bmatrix}+R_{\rm ext,hw}I",
        ]:
            display(Math(line))
        display(Markdown(
            "R^downarrow represents die-side and below-MR-plane spreading. "
            "R^uparrow represents spreading above the MR plane plus external cooling. "
            "The hardware-aware model uses R^uparrow_hw with R_ext,hw supplied by the cold-plate hardware model."
        ))

        display(Markdown("#### Required MR fraction and temperatures"))
        for line in [
            r"\Phi(q)=\frac{\Delta T}{q}-\gamma_H R^\downarrow_{HH}-\gamma_B R^\downarrow_{HB}",
            r"s_{\rm req}(u,q)=\max\left[0,\ 1-\frac{\Phi(q)-\gamma_B R^\uparrow_{HB}}{\gamma_H R^\uparrow_{HH}(u)}\right]",
            r"\Delta T_H=q\left\{\gamma_HR^\downarrow_{HH}+\gamma_BR^\downarrow_{HB}+\gamma_H(1-s)R^\uparrow_{HH}(u)+\gamma_BR^\uparrow_{HB}\right\}",
            r"\Delta T_B=q\left\{\gamma_HR^\downarrow_{BH}+\gamma_BR^\downarrow_{BB}+\gamma_H(1-s)R^\uparrow_{BH}+\gamma_BR^\uparrow_{BB}(u)\right\}",
        ]:
            display(Math(line))
        display(Markdown(
            "s is the fraction of aggregate hotspot heat actively lifted by MR. "
            "s=0 is no MR. s=1 lifts the nominal hotspot heat. s>1 represents subambient local MR operation that can pull additional heat through hotspot/bulk coupling; the hardware-aware plots report -DeltaT_MR when this occurs."
        ))

        display(Markdown("#### Cooling power and COP"))
        for line in [
            r"p_{\rm cool}=p_{\rm pump}(u)+\frac{\alpha_{\rm eff}s q}{COP_\mu}",
            r"COP_{\rm total}=\frac{q}{p_{\rm cool}},\qquad COP_{\rm active}=\frac{q-q_{0,H}}{p_{\rm cool}}",
        ]:
            display(Math(line))
        display(Markdown(
            "The reduced-order tab uses area-normalized cooling burden p_cool. "
            "The hardware-aware tab uses total wall power P_wall = P_transport + P_reject + P_MR, but the MR power term is the same physics: P_MR = alpha_eff s q A / COP_mu."
        ))

        display(Markdown("#### Reference walls"))
        for line in [
            r"q_{0,H}=\frac{\Delta T}{\gamma_H(R^\downarrow_{HH}+R^\uparrow_{HH}(0))+\gamma_B(R^\downarrow_{HB}+R^\uparrow_{HB})}",
            r"q_{0,B}=\frac{\Delta T}{\gamma_H(R^\downarrow_{BH}+R^\uparrow_{BH})+\gamma_B(R^\downarrow_{BB}+R^\uparrow_{BB}(0))}",
            r"q_{\infty,H}=\frac{\Delta T}{\gamma_H(R^\downarrow_{HH}+R^\uparrow_{HH}(\infty))+\gamma_B(R^\downarrow_{HB}+R^\uparrow_{HB})}",
            r"q_{\rm die,H}=\frac{\Delta T}{\gamma_HR^\downarrow_{HH}+\gamma_B(R^\downarrow_{HB}+R^\uparrow_{HB})}",
        ]:
            display(Math(line))
        display(Markdown(
            "These reference walls are most directly tied to the reduced-order cooling surrogate. "
            "In the hardware-aware model, feasibility walls are computed directly from the hardware optimization rather than inferred from fixed R0/Rinf values."
        ))

        display(Markdown("#### Optimization problem"))
        for line in [
            r"\min_{u,s}\ p_{\rm cool}(u,s,q)",
            r"\mathrm{s.t.}\quad \Delta T_H(q,u,s)\le\Delta T,\qquad \Delta T_B(q,u,s)\le\Delta T,\qquad 0\le s\le s_{\max}",
        ]:
            display(Math(line))
        display(Markdown(
            "The hardware-aware tab solves the same hotspot and bulk constraints, replacing the variables u and p_pump with cold-plate geometry/flow and heat-rejection variables."
        ))
    return out



def chip_model_equations_reduced_hardware_box():
    """Reduced-order chip model equations as coupled to the hardware-aware cooling model.

    This version intentionally does not include the reduced-order cooling surrogate
    R_ext(u), R0, Rinf, or p_pump(u).  The external path comes from the
    cold-plate hardware model.
    """
    out = widgets.Output(layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px",
        margin="8px 0"
    ))
    with out:
        display(Markdown("### Reduced-order chip model equations for hardware-aware cooling"))

        display(Markdown("#### Heat flux, areas, and hotspot partition"))
        for line in [
            r"q=\frac{Q}{A},\qquad A\ [\mathrm{mm^2}],\qquad q\ [\mathrm{W/mm^2}]",
            r"\alpha_{\rm spot}=\frac{Q_{H,\rm spot}}{Q},\qquad \beta_{\rm spot}=\frac{A_{H,\rm spot}}{A}",
            r"\alpha_{\rm eff}=N_H\alpha_{\rm spot},\qquad \beta_{\rm eff}=N_H\beta_{\rm spot}",
            r"\gamma_H=\frac{\alpha_{\rm eff}}{\beta_{\rm eff}},\qquad \gamma_B=\frac{1-\alpha_{\rm eff}}{1-\beta_{\rm eff}}",
            r"q_H=\gamma_H q,\qquad q_B=\gamma_B q",
        ]:
            display(Math(line))
        display(Markdown(
            "This is the same reduced-order chip/hotspot abstraction used elsewhere in the notebook. "
            "The distinction in this tab is that the external cooling path is not prescribed by R0/Rinf; it is computed from the selected cold-plate hardware."
        ))

        display(Markdown("#### Hotspot constriction/spreading resistance"))
        for line in [
            r"R_{\rm sp,H}=\chi\left(\beta_{\rm spot}^{-p_{\rm sp}}-1\right)",
            r"R_{\rm sp,B}=\chi\left((1-\beta_{\rm spot})^{-p_{\rm sp}}-1\right)",
            r"R_c=\rho_c\sqrt{R_{\rm sp,H}R_{\rm sp,B}}",
        ]:
            display(Math(line))
        display(Markdown(
            "The local spreading/constriction model still uses beta_spot for each representative hotspot. "
            "The aggregate beta_eff only enters the hotspot/bulk heat-flux partition through gamma_H and gamma_B."
        ))

        display(Markdown("#### Hardware-derived external resistance"))
        for line in [
            r"R_{\rm ext,hw}(\mathcal{G},\dot V,q)=R_{\rm contact}+R_{\rm conv}(\mathcal{G},\dot V)A+\frac{1}{2}\frac{\Delta T_{\rm fluid}}{q}",
            r"\Delta T_{\rm fluid}=\frac{qA}{\rho\dot V c_p}",
            r"P_{\rm transport}=\frac{\Delta p(\mathcal{G},\dot V)\dot V}{\eta_{\rm pump/fan}}",
        ]:
            display(Math(line))
        display(Markdown(
            "Here mathcal{G} denotes the selected/frozen/optimized cold-plate geometry: feature width, height, channel count, footprint, length, and topology multipliers. "
            "This hardware-derived R_ext,hw is the only external resistance used by the hardware-aware model."
        ))

        display(Markdown("#### Two-region resistance matrices with hardware cooling"))
        for line in [
            r"R^\downarrow=R_{\rm die}I+(1-\xi)\begin{bmatrix}R_{\rm sp,H}&R_c\\R_c&R_{\rm sp,B}\end{bmatrix}",
            r"R^\uparrow_{\rm hw}=\xi\begin{bmatrix}R_{\rm sp,H}&R_c\\R_c&R_{\rm sp,B}\end{bmatrix}+R_{\rm ext,hw}I",
        ]:
            display(Math(line))
        display(Markdown(
            "R^downarrow is the die-side/below-MR path. R^uparrow_hw is the upstream path through spreading plus the hardware-derived cold plate resistance."
        ))

        display(Markdown("#### Hotspot and bulk temperature constraints"))
        for line in [
            r"\Delta T_H=q\left\{\gamma_HR^\downarrow_{HH}+\gamma_BR^\downarrow_{HB}+\gamma_H(1-s)R^\uparrow_{{\rm hw},HH}+\gamma_BR^\uparrow_{{\rm hw},HB}\right\}",
            r"\Delta T_B=q\left\{\gamma_HR^\downarrow_{BH}+\gamma_BR^\downarrow_{BB}+\gamma_H(1-s)R^\uparrow_{{\rm hw},BH}+\gamma_BR^\uparrow_{{\rm hw},BB}\right\}",
            r"\Delta T_H\le\Delta T,\qquad \Delta T_B\le\Delta T",
        ]:
            display(Math(line))
        display(Markdown(
            "The hardware-aware tab uses the same hotspot and bulk constraints as the reduced-order chip model, but evaluates them with R_ext,hw from the cold-plate model."
        ))

        display(Markdown("#### MR power, heat rejection, and objective"))
        for line in [
            r"Q_{\rm MR}=\alpha_{\rm eff}s\,qA",
            r"P_{\rm MR}=\frac{Q_{\rm MR}}{COP_\mu}",
            r"Q_{\rm reject}=qA+P_{\rm transport}+P_{\rm MR}",
            r"P_{\rm wall}=P_{\rm transport}+P_{\rm reject}+P_{\rm MR}",
            r"COP_{\rm system}=\frac{qA}{P_{\rm wall}}",
        ]:
            display(Math(line))
        display(Markdown(
            "For s>1, the MR cold side is modeled as subambient local cooling; the plots report -DeltaT_MR when the MR region is below ambient."
        ))

        display(Markdown("#### Hardware-aware optimization problem"))
        for line in [
            r"\min_{\mathcal{G},\dot V,s}\left[P_{\rm transport}(\mathcal{G},\dot V)+P_{\rm reject}(q,\dot V,s)+P_{\rm MR}(q,s)\right]",
            r"\mathrm{s.t.}\quad \Delta T_H(q,\mathcal{G},\dot V,s)\le\Delta T,\quad \Delta T_B(q,\mathcal{G},\dot V,s)\le\Delta T",
            r"0\le s\le s_{\max},\qquad \Delta p\le\Delta p_{\max},\quad \dot V\le\dot V_{\max},\quad v\le v_{\max},\quad \Delta T_{\rm fluid}\le\Delta T_{{\rm fluid},\max}",
        ]:
            display(Math(line))
        display(Markdown(
            "In fixed-hardware mode, mathcal{G} is designed at q_target and then frozen; only the operating flow and MR fraction are optimized as q is swept. "
            "In design-envelope mode, mathcal{G} may be re-optimized at each q."
        ))
    return out


def chip_model_equations_arch_hardware_box():
    """Architecture-aware chip equations as coupled to hardware-aware cooling."""
    out = widgets.Output(layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px",
        margin="8px 0"
    ))
    with out:
        display(Markdown("### Architecture-aware chip model equations for hardware-aware cooling"))
        for line in [
            r"f(\lambda)=f_0\lambda^{1-a}",
            r"Q_{\rm dyn}(\lambda)=Q_{\rm dyn,0}\lambda^{1+b}",
            r"Q_{\rm leak}(V,T)=Q_{\rm leak,0}\left(\frac{V}{V_0}\right)\exp\left[\frac{\eta_D(V-V_0)}{nV_t}\right]",
            r"\alpha_{\rm spot}(\lambda)=(1-f_{\rm leak})\alpha_{\rm dyn,0}+f_{\rm leak}\alpha_{\rm leak,0}",
            r"\alpha_{\rm eff}=N_H\alpha_{\rm spot}(\lambda),\qquad \beta_{\rm eff}=N_H\beta_{\rm spot}",
            r"R_{\rm ext,hw}=R_{\rm contact}+R_{\rm conv}A+\frac{1}{2}\frac{\Delta T_{\rm fluid}}{q}",
            r"P_{\rm wall}=P_{\rm transport}+P_{\rm reject}+P_{\rm MR}",
        ]:
            display(Math(line))
        display(Markdown(
            "The architecture-aware chip model changes how alpha_spot is computed from DVFS, dynamic power, and leakage. "
            "The external cooling path in this tab still comes from the hardware-aware cold-plate model, not from R0/Rinf."
        ))
    return out


def chip_model_equations_arch_box():
    return eq_box("Architecture-aware chip model equations", [
        r"f(\lambda)=f_0\lambda^{1-a}",
        r"Q_{\rm dyn}(\lambda)=Q_{\rm dyn,0}\lambda^{1+b}",
        r"Q_{\rm leak}(V,T)=Q_{\rm leak,0}\left(\frac{V}{V_0}\right)\exp\left[\frac{\eta_D(V-V_0)}{nV_t}\right]",
        r"\alpha_{\rm spot}(\lambda)=(1-f_{\rm leak})\alpha_{\rm dyn,0}+f_{\rm leak}\alpha_{\rm leak,0}",
        r"\alpha_{\rm eff}=N_H\alpha_{\rm spot}(\lambda),\qquad \beta_{\rm eff}=N_H\beta_{\rm spot}",
        r"Q_{\rm MR}=\alpha_{\rm eff}s\,qA,\qquad P_{\rm MR}=Q_{\rm MR}/COP_\mu"
    ])

def make_chip_model_notes_box():
    """Return a fresh copy of the chip/hotspot preset notes for each tab/section."""
    return widgets.HTML(chip_preset_note.value)

def effective_fraction_readout_thermal():
    html = widgets.HTML(layout=widgets.Layout(width="900px"))
    def update(*_):
        try:
            N = getv(thermal, "N_H")
            a = getv(thermal, "alpha_tot")
            b = getv(thermal, "beta")
            html.value = (
                "<b>Derived effective hotspot fractions:</b> "
                f"alpha_spot = {a:.6g}, beta_spot = {b:.6g}, "
                f"alpha_eff = N_H alpha_spot = {N*a:.6g}, "
                f"beta_eff = N_H beta_spot = {N*b:.6g}"
            )
        except Exception as exc:
            html.value = f"<b>Derived effective hotspot fractions:</b> unavailable ({exc})"
    for k in ["N_H", "alpha_tot", "beta"]:
        thermal[k]["text"].observe(update, names="value")
    update()
    return html

def effective_fraction_readout_arch():
    html = widgets.HTML(layout=widgets.Layout(width="900px"))
    def update(*_):
        try:
            bu = BUParams(
                V0=getv(arch, "V0"), f0=getv(arch, "f0"),
                Q_dyn0=getv(arch, "Q_dyn0"), Q_leak0=getv(arch, "Q_leak0"),
                alpha_dyn0=getv(arch, "alpha_dyn0"), alpha_leak0=getv(arch, "alpha_leak0"),
                beta=getv(arch, "beta"), Vth=getv(arch, "Vth"), nu=getv(arch, "nu"),
                eta_D=getv(arch, "eta_D"), n=getv(arch, "sub_n"), V_t=getv(arch, "V_t"),
                a=getv(arch, "a"), b=getv(arch, "b")
            )
            st = bottom_up_state(getv(arch, "lam"), bu)
            N = getv(arch, "N_H")
            a = st["alpha_tot"]
            b = getv(arch, "beta")
            html.value = (
                "<b>Derived effective hotspot fractions:</b> "
                f"alpha_spot(lambda) = {a:.6g}, beta_spot = {b:.6g}, "
                f"alpha_eff = N_H alpha_spot = {N*a:.6g}, "
                f"beta_eff = N_H beta_spot = {N*b:.6g}"
            )
        except Exception as exc:
            html.value = f"<b>Derived effective hotspot fractions:</b> unavailable ({exc})"
    for k in ["N_H", "beta", "lam", "V0", "f0", "Q_dyn0", "Q_leak0", "alpha_dyn0", "alpha_leak0", "Vth", "nu", "eta_D", "sub_n", "V_t", "a", "b"]:
        arch[k]["text"].observe(update, names="value")
    update()
    return html



chip_preset_note = html_note(
    "Chip scenario preset notes",
    "<p><b>Multi-hotspot convention:</b> alpha_spot and beta_spot describe one representative hotspot. N_H is the number of thermally similar hotspots. The model uses alpha_eff = N_H alpha_spot and beta_eff = N_H beta_spot for gamma_H/gamma_B, while beta_spot is used for the local spreading-resistance matrix.</p>"
    "<table style='border-collapse:collapse;'>"
    "<tr><th style='padding:4px 10px'>Scenario</th><th style='padding:4px 10px'>A [mm²]</th><th style='padding:4px 10px'>R_die</th><th style='padding:4px 10px'>N_H</th><th style='padding:4px 10px'>alpha_spot</th><th style='padding:4px 10px'>beta_spot</th><th style='padding:4px 10px'>alpha_eff</th><th style='padding:4px 10px'>beta_eff</th><th style='padding:4px 10px'>gamma_H</th></tr>"
    "<tr><td>NAND / low-power memory</td><td>50</td><td>8</td><td>4</td><td>0.0025</td><td>0.025</td><td>0.01</td><td>0.10</td><td>0.1</td></tr>"
    "<tr><td>DRAM / commodity memory</td><td>80</td><td>7</td><td>4</td><td>0.005</td><td>0.020</td><td>0.02</td><td>0.08</td><td>0.25</td></tr>"
    "<tr><td>HBM stack</td><td>100</td><td>10</td><td>8</td><td>0.010</td><td>0.0025</td><td>0.08</td><td>0.02</td><td>4</td></tr>"
    "<tr><td>Mobile SoC</td><td>100</td><td>8</td><td>2</td><td>0.025</td><td>0.005</td><td>0.05</td><td>0.01</td><td>5</td></tr>"
    "<tr><td>Server CPU</td><td>300</td><td>5</td><td>8</td><td>0.010</td><td>0.001</td><td>0.08</td><td>0.008</td><td>10</td></tr>"
    "<tr><td>GPU / AI accelerator distributed</td><td>800</td><td>4</td><td>10</td><td>0.015</td><td>0.003</td><td>0.15</td><td>0.03</td><td>5</td></tr>"
    "<tr><td>3D logic-on-logic / stacked compute</td><td>200</td><td>2</td><td>4</td><td>0.0625</td><td>0.00075</td><td>0.25</td><td>0.003</td><td>83</td></tr>"    
    "<tr><td>Chiplet / 2.5D HPC package</td><td>600</td><td>4</td><td>4</td><td>0.05</td><td>0.00125</td><td>0.20</td><td>0.005</td><td>40</td></tr>"    
    "<tr><td colspan='9' style='padding:6px 10px; background:#dddddd; font-weight:bold; text-align:center;'>Localized hotspot archetype presets</td></tr>"
    "<tr><td>Localized tensor-core cluster</td><td>800</td><td>4</td><td>1</td><td>0.15</td><td>0.005</td><td>0.15</td><td>0.005</td><td>30</td></tr>"
    "<tr><td>HBM PHY hotspot</td><td>800</td><td>4</td><td>4</td><td>0.025</td><td>0.0015</td><td>0.10</td><td>0.006</td><td>16.7</td></tr>"
    "<tr><td>Power-delivery hotspot</td><td>800</td><td>4</td><td>2</td><td>0.06</td><td>0.0015</td><td>0.12</td><td>0.003</td><td>40</td></tr>"
    "<tr><td>Stacked-cache hotspot</td><td>300</td><td>3</td><td>4</td><td>0.025</td><td>0.002</td><td>0.10</td><td>0.008</td><td>12.5</td></tr>"
    "<tr><td>Localized RF / power block</td><td>25</td><td>3</td><td>1</td><td>0.25</td><td>0.002</td><td>0.25</td><td>0.002</td><td>125</td></tr>"
    "</table>"
    "<p><b>Interpretation:</b> chip presets represent whole-chip/package operating points. Hotspot archetype presets represent localized thermally dominant structures that may exist within a larger chip. Increasing N_H increases total MR burden and increases the effective thermally active area, but the local spreading penalty is still set by beta_spot.</p>"
)

# ---------------------------
# Build and run model
# ---------------------------
output_thermal = widgets.Output()
output_arch = widgets.Output()
# Legacy alias; run() now chooses the active output explicitly.
output = output_thermal
replot_btn_thermal = widgets.Button(description="Replot full hybrid optimum", button_style="success", icon="refresh")
replot_btn_arch = widgets.Button(description="Replot full hybrid optimum", button_style="success", icon="refresh")

def active_groups():
    if mode.value == "Compact / reduced-order model":
        return thermal, plot_thermal
    return arch, plot_arch

def build_params():
    group, plot_group = active_groups()
    state = None

    # Multi-hotspot convention:
    #   alpha_spot = power fraction in one representative hotspot
    #   beta_spot  = area fraction of one representative hotspot
    #   N_H        = number of thermally similar hotspots
    # The compact two-region thermal model uses:
    #   alpha_eff = N_H * alpha_spot and beta_eff = N_H * beta_spot for gamma_H/gamma_B,
    #   beta_spot for the local spreading-resistance terms.
    N_H = int(round(getv(group, "N_H")))
    beta_spot = getv(group, "beta")

    if mode.value == "Compact / reduced-order model":
        alpha_spot = getv(group, "alpha_tot")
    else:
        bu = BUParams(
            V0=getv(group, "V0"), f0=getv(group, "f0"),
            Q_dyn0=getv(group, "Q_dyn0"), Q_leak0=getv(group, "Q_leak0"),
            alpha_dyn0=getv(group, "alpha_dyn0"), alpha_leak0=getv(group, "alpha_leak0"),
            beta=beta_spot, Vth=getv(group, "Vth"), nu=getv(group, "nu"),
            eta_D=getv(group, "eta_D"), n=getv(group, "sub_n"), V_t=getv(group, "V_t"),
            a=getv(group, "a"), b=getv(group, "b")
        )
        state = bottom_up_state(getv(group, "lam"), bu)
        alpha_spot = state["alpha_tot"]

    alpha_eff = N_H * alpha_spot
    beta_eff = N_H * beta_spot

    if not (0.0 < beta_eff < 1.0):
        raise ValueError(f"Invalid multi-hotspot area: N_H*beta_spot = {beta_eff:.4g}; must be between 0 and 1.")
    if not (0.0 < alpha_eff < 1.0):
        raise ValueError(f"Invalid multi-hotspot power: N_H*alpha_spot = {alpha_eff:.4g}; must be between 0 and 1.")

    pump = PumpParams(
        p_static=getv(group, "p_static"), p_scale=getv(group, "p_scale"),
        n=getv(group, "pump_n"), r=getv(group, "pump_r")
    )
    tp = ThermalParams(
        DeltaT=getv(group, "DeltaT"), A_mm2=getv(group, "A_mm2"),
        beta=beta_eff, beta_spot=beta_spot, alpha_tot=alpha_eff,
        R_die=getv(group, "R_die"), R0=getv(group, "R0"), R_inf=getv(group, "R_inf"),
        m=getv(group, "m"), chi=getv(group, "chi"), p_sp=getv(group, "p_sp"),
        rho_c=getv(group, "rho_c"), xi=getv(group, "xi"),
        COP_micro=getv(group, "COP_micro"), pump=pump
    )
    if plot_group["normalize"].value:
        tp = normalize_pump(tp, target_peak=1.0)

    meta = dict(N_H=N_H, alpha_spot=alpha_spot, beta_spot=beta_spot, alpha_eff=alpha_eff, beta_eff=beta_eff)
    return tp, state, plot_group, meta


def add_hotspot_top_axis(ax, gamma_H):
    """Add visible top x-axis for hotspot-average heat flux q_H = gamma_H q."""
    secax = ax.secondary_xaxis(
        "top",
        functions=(lambda q: gamma_H*q, lambda qH: qH/gamma_H if gamma_H != 0 else qH),
    )
    secax.set_xlabel(r"$q_H$ (hotspot flux) [W/mm$^2$]", labelpad=8)
    secax.tick_params(axis="x", which="both", direction="out", pad=2)
    return secax

def run(_=None):
    # Keep plotting modes visually isolated.
    # Compact / reduced-order and architecture-aware plots live inside their own tabs,
    # while the cold-plate hardware-envelope plots live only in the cold-plate tab.
    try:
        cp_output.clear_output(wait=True)
    except NameError:
        pass

    current_output = output_thermal if mode.value == "Compact / reduced-order model" else output_arch
    inactive_output = output_arch if mode.value == "Compact / reduced-order model" else output_thermal
    try:
        inactive_output.clear_output(wait=True)
    except Exception:
        pass

    with current_output:
        clear_output(wait=True)
        try:
            tp, state, pc, meta = build_params()
            walls = q_walls(tp)
            gamma_H, gamma_B = gammas(tp)
            q0 = walls["q0"]
            q0B = q0_bulk_passive(tp)
            qinfB = qinf_bulk_passive(tp)
            x_plot_max = getv(pc, "xmax")
            q = np.linspace(0.001*q0, x_plot_max*q0, 700)
            hyb = solve_curve(q, tp)
            conv = convection_curve(q, tp)
            x = q

            fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

            axs[0,0].plot(x, hyb["COP_total"], lw=2.5, label="Hybrid full optimum")
            axs[0,0].plot(x, conv["COP_total"], "k--", lw=2, label="Convection only")
            axs[0,0].set_title("Total COP")
            axs[0,0].set_ylabel("COP_total [-]")
            axs[0,0].set_ylim(0, getv(pc, "cop_ymax"))

            axs[0,1].plot(x, hyb["COP_active"], lw=2.5, label="Hybrid full optimum")
            axs[0,1].plot(x, conv["COP_active"], "k--", lw=2, label="Convection only")
            axs[0,1].set_title("Active COP = (Q-Q0,H)/Pcool")
            axs[0,1].set_ylabel("COP_active [-]")
            axs[0,1].set_ylim(0, getv(pc, "cop_ymax"))

            axs[1,0].plot(x, hyb["pump_burden"], lw=2, label="Hybrid pump")
            axs[1,0].plot(x, hyb["micro_burden"], lw=2, label="MR")
            axs[1,0].plot(x, hyb["total_burden"], lw=2.5, label="Hybrid total")
            axs[1,0].plot(x, np.minimum(conv["burden"], getv(pc, "clip_conv_burden")), "k--", lw=2, label="Conv burden clipped")
            axs[1,0].set_title("Power burden")
            axs[1,0].set_ylabel("Pcool/Q [-]")
            axs[1,0].set_ylim(0, getv(pc, "burden_ymax"))

            axs[1,1].plot(x, hyb["u"], lw=2, label="u*")
            axs[1,1].plot(x, hyb["s"], lw=2, label="s*")
            axs[1,1].set_title("Optimal variables")
            axs[1,1].set_ylabel("u*, s* [-]")
            axs[1,1].set_ylim(0, getv(pc, "var_ymax"))

            top_axes = []
            for ax in axs.flat:
                ax.axvline(q0, color="gray", ls="-", lw=1, label="q0,H")
                ax.axvline(q0B, color="tab:purple", ls="-.", lw=1.2, label="q0,B")
                ax.axvline(walls["qinf"], color="gray", ls="--", lw=1, label="qinf,H")
                ax.axvline(walls["qdown"], color="gray", ls=":", lw=1, label="qdie,H")
                ax.set_xlim(0, x_plot_max*q0)
                ax.autoscale(enable=False, axis="x")
                ax.set_xlabel("Chip heat flux q [W/mm²]")
                ax.tick_params(axis="x", which="both", bottom=True, labelbottom=True)
                top_axes.append(add_hotspot_top_axis(ax, gamma_H))
                ax.grid(True, alpha=0.3)
                ax.legend(fontsize=8)

            fig.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()

            unique, counts = np.unique(hyb["branch"], return_counts=True)
            print("--- Inputs actually used ---")
            print(f"cooling preset = {selected_cooling_preset_name}")
            print(f"chip preset = {selected_chip_preset_name}")
            print(f"hotspot archetype preset = {selected_hotspot_preset_name}")
            if selected_cooling_preset_name is not None:
                print(f"cooling preset = {selected_cooling_preset_name}")
            if selected_chip_preset_name is not None:
                print(f"chip preset = {selected_chip_preset_name}")
            if selected_hotspot_preset_name is not None:
                print(f"hotspot archetype preset = {selected_hotspot_preset_name}")
            print(f"chip model = {mode.label}")
            print("optimization = full hybrid optimum only")
            print(f"N_H = {meta['N_H']}; alpha_spot = {meta['alpha_spot']:.6g}; beta_spot = {meta['beta_spot']:.6g}; alpha_eff = {meta['alpha_eff']:.6g}; beta_eff = {meta['beta_eff']:.6g}")
            print(f"alpha_tot = {tp.alpha_tot:.6g}, beta = {tp.beta:.6g}, gamma_H = {tp.alpha_tot/tp.beta:.6g}")
            print(f"COP_micro = {tp.COP_micro:.6g}, DeltaT = {tp.DeltaT:.6g} K, A = {tp.A_mm2:.6g} mm^2")
            print(f"q0,H(avg) = {walls['q0']:.6g} W/mm^2, q0,B(avg) = {q0B:.6g} W/mm^2, qinf,H(avg) = {walls['qinf']:.6g} W/mm^2, qinf,B(avg) = {qinfB:.6g} W/mm^2")
            print(f"qH,0 = {(tp.alpha_tot/tp.beta)*walls['q0']:.6g} W/mm^2, qB,0 = {((1-tp.alpha_tot)/(1-tp.beta))*q0B:.6g} W/mm^2, qH,inf = {(tp.alpha_tot/tp.beta)*walls['qinf']:.6g} W/mm^2, qB,inf = {((1-tp.alpha_tot)/(1-tp.beta))*qinfB:.6g} W/mm^2")
            print(f"Q0,H = {walls['q0']*tp.A_mm2:.6g} W, Q0,B = {q0B*tp.A_mm2:.6g} W, Qinf,H = {walls['qinf']*tp.A_mm2:.6g} W, Qinf,B = {qinfB*tp.A_mm2:.6g} W")
            print(f"Q_H,0 = {(tp.alpha_tot/tp.beta)*walls['q0']*tp.A_mm2:.6g} W, Q_B,0 = {((1-tp.alpha_tot)/(1-tp.beta))*q0B*tp.A_mm2:.6g} W, Q_H,inf = {(tp.alpha_tot/tp.beta)*walls['qinf']*tp.A_mm2:.6g} W, Q_B,inf = {((1-tp.alpha_tot)/(1-tp.beta))*qinfB*tp.A_mm2:.6g} W")
            print(f"branch counts = {dict(zip(unique, counts))}")
            print(f"pump: p_static = {tp.pump.p_static:.6g} W/mm^2, p_scale = {tp.pump.p_scale:.6g} W/mm^2, n = {tp.pump.n:.6g}, r = {tp.pump.r:.6g}")
            if state is not None:
                print("\n--- Architecture-aware electrothermal state used ---")
                print(f"lambda = {state['lam']:.6g}; V = {state['V']:.6g} V; f = {state['f']:.6g} GHz")
                print(f"Q_dyn = {state['Q_dyn']:.6g} W; Q_leak = {state['Q_leak']:.6g} W; Q_total = {state['Q_total']:.6g} W")
                print(f"f_leak = {state['f_leak']:.6g}; alpha_tot(lambda) = {state['alpha_tot']:.6g}; gamma_H = {state['gamma_H']:.6g}")
        except Exception as err:
            print("Error:", repr(err))

def set_mode_and_run(new_mode):
    mode.value = new_mode
    run()

replot_btn_thermal.on_click(lambda _: set_mode_and_run("Compact / reduced-order model"))
replot_btn_arch.on_click(lambda _: set_mode_and_run("Architecture-aware electrothermal model"))

# ---------------------------
# Equation/documentation blocks
# ---------------------------
thermal_eqs = [
    r"q=Q/A,\quad A\,[\mathrm{mm^2}],\quad q\,[\mathrm{W/mm^2}]",
    r"\alpha_{\rm eff}=Q_H/Q,\quad \beta_{\rm eff}=A_H/A,\quad \gamma_H=\alpha_{\rm eff}/\beta_{\rm eff},\quad \gamma_B=(1-\alpha_{\rm eff})/(1-\beta_{\rm eff})",
    r"R_{\rm ext}(u)=R_\infty+\dfrac{R_0-R_\infty}{1+u^m}",
    r"R_{\rm sp,H}=\chi(\beta_{\rm spot}^{-p_{\rm sp}}-1),\quad R_{\rm sp,B}=\chi((1-\beta_{\rm spot})^{-p_{\rm sp}}-1),\quad R_c=\rho_c\sqrt{R_{\rm sp,H}R_{\rm sp,B}}",
    r"R^\downarrow=R_{\rm die}I+(1-\xi)\begin{bmatrix}R_{\rm sp,H}&R_c\\R_c&R_{\rm sp,B}\end{bmatrix}",
    r"R^\uparrow(u)=\xi\begin{bmatrix}R_{\rm sp,H}&R_c\\R_c&R_{\rm sp,B}\end{bmatrix}+R_{\rm ext}(u)I",
    r"\Phi(q)=\Delta T/q-\gamma_HR^\downarrow_{HH}-\gamma_BR^\downarrow_{HB}",
    r"s_{\rm req}(u,q)=\max\left[0,1-\dfrac{\Phi(q)-\gamma_BR^\uparrow_{HB}}{\gamma_HR^\uparrow_{HH}(u)}\right]",
    r"\Delta T_H=q\{\gamma_HR^\downarrow_{HH}+\gamma_BR^\downarrow_{HB}+\gamma_H(1-s)R^\uparrow_{HH}(u)+\gamma_BR^\uparrow_{HB}\}",
    r"\Delta T_B=q\{\gamma_HR^\downarrow_{BH}+\gamma_BR^\downarrow_{BB}+\gamma_H(1-s)R^\uparrow_{BH}+\gamma_BR^\uparrow_{BB}(u)\}",
    r"p_{\rm pump}(u)=p_{\rm static}+p_{\rm scale}u^{n_{\rm pump}}\left(\dfrac{1+u^2}{2u}\right)^{r_{\rm pump}}\quad (u>0)",
    r"p_{\rm cool}=p_{\rm pump}(u)+\alpha_{\rm tot}s q/COP_\mu",
    r"COP_{\rm total}=q/p_{\rm cool},\quad COP_{\rm active}=(q-q_{0,H})/p_{\rm cool}",
    r"q_{0,H}=\dfrac{\Delta T}{\gamma_H(R^\downarrow_{HH}+R^\uparrow_{HH}(0))+\gamma_B(R^\downarrow_{HB}+R^\uparrow_{HB})}",
    r"q_{0,B}=\dfrac{\Delta T}{\gamma_H(R^\downarrow_{BH}+R^\uparrow_{BH})+\gamma_B(R^\downarrow_{BB}+R^\uparrow_{BB}(0))}",
    r"q_{\infty,H}=\dfrac{\Delta T}{\gamma_H(R^\downarrow_{HH}+R^\uparrow_{HH}(\infty))+\gamma_B(R^\downarrow_{HB}+R^\uparrow_{HB})}",
    r"q_{\rm die,H}=\dfrac{\Delta T}{\gamma_HR^\downarrow_{HH}+\gamma_B(R^\downarrow_{HB}+R^\uparrow_{HB})}",
    r"\min_{u,s}\;p_{\rm cool}\quad\mathrm{s.t.}\quad \Delta T_H\le\Delta T,\;\Delta T_B\le\Delta T"
]
arch_eqs = [
    r"f(\lambda)=f_0\lambda^{1-a}",
    r"\lambda^{1-a}=\dfrac{V_0}{V}\left(\dfrac{V-V_{\rm th}}{V_0-V_{\rm th}}\right)^\nu",
    r"Q_{\rm dyn}=Q_{\rm dyn,0}\lambda(V/V_0)^2",
    r"Q_{\rm leak}=Q_{\rm leak,0}\lambda^b(V/V_0)\exp[\eta_D(V-V_0)/(nV_t)]",
    r"f_{\rm leak}=Q_{\rm leak}/(Q_{\rm dyn}+Q_{\rm leak})",
    r"\alpha_{\rm tot}(\lambda)=(1-f_{\rm leak})\alpha_{\rm dyn,0}+f_{\rm leak}\alpha_{\rm leak,0}"
]
preset_eqs = [
    r"\mathrm{Preset\ thermal\ values\ set\ }R_0,R_\infty,R_{\rm die},\chi,\rho_c,\xi,m.",
    r"\mathrm{Pump/fan\ parasitics\ are\ calibrated\ using\ two\ conditions:}\ COP(u_*)=COP_{\rm target},\quad dCOP/du|_{u_*}=0.",
    r"q_{0,\rm ref}=\Delta T/(R_{\rm die}+R_0),\quad q_{\infty,\rm ref}=\Delta T/(R_{\rm die}+R_\infty)",
    r"COP_{\rm conv,active}^{\rm ref}=\max_u\frac{\Delta T/(R_{\rm die}+R_{\rm ext}(u))-q_{0,\rm ref}}{p_{\rm pump}(u)}",
    r"\mathrm{Targets:\ forced\ air}\approx15,\quad \mathrm{conduction\ rugged}\approx30,\quad \mathrm{D2C\ cold\ plate}\approx75,\quad \mathrm{microstructured}\approx200,\quad \mathrm{embedded}\approx3000.",
    r"1\,\mathrm{W/cm^2}=0.01\,\mathrm{W/mm^2};\quad 3000\,\mathrm{W/cm^2}=30\,\mathrm{W/mm^2}."
]
plot_eqs = [
    r"x=q\quad[\mathrm{W/mm^2}]",
    r"\mathrm{Convection\ burden\ clipping\ is\ visual\ only.}",
    r"\mathrm{Textbox\ values\ are\ used\ by\ the\ model;\ they\ may\ exceed\ slider\ limits.}"
]



display(widgets.HTML("""
<style>
.chip-note-box {
    background-color: #fff8dc !important;
    border-radius: 6px !important;
}
.chip-note-box .jp-OutputArea-output {
    background-color: #fff8dc !important;
}
</style>
"""))

def cooling_scenario_note():
    out = widgets.Output(layout=widgets.Layout(
        border="1px solid #d8c07a",
        padding="10px 14px",
        margin="8px 0",
        width="100%"
    ))
    out.add_class("chip-note-box")
    with out:
        display(Markdown("### Cooling scenario preset notes"))
        display(Markdown(
            "- **Passive spreader / vapor chamber:** mobile/passive thermal stacks such as graphite sheets, frames, and vapor chambers. No pumped cold plate is implied; pump burden is approximately zero.\n- **Forced-air heatsink:** real fan + heatsink or vapor-chamber package cooling, not passive air. "
            "The preset targets roughly 50–60 W/cm² uniform heat-flux capability at ΔT≈60 K and fan COP≈15.\n"
            "- **Conduction-cooled ruggedized chassis:** sealed or low-airflow rugged compute where heat is spread through a conduction card, cold wall, chassis, or limited-flow loop. This sits between passive/forced-air spreading and direct-to-chip liquid cold plates; preset target COP≈30.\n- **Direct-to-chip liquid cold plate:** server/HPC cold plates, targeting roughly 300 W/cm² uniform heat-flux capability and pump/CDU COP≈75.\n"
            "- **Advanced microstructured cold plate:** microjet/short-channel/microconvective cold plates, targeting roughly 500–600 W/cm² uniform heat flux and COP≈200.\n"
            "- **Embedded microfluidics:** near-junction/backside embedded microfluidics, targeting 3000 W/cm² = 30 W/mm² class heat flux with very low pumping power; COP target≈3000.\n"
            "- **Separation convention:** Cooling presets set the external cooling path and pump/fan parameters. They keep R_die = 5 K mm²/W for non-embedded hardware. Chip presets set A and R_die. Embedded microfluidics is the exception because near-junction/backside integration can reduce the effective die-side resistance.\n- All pump coefficients are calibrated using the uniform-load hardware reference, then the user's actual hotspot α and β are restored."
        ))
        display(Markdown("#### Preset calibration philosophy"))
        display(Markdown("The cooling preset is calibrated on a uniform-load reference by solving for p_static and p_scale from two conditions: the target peak active COP and the prescribed flow location u_peak_target where that peak occurs (the optimal pump operating point). The user-selected hotspot parameters are restored afterward."))
        display(Markdown("""
#### Cooling preset calibration parameters

<table style="border-collapse:collapse; margin-bottom:10px">
<tr>
<th style="padding:4px 10px">Cooling preset</th>
<th style="padding:4px 10px">n_pump</th>
<th style="padding:4px 10px">r_pump</th>
<th style="padding:4px 10px">Peak COP_conv,active</th>
<th style="padding:4px 10px">u*</th>
</tr>
<tr><td>Passive spreader / vapor chamber</td><td>2.0</td><td>0.0</td><td>10000</td><td>0.05</td></tr>
<tr><td>Forced-air heatsink</td><td>3.0</td><td>0.0</td><td>15</td><td>1.0</td></tr>
<tr><td>Conduction-cooled ruggedized chassis</td><td>2.5</td><td>0.5</td><td>30</td><td>0.6</td></tr>
<tr><td>Direct-to-chip liquid cold plate</td><td>3.0</td><td>1.0</td><td>75</td><td>2.0</td></tr>
<tr><td>Advanced microstructured cold plate</td><td>2.5</td><td>0.5</td><td>200</td><td>3.0</td></tr>
<tr><td>Embedded microfluidics</td><td>2.0</td><td>0.5</td><td>3000</td><td>6.0</td></tr>
</table>

Each cooling preset specifies the thermal transport curve through R0, Rinf, and m; the pump/fan transport shape through n_pump and r_pump; the target peak active convection COP; and the target flow location u* where that peak occurs. The notebook solves analytically for p_static and p_scale such that COP_conv,active(u*) = COP_target and dCOP/du|_(u*) = 0.
"""))
        for line in [
            r"\alpha_{\rm ref}=\beta_{\rm ref}=1,\qquad \gamma_H=\gamma_B=1",
            r"q_{0,\rm ref}=\frac{\Delta T}{R_{\rm die}+R_0},\qquad q_{\rm ref}(u)=\frac{\Delta T}{R_{\rm die}+R_{\rm ext}(u)}",
            r"N(u)\equiv q_{\rm ref}(u)-q_{0,\rm ref}",
            r"R_{\rm ext}(u)=R_\infty+\frac{R_0-R_\infty}{1+u^m}",
            r"S(u)=u^{n_{\rm pump}}\left(\frac{1+u^2}{2u}\right)^{r_{\rm pump}}",
            r"p_{\rm pump}(u)=p_{\rm static}+p_{\rm scale}S(u)",
            r"COP_{\rm conv,active}^{\rm ref}(u_*)=\frac{N(u_*)}{p_{\rm static}+p_{\rm scale}S(u_*)}=COP_{\rm target}",
            r"\left.\frac{d}{du}COP_{\rm conv,active}^{\rm ref}(u)\right|_{u=u_*}=0",
            r"p_{\rm scale}=\frac{N'(u_*)}{COP_{\rm target}S'(u_*)},\qquad p_{\rm static}=\frac{N(u_*)}{COP_{\rm target}}-p_{\rm scale}S(u_*)",
        ]:
            display(Math(line))
    return out

source_note = cooling_scenario_note()



nomenclature_note = widgets.HTML("""
<div style='background:#eef7ff;border:1px solid #9ec9e8;padding:10px 14px;margin:8px 0;border-radius:6px'>
<b>Nomenclature used in this notebook</b><br>
<ul style='margin-top:6px'>
<li><b>Reduced-order cooling surrogate:</b> the simple external-cooling model using
R0, Rinf, m, p_static, and p_scale.</li>
<li><b>Hardware-aware cooling model:</b> the cold-plate / heatsink / heat-rejection model
using geometry, flow, pressure drop, pump/fan efficiency, and heat-rejection constraints.</li>
<li><b>Reduced-order chip model:</b> the chip/hotspot source model where alpha_spot,
beta_spot, and N_H are entered directly.</li>
<li><b>Architecture-aware chip model:</b> the chip/hotspot source model where
hotspot power fraction is computed from DVFS, dynamic power, leakage, and architecture state.</li>
</ul>
</div>
""")

# ---------------------------
# Source-specific input menus used inside both top-level tabs
# ---------------------------
reduced_order_source_menu = widgets.VBox([
    widgets.HTML("<h3>Reduced-order chip model</h3>"),
    widgets.HTML("<small>Chip/hotspot architecture is represented by direct parameters: A, R_die, N_H, alpha_spot, beta_spot, spreading parameters, and MR COP.</small>"),
    collapsed_section("Reduced-order chip model equations", chip_model_equations_reduced_box()),
    collapsed_section("Chip/hotspot preset notes and definitions", make_chip_model_notes_box()),
    chip_hotspot_selector_thermal("Reduced-order chip model"),
    widgets.HTML("<b>Hotspot fraction inputs</b>"),
    two_col([thermal[k]["box"] for k in ["N_H", "alpha_tot", "beta", "COP_micro"]]),
    effective_fraction_readout_thermal(),
    widgets.HTML("<b>Chip/package and spreading inputs</b>"),
    two_col([thermal[k]["box"] for k in ["A_mm2", "DeltaT", "R_die", "chi", "p_sp", "rho_c", "xi"]]),
    widgets.HTML("<h3>Cooling model for reduced-order surrogate</h3>"),
    collapsed_section("Cooling surrogate preset notes and calibration equations", cooling_scenario_note()),
    cooling_preset_selector(thermal, plot_thermal, "reduced-order cooling surrogate"),
    collapsed_section("Plot / visual control equations", eq_box("Plot / visual controls", plot_eqs)),
    plot_thermal["normalize"],
    two_col([plot_thermal[k]["box"] for k in ["xmax", "cop_ymax", "burden_ymax", "var_ymax", "clip_conv_burden"]]),
    widgets.HTML("<b>Reduced-order cooling surrogate parameters</b>"),
    two_col([thermal[k]["box"] for k in ["R0", "R_inf", "m", "p_static", "p_scale", "pump_n", "pump_r"]]),
    replot_btn_thermal,
    output_thermal
])

architecture_source_menu = widgets.VBox([
    widgets.HTML("<h3>Architecture-aware chip model</h3>"),
    widgets.HTML("<small>Chip/hotspot architecture is computed from a DVFS/leakage/dynamic-power abstraction plus chip and hotspot presets.</small>"),
    collapsed_section("Architecture-aware chip model equations", chip_model_equations_arch_box()),
    collapsed_section("Chip/hotspot preset notes and definitions", make_chip_model_notes_box()),
    chip_hotspot_selector_arch("Architecture-aware chip model"),
    widgets.HTML("<b>Hotspot fraction inputs and derived fractions</b>"),
    two_col([arch[k]["box"] for k in ["N_H", "beta", "COP_micro", "lam"]]),
    effective_fraction_readout_arch(),
    widgets.HTML("<b>Chip/package and spreading inputs</b>"),
    two_col([arch[k]["box"] for k in ["A_mm2", "DeltaT", "R_die", "chi", "p_sp", "rho_c", "xi"]]),
    widgets.HTML("<b>Architecture and power-state inputs</b>"),
    two_col([arch[k]["box"] for k in [
        "V0", "f0", "Q_dyn0", "Q_leak0", "alpha_dyn0", "alpha_leak0",
        "Vth", "nu", "eta_D", "sub_n", "V_t", "a", "b"
    ]]),
    widgets.HTML("<h3>Cooling model for reduced-order surrogate</h3>"),
    collapsed_section("Cooling surrogate preset notes and calibration equations", cooling_scenario_note()),
    cooling_preset_selector(arch, plot_arch, "reduced-order cooling surrogate"),
    collapsed_section("Plot / visual control equations", eq_box("Plot / visual controls", plot_eqs)),
    plot_arch["normalize"],
    two_col([plot_arch[k]["box"] for k in ["xmax", "cop_ymax", "burden_ymax", "var_ymax", "clip_conv_burden"]]),
    widgets.HTML("<b>Reduced-order cooling surrogate parameters</b>"),
    two_col([arch[k]["box"] for k in ["R0", "R_inf", "m", "p_static", "p_scale", "pump_n", "pump_r"]]),
    replot_btn_arch,
    output_arch
])

# Backward-compatible names used by update_chip_source_menus().
thermal_tab = reduced_order_source_menu
arch_tab = architecture_source_menu

# ---------------------------
# Cold-plate hardware-envelope model
# ---------------------------
# This tab estimates a physically constrained cold-plate/heatsink envelope from
# geometry, flow, pressure drop, coolant temperature rise, and pump/fan electrical
# power. It is independent of the compact R0/Rinf preset model used in the main
# thermal tabs. The goal is to make COP_conv an output of hardware constraints,
# not a preset input.

COLD_PLATE_PRESETS = {
    "Passive spreader / vapor chamber + downstream reject": {
        "kind": "passive",
        "R_path_area": 320.0,       # K mm^2/W, spreader/enclosure-to-ambient equivalent path
        "R_contact_area": 20.0,     # K mm^2/W
        "P_aux_W": 0.0,
        "downstream_reject_allowed": True,
        "provenance": "screening envelope; passive mobile spreader/vapor chamber class; downstream heat rejection optional",
        "calibration_status": "heuristic_screening_with_physical_limits",
        "source_tags": "Passive thermal stack; downstream reject model can represent enclosure/radiator/facility sink",
        "notes": "Passive spreader/vapor chamber plus optional downstream heat rejection. No local pump/fan is included unless heat rejection mode adds downstream wall power."
    },
    "Forced-air finned heatsink with fan curve": {
        "kind": "air_channels",
        "fluid": "air",
        "area_factor": 15.0,
        "length_factor": 1.0,
        "w_mm": (1.0, 5.0),         # fin gap / channel spacing
        "h_mm": (10.0, 50.0),       # fin height
        "wall_mm": 0.8,             # fin thickness
        "dp_max": 250.0,            # Pa, system pressure envelope
        "v_max": 12.0,              # m/s, outlet/channel velocity envelope
        "flow_max": 0.05,           # m^3/s, 3000 L/min
        "dT_fluid_max": 35.0,       # K, air temperature rise
        "eta": 0.25,                # fan electro-aero efficiency
        "R_contact_area": 25.0,     # K mm^2/W
        "K_loss": 4.0,              # entrance/exit/duct/minor-loss lump
        "fin_eff": 0.75,
        "fan_curve_dp0": 300.0,     # Pa at zero flow
        "fan_curve_flow_free": 0.06,# m^3/s at zero pressure
        "noise_dBA_max": 55.0,      # acoustic screening limit
        "noise_dBA_ref": 38.0,
        "noise_flow_ref": 0.01,     # m^3/s
        "volume_L": 2.0,
        "volume_L_max": 3.0,
        "downstream_reject_allowed": False,
        "provenance": "screening envelope anchored to high-performance forced-air heatsink/fan constraints",
        "calibration_status": "heuristic_with_fan_curve_acoustic_volume_limits",
        "source_tags": "Fan curve, acoustic and volume constraints added; replace with vendor fan/system curve when available",
        "notes": "Forced-air finned heatsink with finite fan curve, acoustic limit, and volume envelope. Heat rejection is to local air through fan power, not a liquid chiller."
    },
    "Conduction-cooled rugged chassis / cold wall": {
        "kind": "conduction_chassis",
        "R_path_area": 105.0,       # K mm^2/W, chassis/cold-wall/heat-pipe path
        "R_contact_area": 15.0,     # K mm^2/W
        "R_heatpipe_area": 15.0,    # K mm^2/W extra spreading/heat-pipe path allowance
        "P_aux_W": 5.0,             # W, chassis fan/blower/cold-wall loop allowance
        "P_aux_max_W": 30.0,
        "downstream_reject_allowed": True,
        "provenance": "screening envelope for sealed/rugged conduction card, cold wall, heat pipe or limited-flow chassis loop",
        "calibration_status": "heuristic_screening_with_downstream_reject",
        "source_tags": "Rugged chassis conduction path; downstream facility loop/cold wall/fan burden can be represented by heat-rejection mode plus P_aux",
        "notes": "Rugged conduction-cooled chassis/cold-wall path. Includes contact, chassis/heat-pipe path and a small auxiliary fan/loop burden; downstream heat rejection can be enabled."
    },
    "Direct-to-chip single-phase cold plate": {
        "kind": "liquid_channels",
        "fluid": "water",
        "topology": "single_phase_parallel_channels",
        "area_factor": 1.5,
        "length_factor": 1.0,
        "w_mm": (0.8, 3.0),
        "h_mm": (0.5, 3.0),
        "wall_mm": 0.5,
        "dp_max": 50000.0,          # Pa
        "v_max": 2.5,               # m/s
        "flow_max": 8.0e-5,         # m^3/s, 4.8 L/min
        "dT_fluid_max": 15.0,       # K
        "eta": 0.35,
        "R_contact_area": 8.0,
        "K_loss": 6.0,
        "fin_eff": 1.0,
        "Nu_multiplier": 1.0,
        "area_multiplier": 1.0,
        "downstream_reject_allowed": True,
        "provenance": "OCP-style single-phase D2C screening envelope; flow/pressure/temperature limits are design constraints",
        "calibration_status": "literature_anchored_screening",
        "source_tags": "OCP cold-plate requirements emphasize pressure drop, flow, inlet/outlet temperature, active area, filtering; single-phase channel surrogate",
        "notes": "Single-phase direct-to-chip cold plate. Uses rectangular-channel equivalent geometry, pump pressure/flow limits, and downstream heat rejection."
    },
    "Direct-to-chip two-phase cold plate surrogate": {
        "kind": "liquid_channels",
        "fluid": "water",
        "topology": "two_phase_equivalent_surrogate",
        "area_factor": 1.5,
        "length_factor": 1.0,
        "w_mm": (0.5, 2.5),
        "h_mm": (0.3, 2.0),
        "wall_mm": 0.4,
        "dp_max": 80000.0,
        "v_max": 3.0,
        "flow_max": 6.0e-5,         # m^3/s, 3.6 L/min
        "dT_fluid_max": 8.0,
        "eta": 0.35,
        "R_contact_area": 5.0,
        "K_loss": 8.0,
        "fin_eff": 1.0,
        "Nu_multiplier": 2.0,
        "area_multiplier": 1.2,
        "two_phase_surrogate": True,
        "downstream_reject_allowed": True,
        "provenance": "two-phase D2C surrogate anchored to recent >300 W/cm^2 and multi-kW demonstrations",
        "calibration_status": "literature_anchored_surrogate_not_full_two_phase",
        "source_tags": "ACT two-phase D2C demo: >7.5 kW, >300 W/cm^2, 0.060 C cm^2/W, <0.8 LPM/kW; model remains equivalent-channel surrogate",
        "notes": "Two-phase direct-to-chip surrogate. This is not a full boiling/dryout model; it uses boosted heat transfer and tighter fluid-temperature rise as an equivalent envelope."
    },
    "Manifold / microstructured single-phase cold plate": {
        "kind": "liquid_channels",
        "fluid": "water",
        "topology": "manifold_microchannel_surrogate",
        "area_factor": 1.1,
        "length_factor": 0.65,
        "w_mm": (0.10, 0.60),
        "h_mm": (0.10, 0.80),
        "wall_mm": 0.075,
        "dp_max": 200000.0,
        "v_max": 6.0,
        "flow_max": 3.0e-5,         # m^3/s, 1.8 L/min
        "dT_fluid_max": 20.0,
        "eta": 0.30,
        "R_contact_area": 4.0,
        "K_loss": 10.0,
        "fin_eff": 1.0,
        "Nu_multiplier": 1.6,
        "area_multiplier": 1.5,
        "maldistribution_margin": 0.90,
        "downstream_reject_allowed": True,
        "provenance": "microchannel/manifold single-phase screening envelope",
        "calibration_status": "literature_anchored_screening_with_topology_multiplier",
        "source_tags": "Microchannel/manifold literature spans low-pressure optimized manifolds to high-pressure high-flux microchannels; topology-specific maps needed for final design",
        "notes": "Microstructured/manifold single-phase cold-plate surrogate. Uses shortened effective flow length, area multiplier, heat-transfer multiplier and maldistribution margin."
    },
    "Embedded microfluidic equivalent-channel surrogate": {
        "kind": "liquid_channels",
        "fluid": "water",
        "topology": "embedded_microfluidic_equivalent_channel",
        "area_factor": 1.0,
        "length_factor": 0.45,
        "w_mm": (0.040, 0.250),
        "h_mm": (0.050, 0.300),
        "wall_mm": 0.040,
        "dp_max": 500000.0,
        "v_max": 10.0,
        "flow_max": 1.5e-5,         # m^3/s, 0.9 L/min
        "dT_fluid_max": 30.0,
        "eta": 0.25,
        "R_contact_area": 1.0,
        "K_loss": 15.0,
        "fin_eff": 1.0,
        "Nu_multiplier": 2.5,
        "area_multiplier": 2.0,
        "maldistribution_margin": 0.85,
        "downstream_reject_allowed": True,
        "provenance": "research-grade embedded microfluidic equivalent-channel surrogate",
        "calibration_status": "literature_anchored_research_surrogate",
        "source_tags": "Nature Electronics 2025 embedded jet/manifold microchannels: up to 3000 W/cm^2 at 0.9 W/cm^2 pumping power; model is simplified equivalent-channel, not full 3-layer topology",
        "notes": "Embedded/backside microfluidic surrogate. Represents research-grade near-junction channels with heat-transfer and area multipliers; fabrication/reliability/manifold constraints remain simplified."
    },
}

def fluid_properties(name):
    if name == "air":
        return dict(rho=1.2, mu=1.85e-5, k=0.026, cp=1005.0, Pr=0.71)
    # water, near room temperature
    return dict(rho=997.0, mu=8.9e-4, k=0.60, cp=4180.0, Pr=6.2)


def gordon_ng_calibrated_params(Q_ref_kW, COP_ref, T_chw_ref_K, T_cw_ref_K, a1_kW_per_K=0.0, a3_K_per_kW=0.0):
    """Calibrate the Gordon-Ng/CPO-style a2 coefficient from one rated point.

    Model form, using kW and K:
        W_ch = ((Q_ch + a1*T_chw + a2*(1 - T_chw/T_cw))
                * T_cw/(T_chw - a3*Q_ch)) - Q_ch

    Given a rated point (Q_ref, COP_ref, T_chw_ref, T_cw_ref),
    W_ref = Q_ref/COP_ref.  The function solves a2 so the model exactly
    reproduces that rated COP at the rated temperatures and load.

    Units:
        Q_ref_kW: kW
        COP_ref: dimensionless
        T_chw_ref_K: K
        T_cw_ref_K: K
        a1_kW_per_K: kW/K
        a2 returned: kW
        a3_K_per_kW: K/kW
    """
    Q = max(float(Q_ref_kW), 1e-12)
    COP = max(float(COP_ref), 1e-12)
    Tch = float(T_chw_ref_K)
    Tcw = float(T_cw_ref_K)
    a1 = float(a1_kW_per_K)
    a3 = float(a3_K_per_kW)

    W = Q / COP
    B = 1.0 - Tch / Tcw
    denom = max(abs(B), 1e-12)
    a2 = (((W + Q) * (Tch - a3 * Q) / Tcw) - Q - a1 * Tch) / denom
    return float(a2)


def gordon_ng_chiller_power(Q_reject_W, reject_params, dT_fluid_K=None):
    """Heat-rejection / chiller wall power model.

    This function intentionally separates three system boundaries:

    1. Transport only / ideal downstream sink:
        P_reject = 0.
        Use this only as a cold-plate hydraulic transport diagnostic.

    2. Facility water / dry cooler / economizer:
        P_reject = Q_reject / COP_economizer.
        A simple feasibility check requires the coolant/air return temperature
        to be warm enough to reject to ambient with the specified approach:
            T_return >= T_amb + approach.
        If dT_fluid_K is unavailable, the model applies the finite COP but
        skips the return-temperature feasibility check.

    3. Gordon-Ng calibrated chiller:
        Uses the calibrated semi-empirical Gordon-Ng/CPO-style model.

    Inputs:
        Q_reject_W: scalar or array [W]
        reject_params: dictionary from cold-plate tab controls
        dT_fluid_K: optional coolant/air temperature rise through cold plate [K]

    Returns:
        P_reject_W, COP_reject, feasible_bool
    """
    Q_W = np.asarray(Q_reject_W, dtype=float)
    model = reject_params.get("model", "Transport only / ideal downstream sink")

    # Backward-compatible aliases.
    if model in ["Off", "Transport only / ideal downstream sink"]:
        return np.zeros_like(Q_W, dtype=float), np.full_like(Q_W, np.inf, dtype=float), np.ones_like(Q_W, dtype=bool)

    if model in ["Fixed COP", "Facility water / dry cooler / economizer"]:
        COP_econ = max(float(reject_params.get("economizer_COP", reject_params.get("fixed_COP", 30.0))), 1e-12)
        P = Q_W / COP_econ
        feasible = np.isfinite(P)

        # Optional above-ambient heat-rejection feasibility.
        # This is a simple screening constraint for dry coolers / facility water.
        if dT_fluid_K is not None:
            Tin_C = float(reject_params.get("T_supply_C", reject_params.get("T_chw_C", 25.0)))
            Tamb_C = float(reject_params.get("T_amb_C", 30.0))
            approach_K = float(reject_params.get("reject_approach_K", 8.0))
            T_return_C = Tin_C + np.asarray(dT_fluid_K, dtype=float)
            feasible = feasible & (T_return_C >= Tamb_C + approach_K)

        return P, np.full_like(Q_W, COP_econ, dtype=float), feasible

    # Calibrated Gordon-Ng/CPO-style semi-empirical chiller.
    # Backward-compatible name "Gordon-Ng calibrated" is also accepted.
    Q_kW = Q_W / 1000.0
    T_chw = float(reject_params["T_chw_C"]) + 273.15
    T_cw = float(reject_params["T_cw_C"]) + 273.15
    T_chw_ref = float(reject_params["T_chw_ref_C"]) + 273.15
    T_cw_ref = float(reject_params["T_cw_ref_C"]) + 273.15

    COP_ref = float(reject_params["COP_ref"])
    Q_ref_kW = float(reject_params["Q_ref_kW"])
    a1 = float(reject_params.get("a1_kW_per_K", 0.0))
    a3 = float(reject_params.get("a3_K_per_kW", 0.0))
    PLR_max = float(reject_params.get("PLR_max", 1.2))

    a2 = gordon_ng_calibrated_params(Q_ref_kW, COP_ref, T_chw_ref, T_cw_ref, a1, a3)

    denom = T_chw - a3 * Q_kW
    B = 1.0 - T_chw / T_cw
    with np.errstate(divide="ignore", invalid="ignore"):
        W_kW = ((Q_kW + a1*T_chw + a2*B) * T_cw / denom) - Q_kW
        COP = np.where(W_kW > 0, Q_kW / W_kW, np.nan)

    feasible = (
        np.isfinite(W_kW) &
        np.isfinite(COP) &
        (W_kW >= 0.0) &
        (Q_kW <= PLR_max * max(Q_ref_kW, 1e-12)) &
        (T_cw > T_chw) &
        (denom > 0.0)
    )

    P_W = 1000.0 * W_kW
    return P_W, COP, feasible


def coldplate_select_best_from_candidates(q_W_mm2, A_mm2, DeltaT, R_die_area, preset, reject_params, candidates, reason_if_empty="no feasible hardware candidate"):
    """Select the lowest-wall-power point from shared hardware candidates.

    This helper is used by the standalone cold-plate preset diagnostics.  The
    candidates are generated by hw_ext_resistance_temperature(), which is the
    same transport/geometry model used by the hardware-aware hybrid MR tab.
    """
    Q_W = q_W_mm2 * A_mm2
    if Q_W <= 0:
        return dict(feasible=True, P_wall=0.0, P_transport=0.0, P_reject=0.0, COP=np.inf, reason="zero load")

    if isinstance(candidates, dict):
        candidates = [candidates] if candidates.get("feasible", False) else []
    if not candidates:
        return dict(feasible=False, P_wall=np.nan, COP=np.nan, reason=reason_if_empty)

    best = None
    best_diag = None

    for cand in candidates:
        R_ext_area = cand.get("R_ext_area", np.nan)
        R_area = R_die_area + R_ext_area
        temp_rise = q_W_mm2 * R_area
        P_transport = cand.get("P_transport", 0.0)
        Q_reject = Q_W + P_transport
        P_reject, COP_reject, feasible_reject = gordon_ng_chiller_power(
            Q_reject, reject_params, dT_fluid_K=cand.get("dT_fluid", None)
        )
        P_reject = float(np.asarray(P_reject).ravel()[0])
        COP_reject = float(np.asarray(COP_reject).ravel()[0])
        feasible_reject = bool(np.asarray(feasible_reject).ravel()[0])

        P_wall = P_transport + P_reject
        feasible = (
            np.isfinite(temp_rise) and (temp_rise <= DeltaT) and
            np.isfinite(P_wall) and feasible_reject
        )

        row = dict(
            feasible=feasible,
            P_wall=float(P_wall) if np.isfinite(P_wall) else np.nan,
            P_transport=float(P_transport) if np.isfinite(P_transport) else np.nan,
            P_reject=float(P_reject) if np.isfinite(P_reject) else np.nan,
            COP=float(Q_W/P_wall) if feasible and P_wall > 0 else (np.inf if feasible and P_wall == 0 else np.nan),
            COP_reject=COP_reject,
            temp_rise=float(temp_rise) if np.isfinite(temp_rise) else np.nan,
            R_area=float(R_area) if np.isfinite(R_area) else np.nan,
            R_ext_area=float(R_ext_area) if np.isfinite(R_ext_area) else np.nan,
            reason="feasible" if feasible else "thermal or heat-rejection limit exceeded",
            **{k: cand.get(k, np.nan) for k in [
                "dp", "flow", "velocity", "Re", "Nu", "width_mm", "height_mm", "N_channels", "dT_fluid",
                "dp_util", "flow_util", "velocity_util", "fluid_dT_util"
            ]}
        )

        if feasible:
            if best is None or row["P_wall"] < best["P_wall"]:
                best = row

        # Keep a diagnostic row with the smallest max violation even if infeasible.
        temp_util = temp_rise/DeltaT if np.isfinite(temp_rise) and DeltaT > 0 else np.inf
        reject_util = 0.0 if feasible_reject else 2.0
        max_util = max(temp_util, reject_util)
        row["temp_util"] = float(temp_util) if np.isfinite(temp_util) else np.nan
        row["reject_util"] = float(reject_util)
        row["max_util"] = float(max_util) if np.isfinite(max_util) else np.nan
        if best_diag is None or row.get("max_util", np.inf) < best_diag.get("max_util", np.inf):
            best_diag = row

    if best is not None:
        return best
    best_diag["feasible"] = False
    best_diag["COP"] = np.nan
    return best_diag


def coldplate_channel_evaluate(q_W_mm2, A_mm2, DeltaT, R_die_area, preset, reject_params=None):
    """Grid-search the best feasible geometry/flow at one q.

    The geometry/flow candidates come from the SAME hardware model used by the
    hardware-aware hybrid MR tab: hw_ext_resistance_temperature().
    """
    if reject_params is None:
        reject_params = {"model": "Transport only / ideal downstream sink"}

    candidates = hw_ext_resistance_temperature(q_W_mm2, A_mm2, R_die_area, preset, design=None, v=None)
    return coldplate_select_best_from_candidates(
        q_W_mm2, A_mm2, DeltaT, R_die_area, preset, reject_params, candidates,
        reason_if_empty="no feasible geometry/flow candidate from shared hardware model"
    )


def coldplate_fixed_geometry_evaluate(q_W_mm2, A_mm2, DeltaT, R_die_area, preset, fixed_design, reject_params=None):
    """Evaluate off-design operation for a fixed cold-plate geometry.

    Geometry is frozen from fixed_design. Only flow velocity is optimized.  The
    hardware transport candidates are produced by the same shared model used in
    the hardware-aware hybrid MR tab.
    """
    if reject_params is None:
        reject_params = {"model": "Transport only / ideal downstream sink"}

    if preset["kind"] == "passive":
        return coldplate_channel_evaluate(q_W_mm2, A_mm2, DeltaT, R_die_area, preset, reject_params)

    if (not fixed_design) or (not fixed_design.get("feasible", False)):
        return dict(feasible=False, P_wall=np.nan, COP=np.nan, reason="no feasible fixed geometry")

    candidates = hw_ext_resistance_temperature(q_W_mm2, A_mm2, R_die_area, preset, design=fixed_design, v=None)
    return coldplate_select_best_from_candidates(
        q_W_mm2, A_mm2, DeltaT, R_die_area, preset, reject_params, candidates,
        reason_if_empty="no feasible flow candidate through fixed geometry"
    )

def run_coldplate_envelope(_=None):
    # Keep plotting modes visually isolated: when the cold-plate
    # hardware-envelope model is plotted, clear both main thermal/MR outputs.
    try:
        output_thermal.clear_output(wait=True)
        output_arch.clear_output(wait=True)
    except NameError:
        pass

    with cp_output:
        clear_output(wait=True)

        preset_name = cp_preset.value
        preset = COLD_PLATE_PRESETS[preset_name]
        sweep_mode = cp_sweep_mode.value

        A_mm2 = cp_controls["A_mm2"]["text"].value
        DeltaT = cp_controls["DeltaT"]["text"].value
        R_die = cp_controls["R_die"]["text"].value
        qmax = cp_controls["qmax"]["text"].value
        q_target = cp_controls["qtarget"]["text"].value

        reject_params = dict(
            model=cp_reject_model.value,
            economizer_COP=cp_controls["economizer_COP"]["text"].value,
            T_supply_C=cp_controls["T_supply_C"]["text"].value,
            T_amb_C=cp_controls["T_amb_C"]["text"].value,
            reject_approach_K=cp_controls["reject_approach_K"]["text"].value,
            T_chw_C=cp_controls["T_chw_C"]["text"].value,
            T_cw_C=cp_controls["T_cw_C"]["text"].value,
            T_chw_ref_C=cp_controls["T_chw_ref_C"]["text"].value,
            T_cw_ref_C=cp_controls["T_cw_ref_C"]["text"].value,
            COP_ref=cp_controls["GN_COP_ref"]["text"].value,
            Q_ref_kW=cp_controls["GN_Q_ref_kW"]["text"].value,
            PLR_max=cp_controls["GN_PLR_max"]["text"].value,
            a1_kW_per_K=cp_controls["GN_a1"]["text"].value,
            a3_K_per_kW=cp_controls["GN_a3"]["text"].value,
        )

        q_grid = np.linspace(max(1e-5, qmax/80), qmax, 80)

        # Design-envelope mode:
        #   Redesign/reselect geometry independently at each q.
        #
        # Fixed-hardware mode:
        #   First design the cold plate at q_target, then freeze that geometry
        #   and optimize only flow velocity as q is swept.
        fixed_design = None
        if sweep_mode.startswith("Fixed"):
            fixed_design = coldplate_channel_evaluate(q_target, A_mm2, DeltaT, R_die, preset, reject_params)
            results = [
                coldplate_fixed_geometry_evaluate(q, A_mm2, DeltaT, R_die, preset, fixed_design, reject_params)
                for q in q_grid
            ]
            rtar = coldplate_fixed_geometry_evaluate(q_target, A_mm2, DeltaT, R_die, preset, fixed_design, reject_params)
        else:
            results = [
                coldplate_channel_evaluate(q, A_mm2, DeltaT, R_die, preset, reject_params)
                for q in q_grid
            ]
            rtar = coldplate_channel_evaluate(q_target, A_mm2, DeltaT, R_die, preset, reject_params)

        COP = np.array([r.get("COP", np.nan) for r in results], dtype=float)
        Pwall = np.array([r.get("P_wall", np.nan) for r in results], dtype=float)
        Ptransport = np.array([r.get("P_transport", np.nan) for r in results], dtype=float)
        Preject = np.array([r.get("P_reject", np.nan) for r in results], dtype=float)
        COP_reject = np.array([r.get("COP_reject", np.nan) for r in results], dtype=float)
        Trise = np.array([r.get("temp_rise", np.nan) for r in results], dtype=float)
        dp = np.array([r.get("dp", np.nan) for r in results], dtype=float)
        flow_lpm = np.array([r.get("flow", np.nan) for r in results], dtype=float)*60000.0
        feasible = np.array([r.get("feasible", False) for r in results], dtype=bool)

        display(Markdown(f"### Cold plate hardware-envelope preset: {preset_name}"))
        display(Markdown(f"**Sweep mode:** {sweep_mode}"))
        display(Markdown(preset["notes"]))

        if fixed_design is not None:
            if fixed_design.get("feasible", False) and preset["kind"] != "passive":
                display(Markdown(
                    f"**Frozen design at q_target={q_target:.4g} W/mm²:** "
                    f"N={fixed_design['N_channels']}, width={fixed_design['width_mm']:.4g} mm, "
                    f"height={fixed_design['height_mm']:.4g} mm. "
                    "The sweep then changes only flow velocity / operating point."
                ))
            elif preset["kind"] == "passive":
                display(Markdown("**Passive/conduction preset:** fixed-design and design-envelope modes are identical because there is no flow geometry to optimize."))
            else:
                display(Markdown(f"**No feasible frozen design at q_target={q_target:.4g} W/mm².**"))

        limits_html = f"""
        <table style='border-collapse:collapse; margin:8px 0'>
        <tr><th style='padding:4px 10px'>Parameter</th><th style='padding:4px 10px'>Value</th></tr>
        <tr><td>Architecture kind</td><td>{preset['kind']}</td></tr>
        <tr><td>Allowed ΔT</td><td>{DeltaT:.3g} K</td></tr>
        <tr><td>Chip area A</td><td>{A_mm2:.3g} mm²</td></tr>
        <tr><td>R_die</td><td>{R_die:.3g} K mm²/W</td></tr>
        <tr><td>K_loss / R_contact</td><td>{preset.get('K_loss', '—')} / {preset.get('R_contact_area', '—')} K mm²/W</td></tr>
        <tr><td>R_path / length factor / fin efficiency</td><td>{preset.get('R_path_area', '—')} K mm²/W / {preset.get('length_factor', '—')} / {preset.get('fin_eff', '—')}</td></tr>
        <tr><td>Heat rejection model</td><td>{reject_params['model']}</td></tr>
        """
        if reject_params["model"] == "Transport only / ideal downstream sink":
            limits_html += "<tr><td>System boundary</td><td>Transport pump/fan only; downstream heat sink is treated as ideal/free.</td></tr>"
        if reject_params["model"] == "Facility water / dry cooler / economizer":
            limits_html += f"""
            <tr><td>Facility/economizer COP</td><td>{reject_params['economizer_COP']:.3g}</td></tr>
            <tr><td>Coolant/air supply temperature</td><td>{reject_params['T_supply_C']:.3g} °C</td></tr>
            <tr><td>Ambient/reference temperature</td><td>{reject_params['T_amb_C']:.3g} °C</td></tr>
            <tr><td>Reject approach requirement</td><td>{reject_params['reject_approach_K']:.3g} K</td></tr>
            """
        if reject_params["model"] in ["Gordon-Ng calibrated", "Gordon-Ng calibrated chiller"]:
            a2 = gordon_ng_calibrated_params(
                reject_params["Q_ref_kW"], reject_params["COP_ref"],
                reject_params["T_chw_ref_C"] + 273.15,
                reject_params["T_cw_ref_C"] + 273.15,
                reject_params["a1_kW_per_K"],
                reject_params["a3_K_per_kW"]
            )
            limits_html += f"""
            <tr><td>Gordon-Ng T_chw / T_cw</td><td>{reject_params['T_chw_C']:.3g} °C / {reject_params['T_cw_C']:.3g} °C</td></tr>
            <tr><td>Rated Q_ref, COP_ref</td><td>{reject_params['Q_ref_kW']:.3g} kW, {reject_params['COP_ref']:.3g}</td></tr>
            <tr><td>Calibrated a2</td><td>{a2:.4g} kW</td></tr>
            <tr><td>PLR max</td><td>{reject_params['PLR_max']:.3g}</td></tr>
            """
        if preset["kind"] != "passive":
            limits_html += f"""
            <tr><td>Channel/fin width range</td><td>{preset['w_mm'][0]:.4g}–{preset['w_mm'][1]:.4g} mm</td></tr>
            <tr><td>Channel/fin height range</td><td>{preset['h_mm'][0]:.4g}–{preset['h_mm'][1]:.4g} mm</td></tr>
            <tr><td>Wall/fin thickness</td><td>{preset['wall_mm']:.4g} mm</td></tr>
            <tr><td>Footprint area factor</td><td>{preset['area_factor']:.3g} × chip area</td></tr>
            <tr><td>Max pressure drop</td><td>{preset['dp_max']:.3g} Pa</td></tr>
            <tr><td>Max velocity</td><td>{preset['v_max']:.3g} m/s</td></tr>
            <tr><td>Max flow</td><td>{preset['flow_max']*60000:.3g} L/min</td></tr>
            <tr><td>Max coolant/air temperature rise</td><td>{preset['dT_fluid_max']:.3g} K</td></tr>
            <tr><td>Pump/fan efficiency</td><td>{preset['eta']:.3g}</td></tr>
            """
        limits_html += "</table>"
        display(widgets.HTML(limits_html))

        if rtar.get("feasible", False):
            display(Markdown(
                f"**At q = {q_target:.4g} W/mm²:** feasible with total cooling COP ≈ {rtar['COP']:.4g}, "
                f"P_wall ≈ {rtar['P_wall']:.4g} W, transport power ≈ {rtar.get('P_transport', 0):.4g} W, "
                f"heat-rejection/chiller power ≈ {rtar.get('P_reject', 0):.4g} W, "
                f"reject COP ≈ {rtar.get('COP_reject', np.inf):.4g}, ΔT_model ≈ {rtar['temp_rise']:.4g} K."
            ))
            if preset["kind"] != "passive":
                display(Markdown(
                    f"Operating point at q_target: velocity={rtar['velocity']:.3g} m/s, "
                    f"flow={rtar['flow']*60000:.3g} L/min, Δp={rtar['dp']:.3g} Pa."
                ))
        else:
            display(Markdown(f"**At q = {q_target:.4g} W/mm²:** infeasible ({rtar.get('reason', 'unknown')})."))

        fig, axs = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
        axs[0,0].plot(q_grid[feasible], COP[feasible], lw=2)
        axs[0,0].set_ylabel("COP = Q_chip / P_wall [-]")
        axs[0,0].set_title("Effective total cooling COP")

        axs[0,1].plot(q_grid[feasible], Pwall[feasible], lw=2, label="total")
        axs[0,1].plot(q_grid[feasible], Ptransport[feasible], lw=1.8, label="transport")
        axs[0,1].plot(q_grid[feasible], Preject[feasible], lw=1.8, label="reject/chiller")
        axs[0,1].set_ylabel("Power [W]")
        axs[0,1].set_title("Wall-plug cooling power")
        axs[0,1].legend(fontsize=8)

        axs[1,0].plot(q_grid[feasible], Trise[feasible], lw=2, label="model")
        axs[1,0].axhline(DeltaT, color="k", ls="--", lw=1, label="limit")
        axs[1,0].set_ylabel("Temperature rise [K]")
        axs[1,0].set_title("Thermal constraint")
        axs[1,0].legend(fontsize=8)

        if preset["kind"] != "passive":
            axs[1,1].plot(q_grid[feasible], dp[feasible]/1000, lw=2, label="Δp [kPa]")
            axs[1,1].plot(q_grid[feasible], flow_lpm[feasible], lw=2, label="flow [L/min]")
            if np.any(np.isfinite(COP_reject[feasible])):
                axs[1,1].plot(q_grid[feasible], COP_reject[feasible], lw=2, label="reject COP")
            axs[1,1].set_ylabel("Hydraulic / reject quantities")
            axs[1,1].legend(fontsize=8)
        else:
            axs[1,1].plot(q_grid, feasible.astype(float), lw=2)
            axs[1,1].set_ylabel("feasible [0/1]")

        for ax in axs.flat:
            ax.set_xlabel("Average chip heat flux q [W/mm²]")
            ax.tick_params(axis="x", which="both", bottom=True, labelbottom=True)
            ax.grid(True, alpha=0.3)
            ax.axvline(q_target, color="gray", ls=":", lw=1)

        fig.tight_layout()
        plt.show()

def coldplate_notes_box():
    """Formatted notes and equations for the cold-plate hardware-envelope tab."""
    out = widgets.Output(layout=widgets.Layout(
        border="1px solid #d8c07a",
        padding="10px 14px",
        margin="8px 0",
        width="100%"
    ))
    out.add_class("chip-note-box")

    with out:
        display(Markdown("### Cold plate hardware-envelope model"))

        display(Markdown(
            "This is a **state-of-the-art screening envelope**, not a final vendor-calibrated design tool. "
            "The presets now carry provenance labels that distinguish physically plausible heuristic limits from literature-anchored research or product demonstrations. "
            "The same shared hardware model is used here and in the Hardware-aware cooling model."
        ))

        display(Markdown("#### Provenance / calibration status"))
        display(Markdown(
            "- **heuristic_screening_with_physical_limits:** physically reasonable envelope; use for qualitative trends.\n"
            "- **literature_anchored_screening:** limits chosen to be consistent with public requirements/reports, but still reduced-order.\n"
            "- **literature_anchored_surrogate_not_full_two_phase:** anchored to two-phase demonstrations, but modeled with an equivalent-channel surrogate, not boiling/dryout physics.\n"
            "- **literature_anchored_research_surrogate:** anchored to research demonstrations; packaging, reliability, clogging and manifold constraints remain simplified.\n\n"
            "Key public anchors used in the preset notes: OCP cold-plate requirements emphasize pressure drop, flow rate, active area and inlet/outlet temperatures; ASHRAE describes CDU separation between facility water and technology cooling loops; ACT reports a two-phase direct-to-chip demonstration above 7.5 kW and 300 W/cm²; recent embedded microfluidic work reports 3000 W/cm² at about 0.9 W/cm² pumping power."
        ))

        display(Markdown("#### Core definitions"))
        for line in [
            r"Q_{\rm chip}=qA",
            r"Q_{\rm reject}=Q_{\rm chip}+P_{\rm transport}",
            r"COP_{\rm total}=\frac{Q_{\rm chip}}{P_{\rm wall}}",
            r"P_{\rm transport}=\frac{\Delta p\,\dot V}{\eta_{\rm pump/fan}}",
        ]:
            display(Math(line))

        display(Markdown("#### Geometry and transport model"))
        for line in [
            r"A_{\rm base}=f_A A,\qquad W_{\rm base}=\sqrt{A_{\rm base}},\qquad L=f_LW_{\rm base}",
            r"p_{\rm pitch}=w+t_{\rm wall},\qquad N_{\rm ch}=\left\lfloor W_{\rm base}/p_{\rm pitch}\right\rfloor",
            r"A_{\rm flow}=N_{\rm ch}wh,\qquad A_{\rm wet}=2N_{\rm ch}(w+h)L\eta_f M_A",
            r"D_h=\frac{2wh}{w+h},\qquad Re=\frac{\rho vD_h}{\mu}",
            r"Nu=M_{Nu}\left[(1-b)Nu_{\rm lam}+bNu_{\rm turb}\right]",
            r"R_{\rm ext,hw}=R_{\rm contact}+R_{\rm conv}A+\frac{1}{2}\frac{\Delta T_{\rm fluid}}{q}",
            r"\Delta p=\left(f\frac{L}{D_h}+K_{\rm loss}\right)\frac{\rho v^2}{2}",
        ]:
            display(Math(line))

        display(Markdown(
            "`w` and `h` are fin-gap/fin-height for forced air and channel-width/channel-depth for liquid, microstructured and embedded presets. "
            "`M_A` and `M_Nu` are topology multipliers used for manifold/microstructured/embedded surrogate behavior. "
            "`K_loss` is a lumped minor-loss/manifold/jet/turn coefficient. "
            "The current model is still an equivalent-channel model; topology-specific CFD/experimental maps should replace these surrogates for final design."
        ))

        display(Markdown("#### Preset-specific optimized and fixed quantities"))
        display(Markdown("""
| Preset | Kind | Optimized variables | Key fixed quantities | Provenance label |
|---|---|---|---|---|
| Passive spreader / vapor chamber + downstream reject | passive | none | R_contact, R_path, downstream reject option | heuristic_screening_with_physical_limits |
| Forced-air finned heatsink with fan curve | air channels | w, h, v | fan curve, acoustic limit, volume limit, K_loss, fin efficiency | heuristic_with_fan_curve_acoustic_volume_limits |
| Conduction-cooled rugged chassis / cold wall | conduction chassis | none | R_contact, R_path, R_heatpipe, P_aux, downstream reject | heuristic_screening_with_downstream_reject |
| Direct-to-chip single-phase cold plate | liquid channels | w, h, v | pressure/flow/velocity limits, K_loss, pump efficiency | literature_anchored_screening |
| Direct-to-chip two-phase cold plate surrogate | equivalent liquid channels | w, h, v | boosted heat transfer, small fluid ΔT, two-phase surrogate flag | literature_anchored_surrogate_not_full_two_phase |
| Manifold / microstructured single-phase cold plate | microstructured liquid channels | w, h, v | shortened length, area multiplier, Nu multiplier, maldistribution margin | literature_anchored_screening_with_topology_multiplier |
| Embedded microfluidic equivalent-channel surrogate | embedded liquid channels | w, h, v | high pressure envelope, area/Nu multipliers, maldistribution margin | literature_anchored_research_surrogate |
"""))

        display(Markdown("#### Current preset numerical envelopes"))
        display(Markdown("""
| Preset | width [mm] | height [mm] | wall [mm] | footprint | K_loss | R_contact | R_path/R_heatpipe | Δp max | v max | flow max | fluid ΔT max | eta | extra constraints |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|
| Passive + downstream reject | — | — | — | passive | — | 20 | 320 / — | — | — | — | — | — | downstream reject allowed |
| Forced-air finned heatsink | 1–5 | 10–50 | 0.8 | 15× | 4 | 25 | — | 250 Pa | 12 m/s | 3000 L/min | 35 K | 0.25 | fan curve, 55 dBA, 3 L volume |
| Rugged conduction chassis / cold wall | — | — | — | chassis | — | 15 | 105 / 15 | — | — | — | — | — | P_aux=5 W, downstream reject allowed |
| D2C single-phase cold plate | 0.8–3 | 0.5–3 | 0.5 | 1.5× | 6 | 8 | — | 50 kPa | 2.5 m/s | 4.8 L/min | 15 K | 0.35 | OCP-style single-phase channel envelope |
| D2C two-phase surrogate | 0.5–2.5 | 0.3–2 | 0.4 | 1.5× | 8 | 5 | — | 80 kPa | 3 m/s | 3.6 L/min | 8 K | 0.35 | boosted h, reduced fluid ΔT surrogate |
| Manifold/microstructured single-phase | 0.10–0.60 | 0.10–0.80 | 0.075 | 1.1× | 10 | 4 | — | 200 kPa | 6 m/s | 1.8 L/min | 20 K | 0.30 | M_A=1.5, M_Nu=1.6 |
| Embedded microfluidic surrogate | 0.040–0.250 | 0.050–0.300 | 0.040 | 1.0× | 15 | 1 | — | 500 kPa | 10 m/s | 0.9 L/min | 30 K | 0.25 | M_A=2.0, M_Nu=2.5 |
"""))

        display(Markdown("#### Additional constraints now included"))
        display(Markdown(
            "- **Passive spreader:** can include downstream heat rejection using the facility/economizer or chiller model.\n"
            "- **Rugged conduction chassis:** includes a chassis/cold-wall/heat-pipe path, a small auxiliary fan/loop power, and optional downstream rejection.\n"
            "- **Forced-air heatsink:** includes a simple fan curve, acoustic limit, and volume limit in addition to pressure, flow, and velocity limits.\n"
            "- **Two-phase D2C:** represented only as a surrogate. The notebook does not yet solve saturation, quality, dryout, critical heat flux, or two-phase pressure drop.\n"
            "- **Microstructured/embedded:** represented by equivalent-channel topology multipliers. The notebook does not yet solve manifold distribution, jet-plate, sawtooth-channel, pin-fin, clogging, or reliability constraints explicitly."
        ))


        display(Markdown("#### Preset-by-preset model assumptions"))
        display(Markdown(
            "**Passive spreader / vapor chamber + downstream reject.** "
            "No local fan/pump/channel flow is modeled. The local path is an area-normalized contact resistance plus passive spreader/enclosure resistance. "
            "Selecting transport-only means no downstream heat-rejection electrical power is added. Facility/economizer or chiller modes can be selected to represent an enclosure radiator, facility sink, or active downstream loop."
        ))
        display(Markdown(
            "**Forced-air finned heatsink with fan curve.** "
            "The optimized geometry variables are fin-gap width w, fin height h, and air velocity v. The number of passages follows from pitch = w + t_wall and the available footprint. "
            "The model enforces system pressure drop, maximum channel velocity, maximum volumetric flow, maximum air temperature rise, a simplified fan curve with shutoff pressure and free-flow rate, a 55 dBA acoustic screening limit, and a volume envelope. "
            "Heat is rejected directly to local air through fan work; chiller/facility-water modes are not exposed for this preset."
        ))
        display(Markdown(
            "**Conduction-cooled rugged chassis / cold wall.** "
            "The model treats the hardware as an area-normalized conduction path through contact, chassis/cold-wall, and optional heat-pipe terms, plus auxiliary fan/loop power where specified. "
            "This represents conduction cards, wedge locks, cold walls, heat pipes, blowers, or facility-loop support in a screening sense. "
            "Downstream facility/economizer or chiller modes may be added to represent the cold wall or chassis rejecting heat to a larger system."
        ))
        display(Markdown(
            "**Direct-to-chip single-phase cold plate.** "
            "The optimized variables are channel width w, channel depth h, and coolant velocity v. The model solves equivalent rectangular-channel Reynolds number, Nusselt number, friction factor, pressure drop, pump power, and coolant temperature rise. "
            "The preset uses an OCP-style single-phase envelope: pressure drop, flow, active area, inlet/outlet temperatures, and pump efficiency are the key limits."
        ))
        display(Markdown(
            "**Direct-to-chip two-phase cold plate surrogate.** "
            "This is an equivalent-channel surrogate anchored to two-phase direct-to-chip demonstrations, but it does not solve boiling, saturation temperature, vapor quality, dryout, critical heat flux, or two-phase pressure drop. "
            "It uses boosted effective heat transfer and reduced fluid temperature-rise constraints to mimic the benefit of latent heat transport in a screening model."
        ))
        display(Markdown(
            "**Manifold / microstructured single-phase cold plate.** "
            "This preset represents manifold microchannels, short channels, pin-fin/jet-enhanced plates, or microstructured single-phase designs using an equivalent-channel model with shortened effective length, increased wetted area, enhanced Nusselt multiplier, and maldistribution margin. "
            "The pressure and flow limits are aggressive screening envelopes, not a vendor-specific manifold map."
        ))
        display(Markdown(
            "**Embedded microfluidic equivalent-channel surrogate.** "
            "This preset represents near-junction/backside embedded microfluidics using small equivalent channels, high pressure envelope, and topology multipliers. "
            "It is research-grade and does not explicitly model multilayer manifolds, jet plates, sawtooth channels, dielectric compatibility, leakage reliability, clogging, or fabrication stress."
        ))


        display(Markdown("#### Heat-rejection modes"))
        for line in [
            r"\mathrm{Transport\ only:}\quad P_{\rm wall}=P_{\rm transport}",
            r"\mathrm{Facility/economizer:}\quad P_{\rm wall}=P_{\rm transport}+\frac{Q_{\rm reject}}{COP_{\rm reject,free}}",
            r"\mathrm{Chiller:}\quad P_{\rm wall}=P_{\rm transport}+P_{\rm chiller}(Q_{\rm reject},T_{\rm chw},T_{\rm cw})",
        ]:
            display(Math(line))
        display(Markdown(
            "Transport-only is the no downstream heat-rejection / original diagnostic boundary condition: P_reject = 0. For a passive spreader this is the original no-added-downstream-power case. Facility/economizer and Gordon–Ng chiller modes represent explicit downstream heat rejection and are available for passive and rugged conduction presets as well as liquid cold-plate presets."
        ))

    return out


cp_preset = widgets.Dropdown(
    options=list(COLD_PLATE_PRESETS.keys()),
    value="Direct-to-chip single-phase cold plate",
    description="Preset",
    style=style,
    layout=widgets.Layout(width="620px")
)



cp_sweep_mode = widgets.Dropdown(
    options=[
        "Design envelope (redesign each q)",
        "Fixed hardware sweep (design at q target)"
    ],
    value="Fixed hardware sweep (design at q target)",
    description="Sweep mode",
    style=style,
    layout=widgets.Layout(width="620px")
)

cp_reject_model = widgets.Dropdown(
    options=[
        "Transport only / ideal downstream sink",
        "Facility water / dry cooler / economizer",
        "Gordon-Ng calibrated chiller"
    ],
    value="Facility water / dry cooler / economizer",
    description="Heat rejection",
    style=style,
    layout=widgets.Layout(width="620px")
)

cp_controls = {}
cp_controls["A_mm2"] = linked_float("A", 100.0, 1.0, 2000.0, 1.0, "[mm²]")
cp_controls["DeltaT"] = linked_float("DeltaT", 60.0, 5.0, 150.0, 1.0, "[K]")
cp_controls["R_die"] = linked_float("R_die", 5.0, 0.1, 100.0, 0.1, "[K·mm²/W]")
cp_controls["qmax"] = linked_float("q plot max", 5.0, 0.01, 50.0, 0.05, "[W/mm²]")
cp_controls["qtarget"] = linked_float("q target", 1.0, 0.001, 50.0, 0.01, "[W/mm²]")

# Heat-rejection / chiller controls.
cp_controls["economizer_COP"] = linked_float("facility/economizer COP", 30.0, 0.1, 300.0, 0.5, "[-]")
cp_controls["T_supply_C"] = linked_float("T_supply", 30.0, -10.0, 80.0, 0.5, "[°C]")
cp_controls["T_amb_C"] = linked_float("T_amb", 25.0, -20.0, 60.0, 0.5, "[°C]")
cp_controls["reject_approach_K"] = linked_float("reject approach", 8.0, 0.0, 40.0, 0.5, "[K]")
cp_controls["T_chw_C"] = linked_float("T_chw", 20.0, -10.0, 60.0, 0.5, "[°C]")
cp_controls["T_cw_C"] = linked_float("T_cw", 35.0, 0.0, 80.0, 0.5, "[°C]")
cp_controls["T_chw_ref_C"] = linked_float("T_chw,ref", 20.0, -10.0, 60.0, 0.5, "[°C]")
cp_controls["T_cw_ref_C"] = linked_float("T_cw,ref", 35.0, 0.0, 80.0, 0.5, "[°C]")
cp_controls["GN_Q_ref_kW"] = linked_float("Q_ref", 1.0, 0.01, 100.0, 0.05, "[kW]")
cp_controls["GN_COP_ref"] = linked_float("COP_ref", 5.0, 0.2, 30.0, 0.1, "[-]")
cp_controls["GN_PLR_max"] = linked_float("PLR_max", 1.2, 0.2, 5.0, 0.1, "[-]")
cp_controls["GN_a1"] = linked_float("GN a1", 0.0, 0.0, 0.1, 0.001, "[kW/K]")
cp_controls["GN_a3"] = linked_float("GN a3", 0.0, 0.0, 10.0, 0.05, "[K/kW]")

# Dynamic heat-rejection control visibility.
# The available heat-rejection models are limited by architecture class.
# Passive/forced-air systems do not expose the chiller model by default because
# there is no liquid cold-plate loop requiring chilled water in this simplified
# hardware model. Liquid/conduction cold-wall classes expose all options.
hr_transport_boxes = []
hr_economizer_boxes = []
hr_chiller_boxes = []

def heat_rejection_options_for_preset(preset_name):
    preset = COLD_PLATE_PRESETS.get(preset_name, {})
    kind = preset.get("kind", "")
    # Passive and rugged conduction paths can be attached to downstream reject
    # hardware; forced-air already rejects locally through the fan/heatsink and
    # therefore does not expose a chiller model in this simplified envelope.
    if kind == "air_channels":
        return ["Transport only / ideal downstream sink"]
    if kind in ["passive", "conduction_chassis"]:
        return ["Transport only / ideal downstream sink", "Facility water / dry cooler / economizer", "Gordon-Ng calibrated chiller"]
    return ["Transport only / ideal downstream sink", "Facility water / dry cooler / economizer", "Gordon-Ng calibrated chiller"]

def make_heat_rejection_controls_panel():
    transport_box = widgets.VBox([
        widgets.HTML("<b>No downstream heat rejection / transport-only diagnostic</b>"),
        widgets.HTML("<small>For passive spreader this is the original no-added-downstream-power case: P_reject = 0. For active cold plates/heatsinks it counts only local transport power and is not a full system-level heat-rejection COP.</small>")
    ])
    economizer_box = widgets.VBox([
        widgets.HTML("<b>Facility water / dry cooler / economizer controls</b>"),
        two_col([cp_controls[k]["box"] for k in ["economizer_COP", "T_supply_C", "T_amb_C", "reject_approach_K"]])
    ])
    chiller_box = widgets.VBox([
        widgets.HTML("<b>Gordon–Ng calibrated chiller controls</b>"),
        two_col([cp_controls[k]["box"] for k in ["T_chw_C", "T_cw_C", "T_chw_ref_C", "T_cw_ref_C", "GN_Q_ref_kW", "GN_COP_ref", "GN_PLR_max", "GN_a1", "GN_a3"]])
    ])

    hr_transport_boxes.append(transport_box)
    hr_economizer_boxes.append(economizer_box)
    hr_chiller_boxes.append(chiller_box)

    return widgets.VBox([
        widgets.HTML("<b>Heat rejection controls</b>"),
        transport_box,
        economizer_box,
        chiller_box
    ])

def update_heat_rejection_controls(*_):
    allowed = heat_rejection_options_for_preset(cp_preset.value)
    current = cp_reject_model.value
    cp_reject_model.options = allowed
    if current not in allowed:
        cp_reject_model.value = "Facility water / dry cooler / economizer" if "Facility water / dry cooler / economizer" in allowed else allowed[0]

    mode = cp_reject_model.value
    show_transport = mode == "Transport only / ideal downstream sink"
    show_economizer = mode == "Facility water / dry cooler / economizer"
    show_chiller = mode == "Gordon-Ng calibrated chiller"

    for box in hr_transport_boxes:
        box.layout.display = "" if show_transport else "none"
    for box in hr_economizer_boxes:
        box.layout.display = "" if show_economizer else "none"
    for box in hr_chiller_boxes:
        box.layout.display = "" if show_chiller else "none"

cp_preset.observe(update_heat_rejection_controls, names="value")
cp_reject_model.observe(update_heat_rejection_controls, names="value")

coldplate_heat_rejection_controls = make_heat_rejection_controls_panel()
hardware_heat_rejection_controls = make_heat_rejection_controls_panel()
update_heat_rejection_controls()



cp_button = widgets.Button(description="Run cold plate envelope", button_style="info", icon="play")
cp_output = widgets.Output()
cp_button.on_click(run_coldplate_envelope)

coldplate_tab = widgets.VBox([
    collapsed_section("Cold-plate hardware-envelope notes / equations", coldplate_notes_box()),
    cp_preset,
    cp_sweep_mode,
    cp_reject_model,
    two_col([cp_controls[k]["box"] for k in ["A_mm2", "DeltaT", "R_die", "qmax", "qtarget"]]),
    coldplate_heat_rejection_controls,
    cp_button,
    cp_output
])


# ---------------------------
# Hardware-aware hybrid MR optimization
# ---------------------------
# This coupled tab uses the SAME two-temperature hotspot/bulk constraints as the
# compact / reduced-order model, but replaces the abstract R_ext(u), p_pump(u)
# model with a physically constrained cold-plate / heat-rejection hardware model.

def hw_ext_resistance_temperature(q_W_mm2, A_mm2, R_die_area, preset, design=None, v=None):
    """Return hardware external resistance and transport quantities for one geometry/flow.

    This is the shared transport/geometry model used by both the standalone
    cold-plate diagnostics and the hardware-aware hybrid MR optimizer.

    R_ext,hw = R_contact + R_conv*A + 0.5*DeltaT_fluid/q for forced-flow hardware.
    Passive/conduction presets bypass channel-flow terms and return an equivalent
    area-normalized path resistance plus optional auxiliary wall power.
    """
    Q_W = q_W_mm2 * A_mm2
    if Q_W <= 0:
        return dict(feasible=True, R_ext_area=0.0, P_transport=0.0, dT_fluid=0.0,
                    dp=0.0, flow=0.0, velocity=0.0, Re=np.nan, Nu=np.nan,
                    width_mm=np.nan, height_mm=np.nan, N_channels=0)

    if preset["kind"] in ["passive", "conduction_chassis"]:
        R_ext_area = preset.get("R_contact_area", 0.0) + preset.get("R_path_area", 0.0) + preset.get("R_heatpipe_area", 0.0)
        P_aux = preset.get("P_aux_W", 0.0)
        return dict(
            feasible=True,
            R_ext_area=R_ext_area,
            P_transport=P_aux,
            dT_fluid=0.0,
            dp=0.0,
            flow=0.0,
            velocity=0.0,
            Re=np.nan,
            Nu=np.nan,
            width_mm=np.nan,
            height_mm=np.nan,
            N_channels=0,
            dp_util=0.0,
            flow_util=0.0,
            velocity_util=0.0,
            fluid_dT_util=0.0,
            noise_dBA=np.nan,
            noise_util=0.0,
            volume_L=preset.get("volume_L", np.nan),
            volume_util=0.0,
        )

    props = fluid_properties(preset["fluid"])
    rho, mu, kf, cp, Pr = props["rho"], props["mu"], props["k"], props["cp"], props["Pr"]

    A_chip_m2 = A_mm2 * 1e-6
    A_base = A_chip_m2 * preset["area_factor"]
    W_base = np.sqrt(A_base)
    L = W_base * preset.get("length_factor", 1.0)

    if design is not None:
        w_grid = [float(design["width_mm"]) * 1e-3]
        h_grid = [float(design["height_mm"]) * 1e-3]
        Nch_fixed = int(max(1, design["N_channels"]))
    else:
        w_grid = np.geomspace(preset["w_mm"][0]*1e-3, preset["w_mm"][1]*1e-3, 6)
        h_grid = np.geomspace(preset["h_mm"][0]*1e-3, preset["h_mm"][1]*1e-3, 6)
        Nch_fixed = None

    if v is None:
        v_values = np.geomspace(0.02, preset["v_max"], 70)
    else:
        v_values = np.atleast_1d(float(v))

    candidates = []
    wall = preset["wall_mm"] * 1e-3

    for w in w_grid:
        pitch = w + wall
        Nch = Nch_fixed if Nch_fixed is not None else max(1, int(W_base / pitch))
        for h_ch in h_grid:
            A_flow = Nch * w * h_ch
            if A_flow <= 0:
                continue

            Dh = 2*w*h_ch/(w+h_ch)
            A_wet = Nch * 2*(w+h_ch)*L * preset.get("fin_eff", 1.0) * preset.get("area_multiplier", 1.0)

            vv = np.asarray(v_values, dtype=float)
            Vdot = vv * A_flow
            Re = rho*vv*Dh/mu

            Nu_lam = 4.36*np.ones_like(Re)
            Nu_turb = 0.023*np.maximum(Re, 1.0)**0.8 * Pr**0.4
            blend = np.clip((Re - 1800.0)/(4000.0 - 1800.0), 0.0, 1.0)
            Nu = ((1-blend)*Nu_lam + blend*np.maximum(Nu_lam, Nu_turb)) * preset.get("Nu_multiplier", 1.0)

            hcoef = Nu*kf/Dh
            Rconv_K_W = 1.0/np.maximum(hcoef*A_wet, 1e-30)
            Rconv_area = Rconv_K_W * A_mm2

            dT_fluid = Q_W/(rho*np.maximum(Vdot, 1e-30)*cp)
            if preset.get("two_phase_surrogate", False):
                # Equivalent surrogate for latent heat / two-phase temperature control.
                # Not a boiling/dryout calculation.
                dT_fluid = dT_fluid * 0.35

            R_ext_area = preset["R_contact_area"] + Rconv_area + 0.5*dT_fluid/q_W_mm2

            f_lam = 64.0/np.maximum(Re, 1.0)
            f_turb = 0.3164/np.maximum(Re, 1.0)**0.25
            f = np.where(Re < 2300.0, f_lam, f_turb)

            dp = (f*L/Dh + preset.get("K_loss", 0.0)) * rho*vv*vv/2.0
            P_transport = dp*Vdot/np.maximum(preset.get("eta", 1.0), 1e-9)

            # Additional operational limits for forced-air heatsinks.
            fan_curve_ok = np.ones_like(vv, dtype=bool)
            noise_ok = np.ones_like(vv, dtype=bool)
            volume_ok = np.ones_like(vv, dtype=bool)
            noise_dBA = np.full_like(vv, np.nan, dtype=float)

            if preset["kind"] == "air_channels":
                dp_avail = preset.get("fan_curve_dp0", preset["dp_max"]) * np.maximum(0.0, 1.0 - Vdot/np.maximum(preset.get("fan_curve_flow_free", preset["flow_max"]), 1e-30))
                fan_curve_ok = dp <= dp_avail

                # Simple aeroacoustic screening curve, not a vendor fan model.
                noise_dBA = preset.get("noise_dBA_ref", 38.0) + 50.0*np.log10(np.maximum(Vdot, 1e-9)/max(preset.get("noise_flow_ref", 0.01), 1e-9))
                noise_ok = noise_dBA <= preset.get("noise_dBA_max", 1e9)

                volume_ok = preset.get("volume_L", 0.0) <= preset.get("volume_L_max", 1e9)

            feasible_hw = (
                (dp <= preset["dp_max"]) &
                (vv <= preset["v_max"]) &
                (Vdot <= preset["flow_max"]) &
                (dT_fluid <= preset["dT_fluid_max"]) &
                fan_curve_ok &
                noise_ok &
                volume_ok &
                np.isfinite(P_transport) &
                np.isfinite(R_ext_area)
            )

            # Apply maldistribution derating to feasibility margin by inflating R_ext.
            R_ext_area_eff = R_ext_area / max(preset.get("maldistribution_margin", 1.0), 1e-9)

            for j in np.where(feasible_hw)[0]:
                candidates.append(dict(
                    feasible=True,
                    R_ext_area=float(R_ext_area_eff[j]),
                    P_transport=float(P_transport[j]),
                    dT_fluid=float(dT_fluid[j]),
                    dp=float(dp[j]),
                    flow=float(Vdot[j]),
                    velocity=float(vv[j]),
                    Re=float(Re[j]),
                    Nu=float(Nu[j]),
                    width_mm=float(w*1e3),
                    height_mm=float(h_ch*1e3),
                    N_channels=int(Nch),
                    dp_util=float(dp[j]/preset["dp_max"]),
                    flow_util=float(Vdot[j]/preset["flow_max"]),
                    velocity_util=float(vv[j]/preset["v_max"]),
                    fluid_dT_util=float(dT_fluid[j]/preset["dT_fluid_max"]),
                    fan_curve_util=float(dp[j]/max(dp_avail[j], 1e-30)) if preset["kind"] == "air_channels" else 0.0,
                    noise_dBA=float(noise_dBA[j]) if preset["kind"] == "air_channels" else np.nan,
                    noise_util=float(noise_dBA[j]/preset.get("noise_dBA_max", 1e9)) if preset["kind"] == "air_channels" else 0.0,
                    volume_L=float(preset.get("volume_L", np.nan)),
                    volume_util=float(preset.get("volume_L", 0.0)/preset.get("volume_L_max", 1e9)) if preset["kind"] == "air_channels" else 0.0,
                ))
    return candidates

def hw_temperatures(q, R_ext_area, s, tp):
    """Same hotspot/bulk equations as temperatures(), but with hardware R_ext."""
    e = matrix_elements(tp)
    gH, gB = gammas(tp)
    RHHu_hw = e["RHHu_const"] + R_ext_area
    RBBu_hw = e["RBBu_const"] + R_ext_area
    TH = q * (
        gH * e["RHHd"]
        + gB * e["RHBd"]
        + gH * (1.0 - s) * RHHu_hw
        + gB * e["RHBu"]
    )
    TB = q * (
        gH * e["RBHd"]
        + gB * e["RBBd"]
        + gH * (1.0 - s) * e["RBHu"]
        + gB * RBBu_hw
    )
    return TH, TB


def hw_s_required(q, R_ext_area, tp):
    """Minimum s required to satisfy the SAME hotspot constraint."""
    e = matrix_elements(tp)
    gH, gB = gammas(tp)
    RHHu_hw = e["RHHu_const"] + R_ext_area
    numerator = tp.DeltaT/q - gH*e["RHHd"] - gB*e["RHBd"] - gB*e["RHBu"]
    denom = gH*RHHu_hw
    if denom <= 0 or not np.isfinite(denom):
        return np.nan
    return max(0.0, 1.0 - numerator/denom)


def hw_eval_candidate(q, cand, tp, A_mm2, reject_params, force_s_zero=False):
    """Evaluate one hardware candidate under the original hotspot/bulk constraints.

    If force_s_zero=True, this is external-cooling-only.
    Otherwise, s is set to the minimum value needed to satisfy hotspot temperature,
    then the bulk constraint is checked exactly as in the compact / reduced-order hybrid model.
    """
    Rext = cand["R_ext_area"]
    if force_s_zero:
        s = 0.0
    else:
        s = hw_s_required(q, Rext, tp)

    s_max = hw_s_max()
    if not np.isfinite(s) or s < -1e-12 or s > s_max + 1e-12:
        return None
    s = float(np.clip(s, 0.0, s_max))

    TH, TB = hw_temperatures(q, Rext, s, tp)
    TH_util = TH/tp.DeltaT if np.isfinite(TH) else np.nan
    TB_util = TB/tp.DeltaT if np.isfinite(TB) else np.nan
    e_local = matrix_elements(tp)
    gH_local, _ = gammas(tp)
    RHHu_hw_local = e_local["RHHu_const"] + Rext
    DeltaT_MR = q * gH_local * (1.0 - s) * RHHu_hw_local
    minus_DeltaT_MR = max(0.0, -DeltaT_MR)

    if (not np.isfinite(TH)) or (not np.isfinite(TB)) or TH > tp.DeltaT*(1+1e-9) or TB > tp.DeltaT*(1+1e-9):
        return None

    Q_chip = q * A_mm2
    P_transport = cand["P_transport"]
    P_MR = tp.alpha_tot * s * q * A_mm2 / tp.COP_micro
    Q_reject = Q_chip + P_transport + P_MR

    P_reject, COP_reject, feasible_reject = gordon_ng_chiller_power(
        Q_reject, reject_params, dT_fluid_K=cand.get("dT_fluid", None)
    )
    P_reject = float(np.asarray(P_reject))
    COP_reject = float(np.asarray(COP_reject)) if np.size(COP_reject) == 1 else float(np.asarray(COP_reject).ravel()[0])
    feasible_reject = bool(np.asarray(feasible_reject).ravel()[0])

    if not feasible_reject or not np.isfinite(P_reject):
        return None

    P_wall = P_transport + P_reject + P_MR
    return dict(
        q=q,
        feasible=True,
        s=s,
        s_util=s,
        TH=TH,
        TB=TB,
        TH_util=TH_util,
        TB_util=TB_util,
        DeltaT_MR=DeltaT_MR,
        minus_DeltaT_MR=minus_DeltaT_MR,
        P_wall=P_wall,
        P_transport=P_transport,
        P_reject=P_reject,
        P_MR=P_MR,
        COP_total=Q_chip/P_wall if P_wall > 0 else np.inf,
        R_ext_area=Rext,
        COP_reject=COP_reject,
        branch="cold plate-only" if force_s_zero else ("passive" if P_wall == 0 and s == 0 else "hybrid"),
        **{k: cand.get(k, np.nan) for k in [
            "dp", "flow", "velocity", "Re", "Nu", "width_mm", "height_mm", "N_channels", "dT_fluid",
            "dp_util", "flow_util", "velocity_util", "fluid_dT_util"
        ]}
    )


def hw_design_candidates_for_q(q, tp, preset, fixed_design=None):
    """Generate feasible hardware transport candidates for one q."""
    return hw_ext_resistance_temperature(q, tp.A_mm2, tp.R_die, preset, design=fixed_design)


def hw_candidate_diagnostics(q, tp, preset, reject_params, fixed_design=None, force_s_zero=False):
    """Return best near-feasible diagnostic information at one q.

    The diagnostic is intentionally verbose: it identifies whether the likely blocker is
    hotspot temperature, bulk temperature, required MR fraction, reject/chiller feasibility,
    or the absence of any hardware-feasible transport candidate.
    """
    cand_list = hw_design_candidates_for_q(q, tp, preset, fixed_design=fixed_design)
    if isinstance(cand_list, dict):
        cand_list = [cand_list] if cand_list.get("feasible", False) else []

    if not cand_list:
        return dict(
            feasible=False,
            reason="no hardware-feasible transport candidate",
            detail="No candidate satisfied cold-plate hydraulic/flow/coolant-rise limits before applying hotspot/bulk constraints.",
            max_util=np.nan
        )

    best = None
    for cand in cand_list:
        Rext = cand.get("R_ext_area", np.nan)
        s_req = 0.0 if force_s_zero else hw_s_required(q, Rext, tp)
        s_max = hw_s_max()
        s_valid = np.isfinite(s_req) and (s_req <= s_max) and (s_req >= -1e-12)
        s_eval = float(np.clip(s_req, 0.0, s_max)) if np.isfinite(s_req) else np.nan

        if np.isfinite(s_eval):
            TH, TB = hw_temperatures(q, Rext, s_eval, tp)
        else:
            TH, TB = np.nan, np.nan

        Q_chip = q * tp.A_mm2
        P_transport = cand.get("P_transport", np.nan)
        P_MR = tp.alpha_tot * max(s_eval if np.isfinite(s_eval) else 0.0, 0.0) * q * tp.A_mm2 / tp.COP_micro
        Q_reject = Q_chip + (P_transport if np.isfinite(P_transport) else 0.0) + P_MR
        P_reject, COP_reject, feasible_reject = gordon_ng_chiller_power(
            Q_reject, reject_params, dT_fluid_K=cand.get("dT_fluid", None)
        )
        feasible_reject = bool(np.asarray(feasible_reject).ravel()[0])

        ratios = dict(
            hotspot=float(TH/tp.DeltaT) if np.isfinite(TH) else np.inf,
            bulk=float(TB/tp.DeltaT) if np.isfinite(TB) else np.inf,
            s=float(s_req)/hw_s_max() if np.isfinite(s_req) else np.inf,
            dp=float(cand.get("dp_util", 0.0)),
            flow=float(cand.get("flow_util", 0.0)),
            velocity=float(cand.get("velocity_util", 0.0)),
            fluid_dT=float(cand.get("fluid_dT_util", 0.0)),
            reject=0.0 if feasible_reject else 2.0,
        )
        violation = max(ratios.values())
        reasons = []
        if ratios["hotspot"] > 1.0: reasons.append("hotspot ΔT")
        if ratios["bulk"] > 1.0: reasons.append("bulk ΔT")
        if ratios["s"] > 1.0: reasons.append("s_required > s_max")
        if ratios["dp"] > 1.0: reasons.append("pressure drop")
        if ratios["flow"] > 1.0: reasons.append("flow rate")
        if ratios["velocity"] > 1.0: reasons.append("velocity")
        if ratios["fluid_dT"] > 1.0: reasons.append("coolant/air ΔT")
        if ratios["reject"] > 1.0: reasons.append("heat rejection")

        item = dict(
            feasible=violation <= 1.0,
            reason=", ".join(reasons) if reasons else "near feasible / no single violated ratio",
            ratios=ratios,
            max_util=violation,
            s_required=float(s_req) if np.isfinite(s_req) else np.nan,
            TH=float(TH) if np.isfinite(TH) else np.nan,
            TB=float(TB) if np.isfinite(TB) else np.nan,
            R_ext_area=float(Rext) if np.isfinite(Rext) else np.nan,
            P_transport=float(P_transport) if np.isfinite(P_transport) else np.nan,
            P_MR=float(P_MR) if np.isfinite(P_MR) else np.nan,
            P_reject=float(np.asarray(P_reject).ravel()[0]) if np.size(P_reject) else np.nan,
            COP_reject=float(np.asarray(COP_reject).ravel()[0]) if np.size(COP_reject) else np.nan,
            **{k: cand.get(k, np.nan) for k in [
                "dp", "flow", "velocity", "dT_fluid", "width_mm", "height_mm", "N_channels",
                "dp_util", "flow_util", "velocity_util", "fluid_dT_util"
            ]}
        )
        if best is None or item["max_util"] < best["max_util"]:
            best = item
    return best if best is not None else dict(feasible=False, reason="no diagnostic candidate", max_util=np.nan)


def hw_optimize_at_q(q, tp, preset, reject_params, fixed_design=None):
    """Solve cold plate-only and hybrid MR hardware-aware optimization at one q.

    This preserves the original thermal constraints:
        DeltaT_H <= DeltaT
        DeltaT_B <= DeltaT

    It differs only in the external cooling model:
        R_ext and pump power come from the cold-plate hardware envelope.
    """
    cand_list = hw_design_candidates_for_q(q, tp, preset, fixed_design=fixed_design)
    if isinstance(cand_list, dict):
        cand_list = [cand_list] if cand_list.get("feasible", False) else []

    out_ext = []
    out_hyb = []
    for cand in cand_list:
        r0 = hw_eval_candidate(q, cand, tp, tp.A_mm2, reject_params, force_s_zero=True)
        if r0 is not None:
            out_ext.append(r0)
        r1 = hw_eval_candidate(q, cand, tp, tp.A_mm2, reject_params, force_s_zero=False)
        if r1 is not None:
            out_hyb.append(r1)

    best_ext = min(out_ext, key=lambda r: r["P_wall"]) if out_ext else None
    best_hyb = min(out_hyb, key=lambda r: r["P_wall"]) if out_hyb else None
    return best_ext, best_hyb


def hw_curve(q_grid, tp, preset, reject_params, sweep_mode, q_design):
    """Compute hardware-aware cold plate-only and hybrid curves.

    Design-envelope mode:
        Re-optimizes hardware geometry and flow independently at every q.

    Fixed-hardware mode:
        First designs a reference geometry at q_design, freezes that geometry,
        and then sweeps q by optimizing only flow. If the reference design is
        infeasible, the fixed-hardware curve is marked infeasible rather than
        silently reverting to design-envelope behavior.
    """
    fixed_design = None
    fixed_design_failed = False

    if sweep_mode.startswith("Fixed") and preset["kind"] != "passive":
        fixed_design = coldplate_channel_evaluate(q_design, tp.A_mm2, tp.DeltaT, tp.R_die, preset, reject_params)
        if not fixed_design.get("feasible", False):
            fixed_design_failed = True

    rows_ext = []
    rows_hyb = []

    for q in q_grid:
        if fixed_design_failed:
            ext, hyb = None, None
        else:
            use_design = fixed_design if sweep_mode.startswith("Fixed") else None
            ext, hyb = hw_optimize_at_q(float(q), tp, preset, reject_params, fixed_design=use_design)
        rows_ext.append(ext)
        rows_hyb.append(hyb)

    def pack(rows):
        keys = ["q","s","s_util","TH","TB","TH_util","TB_util","DeltaT_MR","minus_DeltaT_MR","P_wall","P_transport","P_reject","P_MR",
                "COP_total","COP_reject","R_ext_area","dp","flow","velocity","dT_fluid","width_mm","height_mm",
                "N_channels","dp_util","flow_util","velocity_util","fluid_dT_util"]
        out = {}
        for k in keys:
            out[k] = np.array([r.get(k, np.nan) if r is not None else np.nan for r in rows], dtype=float)
        out["feasible"] = np.array([r is not None for r in rows], dtype=bool)
        out["branch"] = np.array([r.get("branch", "infeasible") if r is not None else "infeasible" for r in rows], dtype=object)
        return out

    ext = pack(rows_ext)
    hyb = pack(rows_hyb)
    return ext, hyb, fixed_design

def infer_matching_preset_name(group, preset_dict):
    """Infer currently selected preset from widget values; robust to missing callback state."""
    matches = []
    for name, vals in preset_dict.items():
        ok = True
        for k, v in vals.items():
            if k not in group:
                ok = False
                break
            try:
                gv = getv(group, k)
                if not np.isclose(float(gv), float(v), rtol=1e-6, atol=1e-9):
                    ok = False
                    break
            except Exception:
                ok = False
                break
        if ok:
            matches.append(name)
    return ", ".join(matches) if matches else "custom/manual"

def hw_notes_box():
    out = widgets.Output(layout=widgets.Layout(
        border="1px solid #d8c07a",
        padding="10px 14px",
        margin="8px 0",
        width="100%"
    ))
    out.add_class("chip-note-box")
    with out:
        display(Markdown("### Hardware-aware hybrid MR optimization"))
        display(Markdown(
            "This tab couples the cold-plate hardware-envelope model directly into the original "
            "two-temperature hotspot/bulk microrefrigeration optimization. It uses the same chip, "
            "hotspot, spreading, and MR controls as the direct hotspot and architecture-aware chip models. "
            "The only replacement is the external cooling path: instead of the abstract "
            "`R0`, `Rinf`, `p_static`, and `p_scale` curve, the external resistance and wall power "
            "come from the selected cold-plate hardware model."
        ))

        display(Markdown("#### Inputs used by this tab"))
        display(Markdown(
            "The **chip model selector** controls whether the hardware-aware optimizer uses the "
            "reduced-order chip model or architecture-aware chip model. The chip scenario "
            "and hotspot archetype preset buttons below set the same variables used by the selected chip model section. "
            "The cold-plate and heat-rejection controls set the external cooling hardware."
        ))

        display(Markdown("#### Hotspot and multi-hotspot definitions"))
        for line in [
            r"\alpha_{\rm spot}=\frac{Q_{H,\rm spot}}{Q},\qquad \beta_{\rm spot}=\frac{A_{H,\rm spot}}{A}",
            r"\alpha_{\rm eff}=N_H\alpha_{\rm spot},\qquad \beta_{\rm eff}=N_H\beta_{\rm spot}",
            r"\gamma_H=\frac{\alpha_{\rm eff}}{\beta_{\rm eff}},\qquad \gamma_B=\frac{1-\alpha_{\rm eff}}{1-\beta_{\rm eff}}",
            r"q_H=\gamma_H q,\qquad q_B=\gamma_B q",
        ]:
            display(Math(line))

        display(Markdown(
            "The aggregate fractions alpha_eff and beta_eff define the chip-level power/area partition "
            "used in gamma_H and gamma_B. The local spreading penalty is still controlled by beta_spot, "
            "so multiple small hotspots do not automatically become as easy as one large continuous heated patch."
        ))

        display(Markdown("#### Spreading and resistance matrix"))
        for line in [
            r"R_{\rm sp,H}=\chi\left(\beta_{\rm spot}^{-p_{\rm sp}}-1\right)",
            r"R_{\rm sp,B}=\chi\left((1-\beta_{\rm spot})^{-p_{\rm sp}}-1\right)",
            r"R_c=\rho_c\sqrt{R_{\rm sp,H}R_{\rm sp,B}}",
            r"R^\downarrow=R_{\rm die}I+(1-\xi)\begin{bmatrix}R_{\rm sp,H}&R_c\\R_c&R_{\rm sp,B}\end{bmatrix}",
            r"R^\uparrow_{\rm hw}=\xi\begin{bmatrix}R_{\rm sp,H}&R_c\\R_c&R_{\rm sp,B}\end{bmatrix}+R_{\rm ext,hw}I",
        ]:
            display(Math(line))

        display(Markdown("#### Hardware-aware feasibility walls"))
        display(Markdown("This tab does not plot compact / reduced-order q0/qinf walls. In design-envelope mode, the external resistance changes with q because the hardware can be redesigned at every point, so a single R0/Rinf wall is not well-defined. In fixed-hardware mode, the geometry is fixed but the operating flow is still optimized; the useful walls are therefore the computed hardware-aware feasibility limits: passive/no-transport where applicable, cold plate-only, and cold plate + MR."))

        display(Markdown("#### Hardware-derived external resistance"))
        for line in [
            r"R_{\rm ext,hw}=R_{\rm contact}+R_{\rm conv}A+\frac{1}{2}\frac{\Delta T_{\rm fluid}}{q}",
            r"\Delta T_{\rm fluid}=\frac{qA}{\rho\dot V c_p}",
            r"P_{\rm transport}=\frac{\Delta p\,\dot V}{\eta_{\rm pump/fan}}",
        ]:
            display(Math(line))

        display(Markdown(
            "The hardware-derived R_ext,hw replaces the compact / reduced-order model's R_ext(u). "
            "It includes cold-plate contact resistance, convective resistance, and the average coolant/air "
            "temperature-rise penalty."
        ))

        display(Markdown("#### Same hotspot and bulk temperature constraints"))
        display(Markdown("In the plots and summary, ΔT_H is the hotspot temperature rise and ΔT_B is the bulk/background temperature rise. Both must remain below the allowed ΔT."))
        for line in [
            r"\Delta T_H=q\left[\gamma_HR^\downarrow_{HH}+\gamma_BR^\downarrow_{HB}+\gamma_H(1-s)R^\uparrow_{{\rm hw},HH}+\gamma_BR^\uparrow_{{\rm hw},HB}\right]\le\Delta T",
            r"\Delta T_B=q\left[\gamma_HR^\downarrow_{BH}+\gamma_BR^\downarrow_{BB}+\gamma_H(1-s)R^\uparrow_{{\rm hw},BH}+\gamma_BR^\uparrow_{{\rm hw},BB}\right]\le\Delta T",
        ]:
            display(Math(line))

        display(Markdown("#### Microrefrigerator power and heat rejection"))
        display(Markdown(
            "**Why s > 1 can reduce the bulk temperature in this network:** "
            "s=1 means the MR locally lifts the nominal hotspot heat before it enters the external path. "
            "s>1 means the MR cold side is modeled as a below-ambient sink at the hotspot region, so it can also pull heat laterally from the surrounding/bulk region through the hotspot-bulk coupling resistance. "
            "This is represented by the off-diagonal spreading/coupling term in the two-region thermal matrix. "
            "If one wants a strictly hotspot-only MR with no bulk cooling effect, s should be capped at 1 or the hotspot-bulk upstream coupling term should be removed for the MR branch."
        ))

        display(Markdown("The hardware-aware optimizer allows s to exceed 1 up to the user-set s max. For s>1, the MR overcools the hotspot-side branch below ambient; the plots show -DeltaT_MR as a positive quantity when this occurs."))
        for line in [
            r"Q_{\rm MR}=\alpha_{\rm eff}s\,qA",
            r"P_{\rm MR}=\frac{Q_{\rm MR}}{COP_\mu}",
            r"Q_{\rm reject}=qA+P_{\rm transport}+P_{\rm MR}",
            r"P_{\rm wall}=P_{\rm transport}+P_{\rm reject}+P_{\rm MR}",
            r"COP_{\rm system}=\frac{qA}{P_{\rm wall}}",
        ]:
            display(Math(line))

        display(Markdown("#### Cold plate-only objective"))
        for line in [
            r"s=0",
            r"\min_{\mathcal{G},\dot V}\left[P_{\rm transport}+P_{\rm reject}\right]",
        ]:
            display(Math(line))

        display(Markdown("#### Hybrid MR objective"))
        for line in [
            r"0\le s\le 1",
            r"\min_{s,\mathcal{G},\dot V}\left[P_{\rm transport}+P_{\rm reject}+P_{\rm MR}\right]",
        ]:
            display(Math(line))

        display(Markdown(
            "For a given hardware candidate, the tab sets s to the minimum value that satisfies "
            "the hotspot constraint, then checks the bulk constraint exactly as in the compact / reduced-order hybrid model. "
            "The optimizer then selects the feasible candidate with the lowest wall-plug cooling power."
        ))

        display(Markdown("#### Sweep modes"))
        display(Markdown("qmax is the plotted/swept endpoint, not a physical feasibility wall. The actual hardware-aware max feasible q is computed from the optimization and reported in the summary."))
        display(Markdown(
            "- **Design envelope:** re-optimizes cold-plate geometry and flow at every heat flux.\n"
            "- **Fixed hardware sweep:** designs the cold plate at q_target, freezes geometry (w, h, N_ch, footprint, and length), and sweeps heat flux by optimizing only flow velocity v. Flow rate, Reynolds number, Nusselt number, convective resistance, pressure drop, pump/fan power, fluid temperature rise, and heat-rejection burden still change with q."
        ))
    return out


hw_button = widgets.Button(description="Run / recompute hardware-aware hybrid MR", button_style="warning", icon="play")
hw_redraw_button = widgets.Button(description="Redraw plots only", button_style="", icon="paint-brush")
hw_output = widgets.Output()
hw_last_cache = None



def hw_reduced_order_reference_walls(tp, preset, reject_params, sweep_mode, q_design):
    """Infer reduced-order reference walls from the selected hardware-aware model.

    This avoids using the compact/reduced-order model's unrelated R0/Rinf values
    in the hardware-aware tab.

    Procedure:
      1. Use the selected hardware preset and q_target to choose a reference geometry.
      2. Sweep feasible flow rates through that geometry.
      3. Define R0_hw as the largest feasible external resistance (low-flow end).
      4. Define Rinf_hw as the smallest feasible external resistance (high-flow end).
      5. Compute q0/qinf walls using the same hotspot/bulk equations but with
         R_ext = R0_hw or Rinf_hw.

    For passive/conduction presets, R0_hw = Rinf_hw = R_contact + R_path.
    """
    e = matrix_elements(tp)
    gH, gB = gammas(tp)

    def walls_from_R(Rext):
        qH = tp.DeltaT / (
            gH*(e["RHHd"] + e["RHHu_const"] + Rext)
            + gB*(e["RHBd"] + e["RHBu"])
        )
        qB = tp.DeltaT / (
            gH*(e["RBHd"] + e["RBHu"])
            + gB*(e["RBBd"] + e["RBBu_const"] + Rext)
        )
        return qH, qB

    qdieH = tp.DeltaT / (gH*e["RHHd"] + gB*(e["RHBd"] + e["RHBu"]))

    if preset["kind"] == "passive":
        R0_hw = preset["R_contact_area"] + preset["R_path_area"]
        Rinf_hw = R0_hw
        q0H_hw, q0B_hw = walls_from_R(R0_hw)
        qinfH_hw, qinfB_hw = walls_from_R(Rinf_hw)
        return dict(
            R0_hw=R0_hw, Rinf_hw=Rinf_hw,
            q0H_hw=q0H_hw, q0B_hw=q0B_hw,
            qinfH_hw=qinfH_hw, qinfB_hw=qinfB_hw,
            qdieH=qdieH, reference_design=None,
            note="passive/conduction path: R0_hw = Rinf_hw"
        )

    ref_design = coldplate_channel_evaluate(q_design, tp.A_mm2, tp.DeltaT, tp.R_die, preset, reject_params)
    if not ref_design.get("feasible", False):
        return dict(
            R0_hw=np.nan, Rinf_hw=np.nan,
            q0H_hw=np.nan, q0B_hw=np.nan,
            qinfH_hw=np.nan, qinfB_hw=np.nan,
            qdieH=qdieH, reference_design=None,
            note="no feasible reference hardware design at q_target"
        )

    cand = hw_ext_resistance_temperature(
        q_design, tp.A_mm2, tp.R_die, preset, design=ref_design, v=None
    )
    if isinstance(cand, dict):
        cand = [cand] if cand.get("feasible", False) else []

    Rvals = np.array([c.get("R_ext_area", np.nan) for c in cand], dtype=float)
    Rvals = Rvals[np.isfinite(Rvals)]
    if len(Rvals) == 0:
        return dict(
            R0_hw=np.nan, Rinf_hw=np.nan,
            q0H_hw=np.nan, q0B_hw=np.nan,
            qinfH_hw=np.nan, qinfB_hw=np.nan,
            qdieH=qdieH, reference_design=ref_design,
            note="no feasible reference-flow candidates"
        )

    R0_hw = float(np.nanmax(Rvals))
    Rinf_hw = float(np.nanmin(Rvals))
    q0H_hw, q0B_hw = walls_from_R(R0_hw)
    qinfH_hw, qinfB_hw = walls_from_R(Rinf_hw)
    return dict(
        R0_hw=R0_hw, Rinf_hw=Rinf_hw,
        q0H_hw=q0H_hw, q0B_hw=q0B_hw,
        qinfH_hw=qinfH_hw, qinfB_hw=qinfB_hw,
        qdieH=qdieH,
        reference_design=ref_design,
        note="hardware-inferred reduced-order walls from q_target reference geometry"
    )


def hw_fmt(x, unit="", digits=6):
    try:
        if x is None or not np.isfinite(x):
            return "N/A"
        return f"{float(x):.{digits}g}{(' ' + unit) if unit else ''}"
    except Exception:
        return "N/A"

def hw_html_table(title, rows):
    body = "".join(
        f"<tr><td style='padding:4px 10px'><b>{k}</b></td><td style='padding:4px 10px'>{v}</td></tr>"
        for k, v in rows
    )
    return widgets.HTML(
        f"<h4>{title}</h4>"
        "<table style='border-collapse:collapse; margin:6px 0 12px 0'>"
        f"{body}</table>"
    )

def hw_results_table(title, rows):
    if not rows:
        return widgets.HTML("")
    headers = rows[0].keys()
    head = "".join(f"<th style='padding:4px 8px'>{h}</th>" for h in headers)
    body = ""
    for r in rows:
        body += "<tr>" + "".join(f"<td style='padding:4px 8px'>{r.get(h, 'N/A')}</td>" for h in headers) + "</tr>"
    return widgets.HTML(
        f"<h4>{title}</h4>"
        "<table style='border-collapse:collapse; margin:6px 0 12px 0'>"
        f"<tr>{head}</tr>{body}</table>"
    )

def hw_safe_max_feasible(q_grid, curve):
    if curve is not None and np.any(curve["feasible"]):
        return float(np.nanmax(np.where(curve["feasible"], q_grid, np.nan)))
    return np.nan

def hw_nearest_row(curve, q_grid, qval):
    idx = int(np.nanargmin(np.abs(q_grid - qval)))
    row = {}
    for k, v in curve.items():
        try:
            if hasattr(v, "__len__") and len(v) == len(q_grid):
                row[k] = v[idx]
        except TypeError:
            pass
    return row, idx

def hw_passive_no_transport_limit(tp, preset):
    """Thermal-only passive/no-transport feasibility wall, if applicable.

    This is meaningful only for passive/conduction presets.  For active
    forced-flow cold plates and heatsinks, zero-flow/natural-convection behavior
    is not represented by the channel-flow model, so the passive wall is N/A.
    """
    if preset.get("kind") != "passive":
        return dict(q=np.nan, Q=np.nan, applicable=False, note="N/A for active forced-flow presets")

    Rext = preset["R_contact_area"] + preset["R_path_area"]
    TH1, TB1 = hw_temperatures(1.0, Rext, 0.0, tp)
    candidates = []
    if np.isfinite(TH1) and TH1 > 0:
        candidates.append(tp.DeltaT/TH1)
    if np.isfinite(TB1) and TB1 > 0:
        candidates.append(tp.DeltaT/TB1)
    qlim = float(np.nanmin(candidates)) if candidates else np.nan
    return dict(q=qlim, Q=qlim*tp.A_mm2 if np.isfinite(qlim) else np.nan, applicable=True, note="passive/conduction thermal wall, s=0")

def hw_get_context():
    """Collect current hardware-aware input state without running the solver."""
    bp = build_params()
    if len(bp) == 4:
        tp, state, pc, meta = bp
    else:
        tp, state, pc = bp
        meta = dict(N_H=np.nan, alpha_spot=np.nan, beta_spot=getattr(tp, "beta_spot", tp.beta),
                    alpha_eff=tp.alpha_tot, beta_eff=tp.beta)

    preset_name = cp_preset.value
    preset = COLD_PLATE_PRESETS[preset_name]
    sweep_mode = cp_sweep_mode.value
    q_plot_min = getv(hw_plot_controls, "qmin")
    q_plot_max = getv(hw_plot_controls, "qmax")
    if q_plot_max <= q_plot_min:
        q_plot_max = q_plot_min + 0.01
    q_target = cp_controls["qtarget"]["text"].value

    reject_params = dict(
        model=cp_reject_model.value,
        economizer_COP=cp_controls["economizer_COP"]["text"].value,
        T_supply_C=cp_controls["T_supply_C"]["text"].value,
        T_amb_C=cp_controls["T_amb_C"]["text"].value,
        reject_approach_K=cp_controls["reject_approach_K"]["text"].value,
        T_chw_C=cp_controls["T_chw_C"]["text"].value,
        T_cw_C=cp_controls["T_cw_C"]["text"].value,
        T_chw_ref_C=cp_controls["T_chw_ref_C"]["text"].value,
        T_cw_ref_C=cp_controls["T_cw_ref_C"]["text"].value,
        COP_ref=cp_controls["GN_COP_ref"]["text"].value,
        Q_ref_kW=cp_controls["GN_Q_ref_kW"]["text"].value,
        PLR_max=cp_controls["GN_PLR_max"]["text"].value,
        a1_kW_per_K=cp_controls["GN_a1"]["text"].value,
        a3_K_per_kW=cp_controls["GN_a3"]["text"].value,
    )
    return tp, state, pc, meta, preset_name, preset, sweep_mode, q_plot_min, q_plot_max, q_target, reject_params

def hw_compute_cache():
    tp, state, pc, meta, preset_name, preset, sweep_mode, q_plot_min, q_plot_max, q_target, reject_params = hw_get_context()

    # The q-grid is computed only when the user explicitly runs/recomputes.
    base_grid = np.linspace(max(1e-8, q_plot_min), q_plot_max, 260)
    important = np.array([q_target], dtype=float)
    important = important[(important > 0) & np.isfinite(important) & (important >= q_plot_min) & (important <= q_plot_max)]
    q_grid = np.unique(np.clip(np.r_[base_grid, important], max(1e-8, q_plot_min), q_plot_max))

    ext, hyb, fixed_design = hw_curve(q_grid, tp, preset, reject_params, sweep_mode, q_target)

    ext_target, hyb_target = hw_optimize_at_q(
        q_target, tp, preset, reject_params,
        fixed_design=fixed_design if (sweep_mode.startswith("Fixed") and fixed_design and fixed_design.get("feasible", False)) else None
    )

    fixed_for_diag = fixed_design if (sweep_mode.startswith("Fixed") and fixed_design and fixed_design.get("feasible", False)) else None
    ext_diag_target = hw_candidate_diagnostics(q_target, tp, preset, reject_params, fixed_design=fixed_for_diag, force_s_zero=True)
    hyb_diag_target = hw_candidate_diagnostics(q_target, tp, preset, reject_params, fixed_design=fixed_for_diag, force_s_zero=False)

    return dict(
        tp=tp, state=state, pc=pc, meta=meta,
        preset_name=preset_name, preset=preset, sweep_mode=sweep_mode,
        q_grid=q_grid, q_target=q_target,
        reject_params=reject_params,
        ext=ext, hyb=hyb, fixed_design=fixed_design,
        ext_target=ext_target, hyb_target=hyb_target,
        ext_diag_target=ext_diag_target, hyb_diag_target=hyb_diag_target,
        q_plot_min_compute=q_plot_min, q_plot_max_compute=q_plot_max,
        passive_wall=hw_passive_no_transport_limit(tp, preset)
    )

def hw_render_cached_result(cache, redraw_only=False):
    if cache is None:
        display(Markdown("No cached hardware-aware result is available. Click **Run / recompute hardware-aware hybrid MR** first."))
        return

    # Visual controls are read at render time.  This lets users update y-limits
    # and x-limits without recalculating the hardware curves.
    q_plot_min = getv(hw_plot_controls, "qmin")
    q_plot_max = getv(hw_plot_controls, "qmax")
    if q_plot_max <= q_plot_min:
        q_plot_max = q_plot_min + 0.01

    tp = cache["tp"]
    meta = cache["meta"]
    preset_name = cache["preset_name"]
    preset = cache["preset"]
    sweep_mode = cache["sweep_mode"]
    q_grid = cache["q_grid"]
    q_target = cache["q_target"]
    reject_params = cache["reject_params"]
    ext = cache["ext"]
    hyb = cache["hyb"]
    fixed_design = cache["fixed_design"]
    ext_target = cache["ext_target"]
    hyb_target = cache["hyb_target"]
    ext_diag_target = cache["ext_diag_target"]
    hyb_diag_target = cache["hyb_diag_target"]
    passive_wall = cache["passive_wall"]

    gamma_H, gamma_B = gammas(tp)
    ext_max = hw_safe_max_feasible(q_grid, ext)
    hyb_max = hw_safe_max_feasible(q_grid, hyb)

    if redraw_only:
        display(Markdown(
            f"**Redraw only:** using cached calculation over q = "
            f"{cache['q_plot_min_compute']:.4g}–{cache['q_plot_max_compute']:.4g} W/mm². "
            "Click Run/recompute if you changed the desired sampled q-range."
        ))

    display(Markdown("### Hardware-aware hybrid MR optimization"))
    display(Markdown(
        f"Cold-plate preset: **{preset_name}**; heat rejection: **{reject_params['model']}**; "
        f"sweep mode: **{sweep_mode}**."
    ))

    fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

    # COP comparison.
    cop_ymax = getv(hw_plot_controls, "cop_ymax")
    cop_clip_level = 0.94 * cop_ymax

    def plot_hw_cop(ax, curve, color, ls, lw, label):
        """Plot COP, making off-scale curves visibly clipped inside the axis.

        If all feasible COP values exceed the chosen y-axis maximum, plotting at y_max
        puts the line on top of the axis spine and it can appear invisible.  We instead
        draw clipped points/segments at 94% of y_max and label them explicitly.
        """
        mask = curve["feasible"] & np.isfinite(curve["COP_total"])
        x = q_grid[mask]
        y = curve["COP_total"][mask]
        if len(x) == 0:
            ax.plot([], [], color=color, ls=ls, lw=lw, label=label + " (no feasible points)")
            return

        clipped = y > cop_ymax
        y_plot = np.where(clipped, cop_clip_level, y)
        suffix = " (clipped)" if np.any(clipped) else ""
        ax.plot(x, y_plot, color=color, ls=ls, lw=lw, label=label + suffix)

        if np.any(clipped):
            step = max(1, len(x[clipped]) // 12)
            ax.scatter(
                x[clipped][::step],
                np.full(len(x[clipped][::step]), cop_clip_level),
                color=color,
                marker="^",
                s=28,
                zorder=5,
                label=label + " > y max"
            )
            # Add a small in-plot annotation for the all-clipped case.
            if np.all(clipped):
                xmid = x[len(x)//2]
                ax.text(
                    xmid, 0.86 * cop_ymax,
                    f"{label} > {cop_ymax:g}",
                    color=color,
                    ha="center",
                    va="center",
                    fontsize=8,
                    bbox=dict(facecolor="white", alpha=0.75, edgecolor=color)
                )

    plot_hw_cop(axs[0,0], ext, "black", "--", 2.2, "Cold plate-only COP")
    plot_hw_cop(axs[0,0], hyb, "tab:blue", "-", 2.8, "Cold plate + MR COP")
    axs[0,0].set_ylabel("COP = Q_chip / P_wall [-]")
    axs[0,0].set_title("COP comparison")
    axs[0,0].set_ylim(0, cop_ymax)

    # Power breakdown.
    axs[0,1].plot(q_grid[ext["feasible"]], ext["P_wall"][ext["feasible"]], "k--", lw=2, label="Cold plate-only total")
    axs[0,1].plot(q_grid[hyb["feasible"]], hyb["P_wall"][hyb["feasible"]], lw=2.5, label="Cold plate + MR total")
    axs[0,1].plot(q_grid[hyb["feasible"]], hyb["P_transport"][hyb["feasible"]], lw=1.7, label="Transport")
    axs[0,1].plot(q_grid[hyb["feasible"]], hyb["P_reject"][hyb["feasible"]], lw=1.7, label="Reject/chiller")
    axs[0,1].plot(q_grid[hyb["feasible"]], hyb["P_MR"][hyb["feasible"]], lw=1.7, label="MR")
    axs[0,1].set_ylabel("Power [W]")
    axs[0,1].set_title("Wall-plug cooling power")
    axs[0,1].set_ylim(0, getv(hw_plot_controls, "power_ymax"))

    # Temperature constraints.
    axs[1,0].plot(q_grid[ext["feasible"]], ext["TH"][ext["feasible"]], "k--", lw=2, marker="o", ms=2.5, label="Cold plate-only ΔT_H")
    axs[1,0].plot(q_grid[ext["feasible"]], ext["TB"][ext["feasible"]], "k:", lw=2, marker="o", ms=2.5, label="Cold plate-only ΔT_B")
    axs[1,0].plot(q_grid[hyb["feasible"]], hyb["TH"][hyb["feasible"]], lw=2, label="Cold plate + MR ΔT_H")
    axs[1,0].plot(q_grid[hyb["feasible"]], hyb["TB"][hyb["feasible"]], lw=2, label="Cold plate + MR ΔT_B")
    if "minus_DeltaT_MR" in hyb and np.any(hyb["feasible"]):
        axs[1,0].plot(
            q_grid[hyb["feasible"]],
            hyb["minus_DeltaT_MR"][hyb["feasible"]],
            lw=2,
            ls="--",
            label="-ΔT_MR below ambient [K]"
        )
    axs[1,0].axhline(tp.DeltaT, color="gray", ls="-.", lw=1, label="ΔT limit")
    axs[1,0].set_ylabel("Temperature rise [K]")
    axs[1,0].set_title("Temperature rises and MR cold-side overcooling")
    axs[1,0].set_ylim(0, getv(hw_plot_controls, "temp_ymax"))

    # Optimal variables and hardware operating point.
    axs[1,1].plot(q_grid[hyb["feasible"]], hyb["s"][hyb["feasible"]], lw=2, label="MR fraction s*")
    if np.any(hyb["feasible"]):
        axs[1,1].plot(q_grid[hyb["feasible"]], hyb["flow"][hyb["feasible"]]*60000, lw=2, label="Hybrid flow [L/min]")
        axs[1,1].plot(q_grid[hyb["feasible"]], hyb["dp"][hyb["feasible"]]/1000, lw=2, label="Hybrid Δp [kPa]")
    if np.any(ext["feasible"]):
        axs[1,1].plot(q_grid[ext["feasible"]], ext["flow"][ext["feasible"]]*60000, "k--", lw=1.8, label="Cold plate-only flow [L/min]")
    axs[1,1].set_ylabel("Optimal variables")
    axs[1,1].set_title("MR fraction and hardware operating point")
    axs[1,1].set_ylim(0, getv(hw_plot_controls, "var_ymax"))

    ref_specs = []
    if passive_wall.get("applicable", False) and np.isfinite(passive_wall.get("q", np.nan)):
        ref_specs.append(("passive/no-transport qmax", passive_wall["q"], "gray", ":", 1.3))
    if np.isfinite(ext_max):
        ref_specs.append(("cold plate-only qmax", ext_max, "black", "-.", 1.3))
    if np.isfinite(hyb_max):
        ref_specs.append(("cold plate + MR qmax", hyb_max, "tab:blue", "-.", 1.3))
    ref_specs.append(("q target", q_target, "tab:orange", ":", 1.2))

    def add_reference_lines(ax):
        for _, xval, color, ls, lw in ref_specs:
            ax.axvline(xval, color=color, ls=ls, lw=lw, label="_nolegend_")

    for ax in axs.flat:
        ax.set_xlim(q_plot_min, q_plot_max)
        ax.set_xlabel("Average chip heat flux q [W/mm²]")
        ax.tick_params(axis="x", which="both", bottom=True, labelbottom=True)
        try:
            add_hotspot_top_axis(ax, gamma_H)
        except NameError:
            pass
        add_reference_lines(ax)
        ax.grid(True, alpha=0.3)

    for ax in axs.flat:
        handles, labels = ax.get_legend_handles_labels()
        unique = {}
        for h, lab in zip(handles, labels):
            if lab not in unique and lab != "_nolegend_":
                unique[lab] = h
        ax.legend(unique.values(), unique.keys(), fontsize=7)

    from matplotlib.lines import Line2D
    ref_handles = [Line2D([0], [0], color=c, ls=ls, lw=lw) for _, _, c, ls, lw in ref_specs]
    ref_labels = [lab for lab, _, _, _, _ in ref_specs]
    fig.legend(ref_handles, ref_labels, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=4, fontsize=8, frameon=True)

    fig.tight_layout(rect=[0, 0, 1, 0.90])
    plt.show()

    display(Markdown(
        "**Plot interpretation:** cold plate-only curves are shown only where the no-MR case is feasible. "
        "If cold plate-only qmax is far below q_target, those curves exist only near the left side of the plot. "
        "COP values above the selected COP y max are drawn at 94% of the y-axis height with triangle markers and an in-plot label, rather than exactly on the axis spine, so they remain visible. Increase COP y max to see their actual height."
    ))


    def cop_range_row(name, curve):
        mask = curve["feasible"] & np.isfinite(curve["COP_total"])
        if not np.any(mask):
            return {
                "case": name,
                "feasible q range [W/mm²]": "none",
                "min COP": "N/A",
                "max COP": "N/A",
                "points > COP y max": "0"
            }
        qvals = q_grid[mask]
        cops = curve["COP_total"][mask]
        return {
            "case": name,
            "feasible q range [W/mm²]": f"{np.nanmin(qvals):.6g}–{np.nanmax(qvals):.6g}",
            "min COP": hw_fmt(np.nanmin(cops), ""),
            "max COP": hw_fmt(np.nanmax(cops), ""),
            "points > COP y max": f"{int(np.sum(cops > cop_ymax))}/{len(cops)}"
        }

    display(hw_results_table("COP ranges over feasible points", [
        cop_range_row("cold plate-only", ext),
        cop_range_row("cold plate + MR", hyb),
    ]))

    # Preset inference from current widget values.
    if mode.value == "Compact / reduced-order model":
        chip_inferred = infer_matching_preset_name(thermal, CHIP_PRESETS_THERMAL)
        hotspot_inferred = infer_matching_preset_name(thermal, HOTSPOT_ARCHETYPES_THERMAL)
    else:
        chip_inferred = infer_matching_preset_name(arch, CHIP_PRESETS_ARCH)
        hotspot_inferred = infer_matching_preset_name(arch, HOTSPOT_ARCHETYPES_ARCH)

    selected_chip_name = globals().get("selected_chip_preset_name", None)
    selected_hotspot_name = globals().get("selected_hotspot_preset_name", None)
    preset_label = selected_hotspot_name or selected_chip_name or hotspot_inferred or chip_inferred or "custom/manual"
    if preset_label in ["custom/manual", "", None]:
        preset_label = chip_inferred if chip_inferred != "custom/manual" else hotspot_inferred

    display(hw_html_table("Run context", [
        ("chip model", mode.label),
        ("chip / hotspot preset", preset_label),
        ("cold plate preset", preset_name),
        ("heat rejection", reject_params["model"] if reject_params["model"] != "Transport only / ideal downstream sink" else "Transport only / ideal downstream sink (diagnostic)"),
        ("sweep mode", sweep_mode),
        ("hardware provenance", preset.get("calibration_status", "unknown")),
    ]))

    display(hw_results_table("Inputs at q_target", [
        {"quantity": "q_target", "value": hw_fmt(q_target, "W/mm²")},
        {"quantity": "N_H", "value": hw_fmt(meta.get("N_H", np.nan), "")},
        {"quantity": "alpha_spot", "value": hw_fmt(meta.get("alpha_spot", np.nan), "")},
        {"quantity": "beta_spot", "value": hw_fmt(getattr(tp, "beta_spot", tp.beta), "")},
        {"quantity": "alpha_eff", "value": hw_fmt(tp.alpha_tot, "")},
        {"quantity": "beta_eff", "value": hw_fmt(tp.beta, "")},
        {"quantity": "gamma_H", "value": hw_fmt(gamma_H, "")},
        {"quantity": "gamma_B", "value": hw_fmt(gamma_B, "")},
        {"quantity": "COP_micro", "value": hw_fmt(tp.COP_micro, "")},
        {"quantity": "DeltaT", "value": hw_fmt(tp.DeltaT, "K")},
        {"quantity": "A", "value": hw_fmt(tp.A_mm2, "mm²")},
    ]))

    display(hw_results_table("Hardware-aware feasibility walls / qmax", [
        {"wall": "passive / no transport", "q [W/mm²]": hw_fmt(passive_wall.get("q", np.nan), ""), "Q [W]": hw_fmt(passive_wall.get("Q", np.nan), ""), "note": passive_wall.get("note", "")},
        {"wall": "cold plate-only, optimized hardware", "q [W/mm²]": hw_fmt(ext_max, ""), "Q [W]": hw_fmt(ext_max*tp.A_mm2 if np.isfinite(ext_max) else np.nan, ""), "note": "s=0"},
        {"wall": "cold plate + MR, optimized hardware", "q [W/mm²]": hw_fmt(hyb_max, ""), "Q [W]": hw_fmt(hyb_max*tp.A_mm2 if np.isfinite(hyb_max) else np.nan, ""), "note": "s optimized"},
    ]))

    def row_for_target(name, r):
        if r is None:
            return {
                "case": name, "feasible": "False", "COP": "N/A", "P_wall [W]": "N/A",
                "P_transport [W]": "N/A", "P_reject [W]": "N/A", "P_MR [W]": "N/A",
                "DeltaT_H [K]": "N/A", "DeltaT_B [K]": "N/A", "s*": "N/A",
                "R_ext,hw [K mm²/W]": "N/A", "-DeltaT_MR [K]": "N/A"
            }
        return {
            "case": name,
            "feasible": str(bool(r.get("feasible", False))),
            "COP": hw_fmt(r.get("COP_total", np.nan), ""),
            "P_wall [W]": hw_fmt(r.get("P_wall", np.nan), ""),
            "P_transport [W]": hw_fmt(r.get("P_transport", np.nan), ""),
            "P_reject [W]": hw_fmt(r.get("P_reject", np.nan), ""),
            "P_MR [W]": hw_fmt(r.get("P_MR", np.nan), ""),
            "DeltaT_H [K]": hw_fmt(r.get("TH", np.nan), ""),
            "DeltaT_B [K]": hw_fmt(r.get("TB", np.nan), ""),
            "s*": hw_fmt(r.get("s", np.nan), ""),
            "R_ext,hw [K mm²/W]": hw_fmt(r.get("R_ext_area", np.nan), ""),
            "-DeltaT_MR [K]": hw_fmt(r.get("minus_DeltaT_MR", np.nan), ""),
        }

    display(hw_results_table(f"Optimization result at q_target = {hw_fmt(q_target, 'W/mm²')}", [
        row_for_target("cold plate-only", ext_target),
        row_for_target("cold plate + MR", hyb_target),
    ]))

    feasible_rows = []
    for name, r in [("cold plate-only", ext_target), ("cold plate + MR", hyb_target)]:
        if r is not None and r.get("feasible", False):
            feasible_rows.append({
                "case": name,
                "N_channels": hw_fmt(r.get("N_channels", np.nan), ""),
                "width [mm]": hw_fmt(r.get("width_mm", np.nan), ""),
                "height [mm]": hw_fmt(r.get("height_mm", np.nan), ""),
                "flow [L/min]": hw_fmt(r.get("flow", np.nan)*60000, ""),
                "velocity [m/s]": hw_fmt(r.get("velocity", np.nan), ""),
                "dp [Pa]": hw_fmt(r.get("dp", np.nan), ""),
                "Re": hw_fmt(r.get("Re", np.nan), ""),
                "Nu": hw_fmt(r.get("Nu", np.nan), ""),
                "fluid dT [K]": hw_fmt(r.get("dT_fluid", np.nan), ""),
                "dp util": hw_fmt(r.get("dp_util", np.nan), ""),
                "flow util": hw_fmt(r.get("flow_util", np.nan), ""),
                "velocity util": hw_fmt(r.get("velocity_util", np.nan), ""),
                "fluid dT util": hw_fmt(r.get("fluid_dT_util", np.nan), ""),
            })
    if feasible_rows:
        display(hw_results_table("Cold-plate optimal geometry and operating point at q_target", feasible_rows))

    diagnostic_rows = []
    for name, diag in [("cold plate-only", cache.get("ext_diag_target")), ("cold plate + MR", cache.get("hyb_diag_target"))]:
        if diag is None:
            continue
        # Only show diagnostics when target result is infeasible.
        target = ext_target if name == "cold plate-only" else hyb_target
        if target is not None and target.get("feasible", False):
            continue
        ratios = diag.get("ratios", {})
        if name == "cold plate-only":
            s_status = "s = 0 fixed (no MR)"
        else:
            s_status = f"s_required = {hw_fmt(diag.get('s_required', np.nan), '')}; s/s_max = {hw_fmt(ratios.get('s', np.nan), '')}"
        diagnostic_rows.append({
            "case": name,
            "why shown": "infeasible at q_target",
            "likely limiting reason(s)": diag.get("reason", "unknown"),
            "hotspot util": hw_fmt(ratios.get("hotspot", np.nan), ""),
            "bulk util": hw_fmt(ratios.get("bulk", np.nan), ""),
            "MR status": s_status,
            "dp util": hw_fmt(ratios.get("dp", np.nan), ""),
            "flow util": hw_fmt(ratios.get("flow", np.nan), ""),
            "velocity util": hw_fmt(ratios.get("velocity", np.nan), ""),
            "fluid dT util": hw_fmt(ratios.get("fluid_dT", np.nan), ""),
            "reject util": hw_fmt(ratios.get("reject", np.nan), ""),
        })
    if diagnostic_rows:
        display(Markdown(
            "**Infeasibility diagnostics:** this table lists only cases that are infeasible at q_target. "
            "A cold plate-only row has s = 0 by definition; it is not the same as the cold plate + MR solution."
        ))
        display(hw_results_table(f"Infeasibility diagnostics at q_target = {hw_fmt(q_target, 'W/mm²')}", diagnostic_rows))

def run_hardware_aware_hybrid(_=None):
    global hw_last_cache
    try:
        output_thermal.clear_output(wait=True)
        output_arch.clear_output(wait=True)
        cp_output.clear_output(wait=True)
    except NameError:
        pass

    with hw_output:
        clear_output(wait=True)
        try:
            hw_last_cache = hw_compute_cache()
            hw_render_cached_result(hw_last_cache, redraw_only=False)
        except Exception as exc:
            print("Error:", repr(exc))

def redraw_hardware_aware_hybrid(_=None):
    with hw_output:
        clear_output(wait=True)
        try:
            hw_render_cached_result(hw_last_cache, redraw_only=True)
        except Exception as exc:
            print("Error:", repr(exc))

hw_button.on_click(run_hardware_aware_hybrid)
hw_redraw_button.on_click(redraw_hardware_aware_hybrid)


# Input views for the hardware-aware optimizer.
# These are the same widget objects used by the Compact / reduced-order and Architecture-aware tabs,
# so changing them here changes the actual variables used by build_params().
hw_reduced_chip_menu = widgets.VBox([
    widgets.HTML("<h3>Reduced-order chip model</h3>"),
    widgets.HTML("<small>These chip/hotspot controls feed directly into the hardware-aware cooling optimization. The equations below use R_ext,hw from the cold-plate hardware model, not R0/Rinf.</small>"),
    collapsed_section("Reduced-order chip model equations for hardware-aware cooling", chip_model_equations_reduced_hardware_box()),
    collapsed_section("Chip/hotspot preset notes and definitions", make_chip_model_notes_box()),
    chip_hotspot_selector_thermal("Reduced-order chip model"),
    widgets.HTML("<b>Hotspot fraction inputs</b>"),
    two_col([thermal[k]["box"] for k in ["N_H", "alpha_tot", "beta", "COP_micro"]]),
    effective_fraction_readout_thermal(),
    widgets.HTML("<b>Chip/package and spreading inputs</b>"),
    two_col([thermal[k]["box"] for k in ["A_mm2", "DeltaT", "R_die", "chi", "p_sp", "rho_c", "xi"]])
])

hw_architecture_chip_menu = widgets.VBox([
    widgets.HTML("<h3>Architecture-aware chip model</h3>"),
    widgets.HTML("<small>These architecture-aware controls feed directly into the hardware-aware cooling optimization. The equations below use R_ext,hw from the cold-plate hardware model, not R0/Rinf.</small>"),
    collapsed_section("Architecture-aware chip model equations for hardware-aware cooling", chip_model_equations_arch_hardware_box()),
    collapsed_section("Chip/hotspot preset notes and definitions", make_chip_model_notes_box()),
    chip_hotspot_selector_arch("Architecture-aware chip model"),
    widgets.HTML("<b>Hotspot fraction inputs and derived fractions</b>"),
    two_col([arch[k]["box"] for k in ["N_H", "beta", "COP_micro", "lam"]]),
    effective_fraction_readout_arch(),
    widgets.HTML("<b>Chip/package and spreading inputs</b>"),
    two_col([arch[k]["box"] for k in ["A_mm2", "DeltaT", "R_die", "chi", "p_sp", "rho_c", "xi"]]),
    widgets.HTML("<b>Architecture and power-state inputs</b>"),
    two_col([arch[k]["box"] for k in [
        "V0", "f0", "Q_dyn0", "Q_leak0", "alpha_dyn0", "alpha_leak0",
        "Vth", "nu", "eta_D", "sub_n", "V_t", "a", "b"
    ]])
])

# Backward-compatible names used by update_chip_source_menus().
hw_thermal_inputs = hw_reduced_chip_menu
hw_arch_inputs = hw_architecture_chip_menu






# Dynamic input menus controlled by the chip-input-source toggle.
# The same ToggleButtons widget is shown in both top-level tabs. Changing it
# updates both visible input menus automatically.
reduced_order_source_box = widgets.VBox()
hardware_source_box = widgets.VBox()

def update_chip_source_menus(change=None):
    if mode.value == "Compact / reduced-order model":
        reduced_order_source_box.children = reduced_order_source_menu.children
        hardware_source_box.children = hw_reduced_chip_menu.children
    else:
        reduced_order_source_box.children = architecture_source_menu.children
        hardware_source_box.children = hw_architecture_chip_menu.children

mode.observe(update_chip_source_menus, names="value")
update_chip_source_menus()

reduced_order_tab = widgets.VBox([
    collapsed_section("Nomenclature / model definitions", nomenclature_note),
    widgets.HTML("<h3>Reduced-order cooling surrogate</h3>"),
    widgets.HTML("<small>Decision tree: choose chip model → apply chip/hotspot presets → apply reduced-order cooling preset → adjust controls → run.</small>"),
    mode,
    reduced_order_source_box
])


hw_opt_controls = {}
hw_opt_controls["smax"] = linked_float("s max", 3.0, 1.0, 20.0, 0.1, "[-]")

def hw_s_max():
    try:
        return max(1.0, getv(hw_opt_controls, "smax"))
    except Exception:
        return 3.0

hw_plot_controls = {}
hw_plot_controls["qmin"] = linked_float("q plot min", 0.0, 0.0, 50.0, 0.01, "[W/mm²]")
hw_plot_controls["qmax"] = linked_float("q plot max", 5.0, 0.01, 50.0, 0.05, "[W/mm²]")
hw_plot_controls["cop_ymax"] = linked_float("COP y max", 10000.0, 1.0, 1000000.0, 50.0, "[-]")
hw_plot_controls["power_ymax"] = linked_float("power y max", 500.0, 1.0, 100000.0, 10.0, "[W]")
hw_plot_controls["temp_ymax"] = linked_float("temperature y max", 100.0, 5.0, 500.0, 5.0, "[K]")
hw_plot_controls["var_ymax"] = linked_float("variables y max", 20.0, 0.5, 10000.0, 1.0, "mixed units")

hw_plot_eqs = [
    r"x=q\quad[\mathrm{W/mm^2}]",
    r"q_{\rm plot,min}\ \mathrm{and}\ q_{\rm plot,max}\mathrm{\ set\ the\ x-axis\ display\ range;\ Run/recompute\ also\ resamples\ this\ q\ range.}",
    r"\mathrm{Redraw\ plots\ only\ changes\ visual\ limits\ from\ the\ cached\ calculation;\ it\ does\ not\ recompute\ the\ curves.}",
    r"\mathrm{Vertical\ lines\ show\ passive/no-transport,\ cold\ plate-only,\ cold\ plate+MR,\ and\ q_{target}\ reference\ points.}"
]


def run_coldplate_diagnostic_from_hardware_tab(_=None):
    """Run standalone cold-plate preset diagnostics from the Hardware-aware tab.

    This reuses the exact same cold-plate diagnostic function/model as the old
    Cold plate presets tab, but synchronizes A, DeltaT, R_die, qmax, q_target,
    cold-plate preset, sweep mode, and heat-rejection settings with the current
    Hardware-aware cooling model controls.
    """
    try:
        tp, state, pc, meta, preset_name, preset, sweep_mode, q_plot_min, q_plot_max, q_target, reject_params = hw_get_context()

        # Synchronize the old cold-plate diagnostic controls to the current
        # hardware-aware chip/cold-plate setup.
        setv(cp_controls, "A_mm2", tp.A_mm2)
        setv(cp_controls, "DeltaT", tp.DeltaT)
        setv(cp_controls, "R_die", tp.R_die)
        setv(cp_controls, "qmax", q_plot_max)
        setv(cp_controls, "qtarget", q_target)

        # cp_preset, cp_sweep_mode, cp_reject_model, and heat-rejection controls
        # are shared widgets, so they already match.
        run_coldplate_envelope(_)

    except Exception as exc:
        with cp_output:
            clear_output(wait=True)
            print("Error running synchronized cold-plate diagnostics:", repr(exc))

hw_cp_diag_button = widgets.Button(
    description="Run cold-plate preset diagnostics",
    button_style="info",
    icon="stethoscope",
    layout=widgets.Layout(width="320px")
)
hw_cp_diag_button.on_click(run_coldplate_diagnostic_from_hardware_tab)

hw_cp_diag_panel = widgets.VBox([
    widgets.HTML(
        "<small>This diagnostic reproduces the old Cold plate presets tab inside the Hardware-aware cooling model. "
        "It uses the same shared cold-plate geometry/flow/pressure-drop model as the hardware-aware optimizer, "
        "and synchronizes A, DeltaT, R_die, q plot max, q_target, cold-plate preset, sweep mode, and heat rejection "
        "from the current Hardware-aware controls before running.</small>"
    ),
    hw_cp_diag_button,
    cp_output
])

hardware_tab = widgets.VBox([
    collapsed_section("Hardware-aware equations / definitions", hw_notes_box()),

    widgets.HTML("<h3>Hardware-aware cooling model</h3>"),
    widgets.HTML("<small>Decision tree: choose chip model → apply chip/hotspot presets → select cold-plate preset → select heat rejection → choose sweep mode → run/recompute. Chip-model notes, cold-plate hardware notes, and cold-plate diagnostics are available as collapsible sections in this tab.</small>"),
    mode,
    hardware_source_box,
    widgets.HTML("<h3>MR optimization controls</h3>"),
    two_col([hw_opt_controls[k]["box"] for k in ["smax"]]),

    widgets.HTML("<h3>Cold-plate hardware and heat-rejection inputs</h3>"),
    widgets.HTML("<small>These controls define the external cold-plate hardware, heat-rejection model, and design/target heat-flux point used by the hardware-aware optimizer.</small>"),
    collapsed_section("Cold-plate hardware notes: equations, preset limits, provenance, and heat-rejection modes", coldplate_notes_box()),
    cp_preset,
    cp_sweep_mode,
    cp_reject_model,
    two_col([cp_controls[k]["box"] for k in ["qtarget"]]),
    hardware_heat_rejection_controls,
    collapsed_section("Cold-plate preset diagnostics", hw_cp_diag_panel),

    eq_box("Plot / visual controls", hw_plot_eqs),
    two_col([hw_plot_controls[k]["box"] for k in ["qmin", "qmax", "cop_ymax", "power_ymax", "temp_ymax", "var_ymax"]]),

    widgets.HBox([hw_button, hw_redraw_button]),
    hw_output
])

tabs = widgets.Tab(children=[reduced_order_tab, hardware_tab])
tabs.set_title(0, "Reduced-order cooling surrogate")
tabs.set_title(1, "Hardware-aware cooling model")

display(widgets.VBox([
    widgets.HTML("<h2>chip-microrefrigeration</h2>"),
    tabs
]))
run()


HTML(value='\n<style>\n.chip-note-box {\n    background-color: #fff8dc !important;\n    border-radius: 6px !im…

## Optional architecture-aware electrothermal sweep


In [3]:

# Optional architecture-aware electrothermal sweep.
sweep_button = widgets.Button(description="Plot architecture-aware sweep", button_style="info")
sweep_output = widgets.Output()

def run_sweep(_=None):
    with sweep_output:
        clear_output(wait=True)
        try:
            bu = BUParams(
                V0=getv(arch, "V0"), f0=getv(arch, "f0"),
                Q_dyn0=getv(arch, "Q_dyn0"), Q_leak0=getv(arch, "Q_leak0"),
                alpha_dyn0=getv(arch, "alpha_dyn0"), alpha_leak0=getv(arch, "alpha_leak0"),
                beta=getv(arch, "beta"), Vth=getv(arch, "Vth"), nu=getv(arch, "nu"),
                eta_D=getv(arch, "eta_D"), n=getv(arch, "sub_n"), V_t=getv(arch, "V_t"),
                a=getv(arch, "a"), b=getv(arch, "b")
            )
            lam = np.linspace(0.5, max(3.0, getv(arch, "lam")*1.5), 160)
            states = [bottom_up_state(x, bu) for x in lam]
            Qdyn = np.array([s["Q_dyn"] for s in states])
            Qleak = np.array([s["Q_leak"] for s in states])
            Q = np.array([s["Q_total"] for s in states])
            alpha = np.array([s["alpha_tot"] for s in states])
            gamma = np.array([s["gamma_H"] for s in states])
            V = np.array([s["V"] for s in states])
            f = np.array([s["f"] for s in states])

            fig, axs = plt.subplots(1, 2, figsize=(13, 4.8))
            axs[0].plot(lam, Qdyn, label="Q_dyn [W]", lw=2)
            axs[0].plot(lam, Qleak, label="Q_leak [W]", lw=2)
            axs[0].plot(lam, Q, label="Q_total [W]", lw=2.5)
            axs[0].set_xlabel("lambda [-]")
            axs[0].set_ylabel("Power [W]")
            axs[0].grid(True, alpha=0.3)
            ax2 = axs[0].twinx()
            ax2.plot(lam, alpha, "k:", label="alpha_tot [-]", lw=2)
            ax2.plot(lam, gamma, "k--", label="gamma_H [-]", lw=2)
            ax2.set_ylabel("alpha_tot / gamma_H [-]")
            lines, labels = axs[0].get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            axs[0].legend(lines+lines2, labels+labels2, loc="best")

            axs[1].plot(lam, V, lw=2, label="V [V]")
            axs[1].plot(lam, f, lw=2, label="f [GHz]")
            axs[1].set_xlabel("lambda [-]")
            axs[1].set_title("DVFS trajectory")
            axs[1].grid(True, alpha=0.3)
            axs[1].legend()
            fig.tight_layout()
            plt.show()
        except Exception as err:
            print("Error:", repr(err))

sweep_button.on_click(run_sweep)
display(widgets.VBox([sweep_button, sweep_output]))
run_sweep()
